# US-Aktien-Bot V1.5 – Dual-Bot-System (IEX + SIP)

Zwei streng getrennte Instanzen mit **derselben** 15-Minuten-ORB-Strategie aus V1.4.3. Alle Strategie-, Risiko- und Scannerparameter sind unverändert und kommen weiterhin aus dem geprüften Scannerlauf (`PREVIOUS_SCANNER_RUN`, Hash-Prüfung von v02/v03).

**Einzige freigegebene Regeländerung gegenüber V1.4.3 (Entscheidung vom 24.09.2026):** Das relative Volumen (RVOL) wird berechnet, wenn mindestens **18 der 20** Vergleichstage gültig sind. Maßgeblich ist dann der Durchschnitt über die gültigen Tage (`min_rvol_reference_days = 18`, vorher 20 von 20). Zu jedem ungültigen Vergleichstag steht der genaue Grund in `reference_checks/*.csv`, Spalte `error_detail`.

| | **BOT 1 – IEX-Echtzeit-Bot** (`IEX_REALTIME_PAPER`) | **BOT 2 – SIP-Delayed-Bot** (`SIP_DELAYED_SHADOW`) |
|---|---|---|
| Daten | Alpaca IEX, Echtzeit (auch RVOL-Referenzen aus IEX) | Alpaca SIP, **16 Min. verzögert** (kostenloser Zugang) |
| Ausführung | Alpaca-**Paper**-Orders nur mit gültiger Freigabe, sonst nur interne IEX-Simulation | **ausschließlich SIMULIERT**, kein Order-Client |
| Konto | echtes Paper-Konto (Orders) + virtuelles 10.000-USD-Vergleichskonto | eigenes virtuelles 10.000-USD-Konto, über Tage fortgeführt |
| Ablage | `MyDrive/US_Aktien_Bot/v1_5/IEX_REALTIME_PAPER/…` | `MyDrive/US_Aktien_Bot/v1_5/SIP_DELAYED_SHADOW/…` |

**Betriebsmodi** (`BOT_MODE` in der nächsten Zelle):
- `SIP_ONLY` – **Auslieferungszustand.** Nur BOT 2 läuft. Keine Alpaca-Orderfunktion ist erreichbar.
- `IEX_ONLY` – nur BOT 1.
- `DUAL_MODE` – beide Bots. BOT 2 läuft in einem **eigenen Prozess**, damit er BOT 1 nie blockieren kann.

In `IEX_ONLY` und `DUAL_MODE` sendet BOT 1 Paper-Orders **nur**, wenn `PAPER_ORDER_APPROVAL` exakt `PAPER-ORDERS-OK <Paper-Kontonummer> <heutiges NY-Datum>` lautet. Vor jeder Order wird geprüft:
- Paper-Endpunkt `paper-api.alpaca.markets`, niemals Echtgeld
- aktives USD-Konto, das dem freigegebenen Konto entspricht
- frisches Signal und frische Quote
- Abgleich von Positionen und Orders mit Alpaca
- Kapital-, Risiko-, Sektor-, Positions- und Tagesverlustgrenze

Fehlt die Freigabe oder ist sie ungültig, läuft BOT 1 nur als IEX-Simulation.

**Die Orders von BOT 1:**
- **Einstieg:** IOC-Limit zum frischen Ask.
- **Trailing-Stop:** läuft beim Broker, 1 %, gültig für den Tag.
- **Tagesabschluss:** Glattstellung um 15:30 NY mit bestätigter Position 0.
- **Doppelorder-Schutz:** eindeutige Order-IDs je Tag und Aktie. Nach einem Neustart wird kein Signal nachträglich gekauft.
- **Wichtig:** Colab ist nicht ausfallsicher. Bricht die Sitzung ab, gibt es keine garantierte Glattstellung. Die Trailing-Stops beim Broker bleiben bis Handelsschluss aktiv.

**BOT 2:**
- Bei Start oder Neustart simuliert er den Tag chronologisch ab Börsenöffnung nach und folgt dann mit 16 Minuten Versatz. Jede Anfrage endet mindestens 16 Minuten vor der Echtzeit.
- Ein Handelstag wird genau einmal gebucht, ohne Doppelbuchungen.
- Alle Ereignisse tragen den Marktzeitpunkt (`at_utc`) und den Verarbeitungszeitpunkt (`processing_utc`).

**Cockpit:** Drei Bereiche – BOT 1, BOT 2 (SIMULIERT) und der Vergleich IEX gegen SIP. Solange SIP zurückliegt, wird der Vergleich als VORLÄUFIG gekennzeichnet. Es erscheint in Colab und über den Link „Cockpit im Browser öffnen“. Zusätzlich liegt `MyDrive/US_Aktien_Bot/cockpit_latest.html` auf Drive.

**Protokolle** unter `MyDrive/US_Aktien_Bot/v1_5/`:
- `<BOT>_trades.csv`, `<BOT>_signals.csv` und `<BOT>_daily_performance.csv` je Bot
- `IEX_REALTIME_PAPER_SIM_*` für die interne IEX-Simulation
- `IEX_SIP_comparison.csv`: je Aktie und Tag Opening Range, RVOL, Signal, Signalzeit, Kauf oder Ablehnung
- `performance_series.csv` mit drei Reihen: IEX-Paper-Fills, IEX-Simulation, SIP-Simulation

Die Laufordner werden nie überschrieben. Replays vergangener Tage bleiben im Notebook V1.4.3.


In [ ]:
%pip -q install alpaca-py requests


In [ ]:
PREVIOUS_SCANNER_RUN = '20260923T124355Z_838f60df'

# V1.5 – Dual-Bot-System (IEX + SIP)
# 'SIP_ONLY'  = nur BOT 2 (SIP verzögert, SIMULIERT)            <- Auslieferungszustand
# 'IEX_ONLY'  = nur BOT 1 (IEX Echtzeit)
# 'DUAL_MODE' = beide Bots
BOT_MODE = 'SIP_ONLY'

# Alpaca-PAPER-Orders von BOT 1 NUR mit exakt dieser Freigabe für HEUTE und DIESES Paper-Konto:
#   PAPER_ORDER_APPROVAL = 'PAPER-ORDERS-OK <Paper-Kontonummer> <YYYY-MM-DD, NY-Datum>'
# None (oder jede andere Angabe) = keine Orders; BOT 1 läuft dann nur als IEX-Simulation.
PAPER_ORDER_APPROVAL = None


In [ ]:
%%writefile /content/us_orb_test_v02.py
"""US-ORB V0.2: historisches Testlabor. Keine Ordermethoden enthalten."""
from dataclasses import dataclass, asdict
from decimal import Decimal, ROUND_FLOOR
from datetime import datetime, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo
import hashlib
import importlib.metadata
import json
import platform
import uuid
import numpy as np
import pandas as pd

NY = ZoneInfo("America/New_York")
VERSION = "0.2-testlabor"
PENDING = [
    "Aktuelles Nasdaq-100-Universum und zeitgerechte historische Mitglieder",
    "Mindestliquidität, historische Quotes/Spread, Ranking und relative Stärke",
    "Signalgültigkeit und Wiedereinstiege",
    "Gemeinsame Portfolio-/Risikosimulation mit Orderreservierungen",
    "Ausführungsmodell, Kosten, Slippage, MFE/MAE und Benchmark",
    "Broker-Trailing-Stop, Teilfüllungen, Abbruch- und Neustartabgleich",
    "Persistente operative Tagessperre und Maßnahmen für offene Positionen",
    "Echtzeit-Feed-Entscheidung und sieben Handelstage Paper Trading",
]

@dataclass(frozen=True)
class Config:
    test_date: str = "2026-09-22"
    symbols: tuple = ("AAPL", "MSFT", "NVDA")
    opening_minutes: int = 15
    breakout_buffer: str = "0.0005"
    entry_cap: str = "0.003"
    trailing: str = "0.01"
    rvol_days: int = 20
    minimum_rvol: float = 1.5
    max_position_capital: str = "0.30"
    max_trade_risk: str = "0.01"
    max_total_risk: str = "0.03"
    max_positions: int = 3
    max_per_sector: int = 2
    daily_loss: str = "0.02"
    entry_cutoff_minutes: int = 45
    close_before_minutes: int = 30
    minimum_price: float = 20.0

def D(value):
    value = Decimal(str(value))
    if not value.is_finite():
        raise ValueError("Nicht endlicher Zahlenwert")
    return value

def utc(value):
    stamp = pd.Timestamp(value)
    if stamp.tzinfo is None:
        stamp = stamp.tz_localize(NY)
    return stamp.tz_convert("UTC")

def validate_config(c):
    datetime.strptime(c.test_date, "%Y-%m-%d")
    if not c.symbols or len(set(c.symbols)) != len(c.symbols):
        raise ValueError("Aktienliste ist leer oder enthält Duplikate")
    if any(not isinstance(s, str) or not s.strip() for s in c.symbols):
        raise ValueError("Ungültiges Symbol")
    for name in ("trailing", "max_position_capital", "max_trade_risk",
                 "max_total_risk", "daily_loss"):
        if not 0 < D(getattr(c, name)) < 1:
            raise ValueError(f"Ungültiger Parameter: {name}")
    if not 0 <= D(c.breakout_buffer) <= D(c.entry_cap) < 1:
        raise ValueError("Ungültiger Ausbruchsabstand")
    if not 0 < c.close_before_minutes < c.entry_cutoff_minutes:
        raise ValueError("Ungültige Schlusszeiten")
    if min(c.opening_minutes, c.rvol_days, c.max_positions, c.max_per_sector) < 1:
        raise ValueError("Ungültige Anzahl")
    if not np.isfinite(c.minimum_rvol) or c.minimum_rvol <= 0:
        raise ValueError("Ungültiger Volumenfaktor")

def size_position(equity, free_cash, existing_risk, entry, c):
    eq, cash, risk, price = map(D, (equity, free_cash, existing_risk, entry))
    if eq <= 0 or price <= 0 or cash < 0 or risk < 0:
        raise ValueError("Ungültige Größenberechnung")
    capital = min(eq * D(c.max_position_capital), cash)
    budget = min(eq * D(c.max_trade_risk), max(D(0), eq * D(c.max_total_risk) - risk))
    distance = price * D(c.trailing)
    qty = int(min(capital / price, budget / distance).to_integral_value(rounding=ROUND_FLOOR))
    return dict(qty=qty, notional=qty * price, planned_risk=qty * distance,
                risk_budget=budget)

def entry_gate(start_equity, current_equity, confirmed_net_cashflows,
               occupied_sectors, candidate_sector, day_locked, c):
    start, current, flows = map(D, (start_equity, current_equity, confirmed_net_cashflows))
    if start <= 0 or current <= 0:
        raise ValueError("Ungültiges Eigenkapital")
    pnl = current - start - flows
    locked = bool(day_locked) or pnl <= -start * D(c.daily_loss)
    reasons = []
    if locked:
        reasons.append("Tagesverlustsperre")
    if len(occupied_sectors) >= c.max_positions:
        reasons.append("Positionsplätze belegt/reserviert")
    if not candidate_sector or any(not s for s in occupied_sectors):
        reasons.append("Sektorzuordnung fehlt")
    elif occupied_sectors.count(candidate_sector) >= c.max_per_sector:
        reasons.append("Sektorlimit erreicht")
    return dict(allowed=not reasons, day_locked=locked, day_pnl=pnl, reasons=reasons)

def simulate_exit(entry_price, entry_time, ticks, session_close, c):
    price, trail = D(entry_price), D(c.trailing)
    if price <= 0 or not 0 < trail < 1:
        raise ValueError("Ungültiger Trailing-Test")
    start, close = utc(entry_time), utc(session_close)
    cutoff = close - pd.Timedelta(minutes=c.close_before_minutes)
    if start >= cutoff:
        raise ValueError("Einstieg liegt nach Schließungsgrenze")
    high, stop, last_time = price, price * (1 - trail), start
    rows = [dict(time=start, price=price, high=high, stop=stop, event="START")]
    for stamp, raw_price in ticks:
        stamp, price = utc(stamp), D(raw_price)
        if stamp <= last_time or price <= 0:
            raise ValueError("Ungültige Kursfolge")
        if stamp >= cutoff:
            event = "TAGESABSCHLUSS"
        elif price <= stop:
            event = "TRAILING_STOP"
        else:
            high = max(high, price)
            stop = max(stop, high * (1 - trail))
            event = "HALTEN"
        rows.append(dict(time=stamp, price=price, high=high, stop=stop, event=event))
        last_time = stamp
        if event in ("TRAILING_STOP", "TAGESABSCHLUSS"):
            break
    return rows

def self_tests():
    """Feste Referenzparameter, getrennt von späteren Nutzer-Varianten."""
    c = Config()
    results, stop_rows = [], []
    for label, cash, risk, qty, notional, planned in [
        ("Normalfall", 10000, 0, 13, "2990", "29.90"),
        ("Wenig Cash", 500, 0, 2, "460", "4.60"),
        ("10 USD Risikospielraum", 10000, 290, 4, "920", "9.20"),
        ("Risiko ausgeschöpft", 10000, 300, 0, "0", "0"),
    ]:
        r = size_position(10000, cash, risk, 230, c)
        passed = r["qty"] == qty and r["notional"] == D(notional) and r["planned_risk"] == D(planned)
        results.append(dict(Bereich="Positionsgröße", Test=label, Bestanden=passed))
    for label, eq, flow, sectors, locked, expected in [
        ("Normalfall", 10000, 0, [], False, True),
        ("3 Plätze belegt", 10000, 0, ["Tech", "Industrie", "Gesundheit"], False, False),
        ("2 Tech-Positionen", 10000, 0, ["Tech", "Tech"], False, False),
        ("Genau 200 USD Verlust", 9800, 0, [], False, False),
        ("Einzahlung", 11000, 1000, [], False, True),
        ("Entnahme", 9000, -1000, [], False, True),
        ("Einzahlung plus Verlust", 10800, 1000, [], False, False),
        ("Tag bleibt gesperrt", 10000, 0, [], True, False),
    ]:
        r = entry_gate(10000, eq, flow, sectors, "Tech", locked, c)
        passed = r["allowed"] == expected
        if label in ("Einzahlung", "Entnahme"):
            passed = passed and r["day_pnl"] == 0
        results.append(dict(Bereich="Einstiegssperren", Test=label, Bestanden=passed))
    def at(hm):
        return pd.Timestamp(f"2026-09-22 {hm}", tz=NY)
    for label, start, ticks, close, reason, final_stop in [
        ("Steigt und fällt", "10:00", [("10:01", 101), ("10:02", 102), ("10:03", 101.5), ("10:04", 100.98)], "16:00", "TRAILING_STOP", "100.98"),
        ("Kurssprung", "10:00", [("10:01", 97)], "16:00", "TRAILING_STOP", "99"),
        ("Regulärer Abschluss", "15:28", [("15:29", 100.5), ("15:30", 100.4)], "16:00", "TAGESABSCHLUSS", "99.495"),
        ("Synthetischer Kurzhandelstag", "12:28", [("12:29", 100.5), ("12:30", 100.4)], "13:00", "TAGESABSCHLUSS", "99.495"),
    ]:
        rows = simulate_exit(100, at(start), [(at(t), p) for t, p in ticks], at(close), c)
        passed = rows[-1]["event"] == reason and rows[-1]["stop"] == D(final_stop)
        passed = passed and all(b["stop"] >= a["stop"] for a, b in zip(rows, rows[1:]))
        if label == "Steigt und fällt":
            passed = passed and [r["stop"] for r in rows] == list(map(D, ["99", "99.99", "100.98", "100.98", "100.98"]))
        results.append(dict(Bereich="Ausstieg", Test=label, Bestanden=passed))
        stop_rows.extend(dict(Test=label, **row) for row in rows)
    return pd.DataFrame(results), pd.DataFrame(stop_rows)

def normalize_bars(frame):
    df = frame.reset_index() if "timestamp" not in frame.columns else frame.copy()
    columns = ["symbol", "timestamp", "open", "high", "low", "close", "volume"]
    if not set(columns).issubset(df.columns):
        raise ValueError("Benötigte Kursdatenspalten fehlen")
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    numeric = ["open", "high", "low", "close", "volume"]
    df[numeric] = df[numeric].apply(pd.to_numeric, errors="raise")
    df["feed"] = "sip"
    return df.sort_values(["symbol", "timestamp"]).reset_index(drop=True)

def checked_window(df, symbol, start, end):
    window = df[(df.symbol == symbol) & (df.timestamp >= start) & (df.timestamp < end)].sort_values("timestamp")
    expected = pd.date_range(start, end, freq="min", inclusive="left")
    if not pd.DatetimeIndex(window.timestamp).equals(expected):
        raise ValueError(f"{symbol}: Minuten fehlen, sind doppelt oder zeitlich falsch")
    cols = ["open", "high", "low", "close", "volume"]
    if not np.isfinite(window[cols].to_numpy()).all():
        raise ValueError(f"{symbol}: Ungültige Zahlen")
    if not ((window[["open", "high", "low", "close"]] > 0).all(axis=1)
            & (window.volume >= 0)
            & (window.high >= window[["open", "close", "low"]].max(axis=1))
            & (window.low <= window[["open", "close", "high"]].min(axis=1))).all():
        raise ValueError(f"{symbol}: Widersprüchliche Kurswerte")
    return window

def scan_day(df, opening, closing, c):
    or_end = opening + pd.Timedelta(minutes=c.opening_minutes)
    cutoff = closing - pd.Timedelta(minutes=c.entry_cutoff_minutes)
    if or_end >= cutoff:
        raise ValueError("Handelssitzung ist für diese Zeitregeln zu kurz")
    decisions, ranges = [], []
    for symbol in c.symbols:
        stock = checked_window(df, symbol, opening, closing)
        initial = stock[stock.timestamp < or_end]
        high, low = D(initial.high.max()), D(initial.low.min())
        threshold = high * (1 + D(c.breakout_buffer))
        cap = high * (1 + D(c.entry_cap))
        ranges.append(dict(Aktie=symbol, OR_Hoch=float(high), OR_Tief=float(low),
                           Ausbruchsschwelle=float(threshold), Preisobergrenze=float(cap),
                           OR_Volumen=float(initial.volume.sum())))
        previous = D(initial.close.iloc[-1])
        for bar in stock[stock.timestamp >= or_end].itertuples():
            decision = bar.timestamp + pd.Timedelta(minutes=1)
            if decision >= cutoff:
                break
            close = D(bar.close)
            crossed = previous <= threshold and close > threshold
            if not crossed:
                result = "NO_CROSSING"
            elif close < D(c.minimum_price):
                result = "BELOW_MINIMUM_PRICE"
            elif close > cap:
                result = "TOO_EXTENDED"
            else:
                result = "PRICE_CANDIDATE"
            decisions.append(dict(symbol=symbol, bar_time_utc=bar.timestamp.isoformat(),
                decision_time_utc=decision.isoformat(), decision_time_ny=decision.tz_convert(NY).isoformat(),
                or_high=float(high), threshold=float(threshold), price_cap=float(cap),
                previous_close=float(previous), close=float(close), result=result, feed="sip"))
            previous = close
    return pd.DataFrame(ranges), pd.DataFrame(decisions)

RVOL_COLUMNS = ["Aktie", "Signalzeit_NY", "decision_time_utc", "Volumen_bis_Signal",
    "Durchschnitt_20_Tage", "RVOL", "Mindest_RVOL", "Vollständige_Vergleichstage", "Ergebnis"]

def check_rvol(candidates, current, history, past_sessions, current_open, c):
    outputs, references = [], []
    for candidate in candidates.itertuples():
        decision = utc(candidate.decision_time_utc)
        elapsed = decision - current_open
        actual = float(checked_window(current, candidate.symbol, current_open, decision).volume.sum())
        volumes = []
        for session in past_sessions:
            start, close = session["open"], session["close"]
            end = start + elapsed
            volume, status = None, "OK"
            if end > close:
                status = "Handelstag für Vergleich zu kurz"
            else:
                try:
                    volume = float(checked_window(history, candidate.symbol, start, end).volume.sum())
                except ValueError:
                    status = "Minuten fehlen oder sind ungültig"
            references.append(dict(symbol=candidate.symbol, signal_time_utc=decision.isoformat(),
                reference_date=session["date"], volume=volume, status=status))
            if volume is not None:
                volumes.append(volume)
        avg = float(np.mean(volumes)) if len(volumes) == c.rvol_days else None
        rvol = actual / avg if avg is not None and avg > 0 else None
        result = "NOT_CHECKABLE" if rvol is None else ("VOLUME_PASS" if rvol >= c.minimum_rvol else "VOLUME_REJECT")
        outputs.append(dict(Aktie=candidate.symbol, Signalzeit_NY=decision.tz_convert(NY).strftime("%H:%M"),
            decision_time_utc=decision.isoformat(), Volumen_bis_Signal=actual, Durchschnitt_20_Tage=avg,
            RVOL=rvol, Mindest_RVOL=c.minimum_rvol, Vollständige_Vergleichstage=len(volumes), Ergebnis=result))
    return pd.DataFrame(outputs, columns=RVOL_COLUMNS), pd.DataFrame(references,
        columns=["symbol", "signal_time_utc", "reference_date", "volume", "status"])

def json_write(path, content):
    with Path(path).open("x", encoding="utf-8") as f:
        json.dump(content, f, ensure_ascii=False, indent=2, default=str, allow_nan=False)

def csv_write(path, frame):
    frame.to_csv(path, index=False, mode="x")

def calendar_rows(items):
    rows = [dict(date=str(s.date), open=utc(s.open), close=utc(s.close)) for s in items]
    rows.sort(key=lambda s: s["date"])
    if len({r["date"] for r in rows}) != len(rows):
        raise ValueError("Börsenkalender enthält doppelte Tage")
    return rows

def run_lab(broker, data_client, project_root, c):
    """Nur lesende Brokerzugriffe; alle Ergebnisse in einem eigenen Laufordner."""
    from alpaca.trading.requests import GetCalendarRequest, GetOrdersRequest
    from alpaca.trading.enums import QueryOrderStatus
    from alpaca.data.requests import StockBarsRequest
    from alpaca.data.enums import DataFeed, Adjustment
    from alpaca.data.timeframe import TimeFrame
    validate_config(c)
    root = Path(project_root)
    if not root.is_dir():
        raise ValueError("Projektordner fehlt; Google Drive prüfen")
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    out = root / "v0_2_runs" / run_id
    out.mkdir(parents=True, exist_ok=False)
    source = Path(__file__).read_text(encoding="utf-8")
    (out / "source.py").write_text(source, encoding="utf-8")
    package_versions = {}
    for package in ("alpaca-py", "pandas", "numpy"):
        package_versions[package] = importlib.metadata.version(package)
    json_write(out / "manifest.json", dict(version=VERSION, run_id=run_id,
        created_at_utc=datetime.now(timezone.utc).isoformat(), mode="historical_read_only",
        config=asdict(c), feed="sip", code_sha256=hashlib.sha256(source.encode()).hexdigest(),
        packages=package_versions, python=platform.python_version(),
        scope="3-Symbol-Testlabor; keine Orders, Ausführungen oder Performance-Simulation",
        daily_loss_definition="Equity jetzt minus Equity Tagesstart minus bestätigte Nettoeinzahlungen; Grenze 2% Tagesstart, Sperre bis nächste Sitzung (nur Logiktest)",
        assumptions="Strategieparameter Entwurf; keine Optimierung oder Profitabilitätsaussage", pending=PENDING))
    try:
        print("1/4 Prüfe die 16 bekannten Rechen- und Ausstiegstests …")
        tests, stops = self_tests()
        csv_write(out / "self_tests.csv", tests)
        csv_write(out / "synthetic_stop_paths.csv", stops)
        if not tests.Bestanden.all():
            raise ValueError("Mindestens ein Referenztest ist fehlgeschlagen")
        print("✅ 16/16 Referenztests bestanden")

        print("2/4 Lese Paper-Konto und Börsenkalender …")
        account = broker.get_account()
        clock = broker.get_clock()
        positions = broker.get_all_positions()
        orders = broker.get_orders(filter=GetOrdersRequest(status=QueryOrderStatus.OPEN, limit=500))
        account_status = getattr(account.status, "value", str(account.status))
        snapshot = dict(read_at_utc=datetime.now(timezone.utc).isoformat(), status=account_status,
            currency=account.currency, equity=str(account.equity), cash=str(account.cash),
            account_blocked=account.account_blocked, trading_blocked=account.trading_blocked,
            positions_count=len(positions), open_orders_returned=len(orders),
            orders_list_may_be_truncated=len(orders) >= 500, market_open=clock.is_open)
        json_write(out / "paper_account_read_only.json", snapshot)
        if account.currency != "USD":
            raise ValueError("Testprojekt erwartet ein USD-Konto")
        day = datetime.strptime(c.test_date, "%Y-%m-%d").date()
        sessions = calendar_rows(broker.get_calendar(GetCalendarRequest(start=day, end=day)))
        if len(sessions) != 1:
            raise ValueError("Gewähltes Datum ist kein eindeutiger Handelstag")
        session = sessions[0]
        if session["close"] > utc(clock.timestamp) - pd.Timedelta(minutes=15):
            raise ValueError("Bitte einen vollständig vergangenen Handelstag wählen (mindestens 15 Minuten nach Schluss)")
        json_write(out / "session_calendar.json", session)

        print(f"3/4 Lade SIP-Daten für {c.test_date}: {', '.join(c.symbols)} …")
        def fetch(symbols, start, end):
            bars = data_client.get_stock_bars(StockBarsRequest(
                symbol_or_symbols=list(symbols), timeframe=TimeFrame.Minute,
                start=start.to_pydatetime(), end=end.to_pydatetime(),
                feed=DataFeed.SIP, adjustment=Adjustment.RAW))
            if not any(bars.data.values()):
                raise ValueError("Keine SIP-Kursdaten geliefert")
            return normalize_bars(bars.df)
        current = fetch(c.symbols, session["open"], session["close"])
        current = current[(current.timestamp >= session["open"]) & (current.timestamp < session["close"])].copy()
        csv_write(out / "session_bars_sip.csv", current)
        ranges, decisions = scan_day(current, session["open"], session["close"], c)
        csv_write(out / "opening_ranges.csv", ranges)
        csv_write(out / "minute_decisions.csv", decisions)
        candidates = decisions[decisions.result == "PRICE_CANDIDATE"].copy()

        print(f"4/4 Prüfe Volumen für {len(candidates)} Preiskandidat(en) …")
        if not candidates.empty:
            past = calendar_rows(broker.get_calendar(GetCalendarRequest(
                start=day - timedelta(days=max(60, c.rvol_days * 3)), end=day - timedelta(days=1))))[-c.rvol_days:]
            if len(past) != c.rvol_days:
                raise ValueError("Nicht genug historische Vergleichstage")
            json_write(out / "reference_calendar.json", past)
            history = fetch(sorted(candidates.symbol.unique()), past[0]["open"], past[-1]["close"])
            csv_write(out / "reference_bars_sip.csv", history)
            volumes, reference_checks = check_rvol(candidates, current, history, past, session["open"], c)
        else:
            volumes = pd.DataFrame(columns=RVOL_COLUMNS)
            reference_checks = pd.DataFrame(columns=["symbol", "signal_time_utc", "reference_date", "volume", "status"])
        csv_write(out / "volume_candidates.csv", volumes)
        csv_write(out / "volume_reference_checks.csv", reference_checks)
        summary_rows = []
        for symbol in c.symbols:
            signal_rows = candidates[candidates.symbol == symbol]
            volume_rows = volumes[volumes.Aktie == symbol]
            pass_count = int((volume_rows.Ergebnis == "VOLUME_PASS").sum())
            unknown = int((volume_rows.Ergebnis == "NOT_CHECKABLE").sum())
            summary_rows.append(dict(Aktie=symbol, Preiskandidaten=len(signal_rows),
                Volumen_bestanden=pass_count, Volumen_nicht_prüfbar=unknown,
                Ergebnis=("Weitere Filter offen" if pass_count else
                          "Volumendaten unzureichend" if unknown else
                          "Volumenfilter nicht bestanden" if len(signal_rows) else "Kein Preiskandidat")))
        summary = pd.DataFrame(summary_rows)
        csv_write(out / "summary.csv", summary)
        json_write(out / "completed.json", dict(status="completed", reference_tests_passed=16,
            price_candidates=len(candidates), volume_pass=int((volumes.Ergebnis == "VOLUME_PASS").sum()),
            order_methods_called=0, executions_simulated=False,
            finished_at_utc=datetime.now(timezone.utc).isoformat()))
        return dict(folder=out, account=snapshot, tests=tests, ranges=ranges,
                    volumes=volumes, summary=summary)
    except Exception as exc:
        # Keine API-Antworten, Zugangsdaten oder vollständigen Accountobjekte protokollieren.
        status = getattr(exc, "status_code", None)
        detail = str(exc) if type(exc) is ValueError else "API-/Laufzeitfehler; HTTP-Status und Fehlertyp prüfen"
        json_write(out / "failed.json", dict(status="failed", type=type(exc).__name__,
            http_status=status, detail=detail, time_utc=datetime.now(timezone.utc).isoformat()))
        raise RuntimeError(f"Testlauf gestoppt: {detail}. Typ: {type(exc).__name__}; HTTP: {status}. Protokoll: {out}") from None


In [ ]:
%%writefile /content/us_orb_scanner_v03.py
"""V0.3 historical scanner. Read-only; no order submission or fill simulation."""
from dataclasses import dataclass, asdict, replace
from datetime import datetime, timedelta, timezone
from pathlib import Path
from types import SimpleNamespace
import hashlib
import importlib.metadata
import json
import time
import urllib.request
import uuid
import numpy as np
import pandas as pd
import us_orb_test_v02 as base

VERSION = '0.3-scanner-draft'
MEMBERS_URL = 'https://api.nasdaq.com/api/quote/list-type/nasdaq100'
SECTORS_URL = 'https://api.nasdaq.com/api/screener/stocks?tableonly=true&limit=10000&download=true'
EMPTY_BARS = ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'vwap']

@dataclass(frozen=True)
class ScannerConfig:
    minimum_average_dollar_volume: float = 50_000_000
    maximum_spread: float = 0.001  # 0.10%, divided by midpoint
    maximum_quote_age_seconds: float = 2.0
    quote_search_seconds: int = 10
    watchlist_size: int = 5
    signal_ttl_seconds: int = 60
    weight_rvol: float = 40
    weight_relative_strength: float = 25
    weight_breakout: float = 20
    weight_spread: float = 15
    rvol_score_ceiling: float = 3.0
    relative_strength_score_ceiling: float = 0.01  # 1 percentage point
    batch_size: int = 20
    request_spacing_seconds: float = 0.4


def validate_scanner(c, s):
    base.validate_config(c)
    for name, value in asdict(s).items():
        if not np.isfinite(value) or value <= 0:
            raise ValueError(f'Ungültiger Scannerparameter: {name}')
    if s.watchlist_size > 5 or s.rvol_score_ceiling <= c.minimum_rvol:
        raise ValueError('Ungültige Watchlist oder RVOL-Skalierung')
    if s.quote_search_seconds < s.maximum_quote_age_seconds:
        raise ValueError('Quote-Suchfenster ist zu kurz')
    if s.signal_ttl_seconds > 60:
        raise ValueError('Diese Version unterstützt höchstens 60 Sekunden Signalgültigkeit')
    if float(c.entry_cap) <= float(c.breakout_buffer):
        raise ValueError('Ranking benötigt einen positiven Preisabstandsbereich')
    if not np.isclose(sum([s.weight_rvol, s.weight_relative_strength,
                           s.weight_breakout, s.weight_spread]), 100):
        raise ValueError('Rankinggewichte müssen zusammen 100 ergeben')


def fetch_json(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json'})
    with urllib.request.urlopen(req, timeout=20) as response:
        return json.load(response)


def assemble_universe(members, sectors):
    rows = members['data']['data']['rows']
    sector_rows = sectors['data']['rows']
    mapping = {}
    for row in sector_rows:
        symbol = row['symbol'].strip().upper()
        mapping.setdefault(symbol, set()).add((row.get('sector') or '').strip())
    output = []
    for row in rows:
        symbol = row['symbol'].strip().upper()
        choices = mapping.get(symbol, set()) - {''}
        output.append(dict(symbol=symbol, company=row.get('companyName', ''),
                           sector=next(iter(choices)) if len(choices) == 1 else ''))
    frame = pd.DataFrame(output)
    if len(frame) < 90 or len(frame) > 120 or frame.symbol.duplicated().any():
        raise ValueError('Nasdaq-Mitgliederliste ist unplausibel')
    return frame.sort_values('symbol').reset_index(drop=True)


def load_universe(out, embedded):
    # This is a diagnostic snapshot, NEVER a claimed historical point-in-time universe.
    try:
        members = fetch_json(MEMBERS_URL)
        sectors = fetch_json(SECTORS_URL)
        frame = assemble_universe(members, sectors)
    except Exception as exc:
        frame = pd.DataFrame(embedded['rows'])
        metadata = {k: v for k, v in embedded.items() if k != 'rows'}
        metadata.update(source_mode='embedded_snapshot', refresh_failure_type=type(exc).__name__)
        print('Nasdaq-Aktualisierung nicht verfügbar; verwende gekennzeichneten Snapshot vom',
              metadata['retrieved_at_utc'])
    else:
        base.json_write(out / 'nasdaq_members_source.json', members)
        base.json_write(out / 'nasdaq_sectors_source.json', sectors)
        metadata = dict(source_mode='fresh_current_snapshot',
                        retrieved_at_utc=datetime.now(timezone.utc).isoformat(),
                        quote_date_label=members['data'].get('date'),
                        member_source=MEMBERS_URL, sector_source=SECTORS_URL)
    if frame.empty or frame.symbol.duplicated().any() or not {'symbol', 'sector'}.issubset(frame.columns):
        raise ValueError('Universum fehlt oder ist uneindeutig')
    metadata.update(historical_membership_verified=False, historical_sectors_verified=False,
                    scope='Retrospektiver Funktionstest mit einem datierten Mitglieder-/Sektorsnapshot')
    base.csv_write(out / 'universe_snapshot.csv', frame)
    base.json_write(out / 'universe_provenance.json', metadata)
    return frame, metadata


def liquidity_check(daily, symbol, dates, s):
    rows = daily[daily.symbol == symbol].copy()
    if not rows.empty:
        rows['session_date'] = rows.timestamp.dt.tz_convert(base.NY).dt.strftime('%Y-%m-%d')
        rows = rows[rows.session_date.isin(dates)].sort_values('session_date')
    if len(rows) != len(dates) or rows.empty or set(rows.session_date) != set(dates):
        return dict(average_dollar_volume=None, liquidity_result='DAILY_HISTORY_INCOMPLETE')
    if rows.session_date.duplicated().any() or 'vwap' not in rows:
        return dict(average_dollar_volume=None, liquidity_result='DAILY_HISTORY_INVALID')
    values = rows[['vwap', 'volume']].to_numpy(dtype=float)
    if not np.isfinite(values).all() or (values <= 0).any():
        return dict(average_dollar_volume=None, liquidity_result='DAILY_HISTORY_INVALID')
    average = float((rows.vwap * rows.volume).mean())
    return dict(average_dollar_volume=average,
                liquidity_result='PASS' if average >= s.minimum_average_dollar_volume else 'ILLIQUID')


def scan_prefix(current, symbol, opening, closing, c):
    """Only a complete prefix can generate a signal; later gaps do not erase earlier signals."""
    or_end = opening + pd.Timedelta(minutes=c.opening_minutes)
    cutoff = closing - pd.Timedelta(minutes=c.entry_cutoff_minutes)
    if or_end >= cutoff:
        raise ValueError('Handelssitzung ist zu kurz')
    current = current[current.symbol == symbol]
    initial = base.checked_window(current, symbol, opening, or_end)
    high, low = base.D(initial.high.max()), base.D(initial.low.min())
    threshold = high * (1 + base.D(c.breakout_buffer))
    cap = high * (1 + base.D(c.entry_cap))
    previous = base.D(initial.close.iloc[-1])
    prefix_valid, decisions = True, []
    for stamp in pd.date_range(or_end, cutoff - pd.Timedelta(minutes=1), freq='min', inclusive='left'):
        decision = stamp + pd.Timedelta(minutes=1)
        close = None
        if prefix_valid:
            try:
                one = base.checked_window(current, symbol, stamp, decision)
                close = base.D(one.close.iloc[0])
            except ValueError:
                prefix_valid = False
        if not prefix_valid:
            reason = 'INCOMPLETE_DATA_PREFIX'
        elif not (previous <= threshold and close > threshold):
            reason = 'NO_CROSSING'
        elif close < base.D(c.minimum_price):
            reason = 'BELOW_MINIMUM_PRICE'
        elif close > cap:
            reason = 'TOO_EXTENDED'
        else:
            reason = 'PRICE_CANDIDATE'
        decisions.append(dict(symbol=symbol, bar_time_utc=stamp.isoformat(),
            decision_time_utc=decision.isoformat(), decision_time_ny=decision.tz_convert(base.NY).isoformat(),
            or_high=float(high), threshold=float(threshold), price_cap=float(cap),
            close=float(close) if close is not None else None, result=reason))
        if close is not None:
            previous = close
    opening_range = dict(symbol=symbol, or_high=float(high), or_low=float(low),
                         or_open=float(initial.open.iloc[0]), or_volume=float(initial.volume.sum()))
    return opening_range, pd.DataFrame(decisions)


def quote_check(quote, decision, threshold, cap, s, minimum_price=20.0):
    result = dict(quote_result='QUOTE_MISSING', quote_time_utc=None,
                  bid=None, ask=None, bid_size=None, ask_size=None,
                  quote_age_seconds=None, spread=None)
    if quote is None:
        return result
    try:
        stamp = base.utc(quote.timestamp)
        bid, ask, bsize, asize = map(float, [quote.bid_price, quote.ask_price, quote.bid_size, quote.ask_size])
        age = (base.utc(decision) - stamp).total_seconds()
        result.update(quote_time_utc=stamp.isoformat(), bid=bid, ask=ask,
                      bid_size=bsize, ask_size=asize, quote_age_seconds=age)
        if not np.isfinite([bid, ask, bsize, asize, age]).all() or min(bid, ask, bsize, asize) <= 0 or bid > ask:
            result['quote_result'] = 'QUOTE_INVALID'
        elif age < 0:
            result['quote_result'] = 'QUOTE_FROM_FUTURE'
        elif age > s.maximum_quote_age_seconds:
            result['quote_result'] = 'QUOTE_STALE'
        else:
            spread = (ask - bid) / ((ask + bid) / 2)
            result['spread'] = spread
            result['quote_result'] = ('ASK_BELOW_MINIMUM_PRICE' if ask < minimum_price else
                                     'SPREAD_TOO_WIDE' if spread > s.maximum_spread else
                                     'ASK_OUTSIDE_ENTRY_BAND' if not float(threshold) < ask <= float(cap) else 'PASS')
    except (AttributeError, TypeError, ValueError):
        result['quote_result'] = 'QUOTE_INVALID'
    return result


def timing_status(decision, started_at, now, cutoff, reconciled, already_seen, s):
    decision, started, now, cutoff = map(base.utc, [decision, started_at, now, cutoff])
    if decision < started:
        return 'HISTORICAL_ONLY'
    if decision > now:
        return 'FUTURE_SIGNAL'
    if decision >= cutoff or now >= cutoff:
        return 'ENTRY_CUTOFF'
    if (now - decision).total_seconds() >= s.signal_ttl_seconds:
        return 'EXPIRED'
    if already_seen:
        return 'DUPLICATE_SIGNAL'
    if not reconciled:
        return 'BROKER_RECONCILIATION_REQUIRED'
    return 'CURRENT_SIGNAL_REQUIRES_ORDER_ENGINE'  # No trade authorization here.


def rank_candidates(frame, c, s):
    if frame.empty:
        return pd.DataFrame(columns=list(frame.columns) + ['score', 'rank', 'watchlist', 'ranking_result'])
    if frame.duplicated(['decision_time_utc', 'symbol']).any():
        raise ValueError('Doppeltes Signal in Rangfolge')
    ranked = frame.copy()
    ranked['rvol_component'] = ((ranked.rvol - c.minimum_rvol) /
                                (s.rvol_score_ceiling - c.minimum_rvol)).clip(0, 1) * s.weight_rvol
    ranked['strength_component'] = (ranked.relative_strength / s.relative_strength_score_ceiling).clip(0, 1) * s.weight_relative_strength
    ranked['breakout_component'] = ((ranked.breakout - float(c.breakout_buffer)) /
                                    (float(c.entry_cap) - float(c.breakout_buffer))).clip(0, 1) * s.weight_breakout
    ranked['spread_component'] = (1 - ranked.spread / s.maximum_spread).clip(0, 1) * s.weight_spread
    ranked['score'] = ranked[['rvol_component', 'strength_component', 'breakout_component', 'spread_component']].sum(axis=1)
    ranked = ranked.sort_values(['decision_time_utc', 'score', 'rvol', 'spread', 'symbol'],
                                 ascending=[True, False, False, True, True], kind='stable')
    ranked['rank'] = ranked.groupby('decision_time_utc').cumcount() + 1
    ranked['watchlist'] = ranked['rank'] <= s.watchlist_size
    ranked['ranking_result'] = np.where(ranked.watchlist, 'WATCHLIST_ONLY', 'OUTSIDE_TOP_FIVE')
    return ranked.reset_index(drop=True)


def reservation_test_model(ranked, portfolio, c):
    """Pure synthetic reservation model; not used to imply historical trades."""
    sectors = list(portfolio['occupied_sectors'])
    held = set(portfolio['occupied_symbols'])
    cash, risk = base.D(portfolio['free_cash']), base.D(portfolio['existing_risk'])
    output = []
    for row in ranked[ranked.watchlist].itertuples():
        qty, reason = 0, 'RESERVED_IN_SYNTHETIC_TEST'
        gate = base.entry_gate(portfolio['start_equity'], portfolio['equity'],
            portfolio['confirmed_flows'], sectors, row.sector, portfolio['day_locked'], c)
        if not portfolio['reconciled']:
            reason = 'RECONCILIATION_REQUIRED'
        elif row.symbol in held:
            reason = 'SYMBOL_ALREADY_OCCUPIED'
        elif not gate['allowed']:
            reason = '; '.join(gate['reasons'])
        else:
            size = base.size_position(portfolio['equity'], cash, risk, row.ask, c)
            qty = size['qty']
            if qty == 0:
                reason = 'NO_CAPITAL_OR_RISK_BUDGET'
            else:
                cash -= size['notional']
                risk += size['planned_risk']
                sectors.append(row.sector)
                held.add(row.symbol)
        output.append(dict(symbol=row.symbol, qty=qty, reason=reason,
                           remaining_cash=float(cash), reserved_risk=float(risk)))
    return pd.DataFrame(output)


def scanner_tests():
    c, s = base.Config(), ScannerConfig()
    stamp = pd.Timestamp('2026-09-22 14:00:00Z')
    tests = []
    def check(name, condition):
        tests.append(dict(Bereich='Scanner', Test=name, Bestanden=bool(condition)))
    def quote(age=0, bid=100, ask=100.05):
        return SimpleNamespace(timestamp=stamp-pd.Timedelta(seconds=age), bid_price=bid,
                               ask_price=ask, bid_size=10, ask_size=10)
    check('Frischer enger Spread', quote_check(quote(), stamp, 100.01, 100.3, s)['quote_result'] == 'PASS')
    check('Veraltete Quote', quote_check(quote(age=3), stamp, 100.01, 100.3, s)['quote_result'] == 'QUOTE_STALE')
    check('Quote aus Zukunft', quote_check(quote(age=-1), stamp, 100.01, 100.3, s)['quote_result'] == 'QUOTE_FROM_FUTURE')
    check('Breiter Spread', quote_check(quote(ask=100.2), stamp, 100.01, 100.3, s)['quote_result'] == 'SPREAD_TOO_WIDE')
    check('Briefkurs ueber Preislimit', quote_check(quote(bid=100.4, ask=100.41), stamp, 100.01, 100.3, s)['quote_result'] == 'ASK_OUTSIDE_ENTRY_BAND')
    cutoff = stamp + pd.Timedelta(hours=4)
    check('Spaetstart kauft kein altes Signal', timing_status(stamp, stamp+pd.Timedelta(seconds=1), stamp+pd.Timedelta(seconds=1), cutoff, True, False, s) == 'HISTORICAL_ONLY')
    check('Neues Signal braucht Brokerabgleich', timing_status(stamp, stamp, stamp, cutoff, False, False, s) == 'BROKER_RECONCILIATION_REQUIRED')
    check('Signal bei 60 Sekunden abgelaufen', timing_status(stamp, stamp, stamp+pd.Timedelta(seconds=60), cutoff, True, False, s) == 'EXPIRED')
    check('Kein doppeltes Signal', timing_status(stamp, stamp, stamp, cutoff, True, True, s) == 'DUPLICATE_SIGNAL')
    check('Frisches Signal bleibt ohne Orderfreigabe', timing_status(stamp, stamp, stamp, cutoff, True, False, s) == 'CURRENT_SIGNAL_REQUIRES_ORDER_ENGINE')
    rows = [dict(symbol=symbol, sector='Tech' if i < 3 else 'Other', decision_time_utc=stamp.isoformat(),
                 rvol=2, relative_strength=.005, breakout=.002, spread=.0005, ask=100.05)
            for i, symbol in enumerate('ABCDEF')]
    ranked = rank_candidates(pd.DataFrame(rows[::-1]), c, s)
    check('Sechs gleiche Signale deterministisch / Top 5', list(ranked.symbol) == list('ABCDEF') and ranked.watchlist.sum() == 5)
    portfolio = dict(occupied_sectors=[], occupied_symbols=[], free_cash=10000, existing_risk=0,
                     start_equity=10000, equity=10000, confirmed_flows=0, day_locked=False, reconciled=True)
    reservations = reservation_test_model(ranked, portfolio, c)
    check('Maximal 3 Plaetze und 2 je Sektor', list(reservations[reservations.qty > 0].symbol) == ['A', 'B', 'D'])
    poor = reservation_test_model(ranked, dict(portfolio, free_cash=150), c)
    check('Cash wird zwischen Reservierungen abgezogen', poor.qty.sum() == 1 and poor.remaining_cash.min() >= 0)
    locked = reservation_test_model(ranked, dict(portfolio, day_locked=True), c)
    check('Tagessperre blockiert alle Reservierungen', locked.qty.sum() == 0)
    other = pd.DataFrame([dict(rows[0], symbol='Z', decision_time_utc=(stamp+pd.Timedelta(minutes=1)).isoformat(), rvol=3)])
    mixed = rank_candidates(pd.concat([pd.DataFrame(rows), other]), c, s)
    check('Spaeteres Signal aendert frueheres Ranking nicht', list(mixed[mixed.decision_time_utc == stamp.isoformat()].symbol) == list('ABCDEF'))
    return pd.DataFrame(tests), reservations


def run_scanner(broker, data_client, project_root, c, s, embedded):
    from alpaca.trading.requests import GetCalendarRequest, GetAssetsRequest, GetOrdersRequest
    from alpaca.trading.enums import AssetStatus, AssetClass, QueryOrderStatus
    from alpaca.data.requests import StockBarsRequest, StockQuotesRequest
    from alpaca.data.enums import DataFeed, Adjustment
    from alpaca.data.timeframe import TimeFrame
    from alpaca.common.enums import Sort
    validate_scanner(c, s)
    started = pd.Timestamp.now(tz='UTC')
    out = Path(project_root) / 'v0_3_runs' / (started.strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
    out.mkdir(parents=True, exist_ok=False)
    print('Ergebnisordner:', out)
    sources = {}
    for name, path in [('scanner_source.py', __file__), ('base_source.py', base.__file__)]:
        content = Path(path).read_text(encoding='utf-8')
        (out / name).write_text(content, encoding='utf-8')
        sources[name] = hashlib.sha256(content.encode()).hexdigest()
    base.json_write(out / 'manifest.json', dict(version=VERSION, created_at_utc=started.isoformat(),
        mode='historical_read_only_snapshot_universe', base_config=asdict(c), scanner_config=asdict(s),
        code_hashes=sources, packages={p: importlib.metadata.version(p) for p in ['alpaca-py', 'pandas', 'numpy']},
        membership_is_point_in_time=False, orders_enabled=False,
        assumptions=['Alle neuen Scannerparameter sind unvalidierte Entwurfswerte.',
                     'Aktuelle Indexmitglieder, Sektoren und Broker-Handelbarkeit; keine historischen Stammdaten.',
                     'RAW-Daten; Kapitalmassnahmen und spaetere Datenkorrekturen nicht rekonstruiert.',
                     'Keine Ausfuehrungssimulation oder Performance; keine Portfoliopositionen aus Rankings.']))
    last_request = [0.0]
    def call(method, *args, **kwargs):
        wait = s.request_spacing_seconds - (time.monotonic() - last_request[0])
        if wait > 0:
            time.sleep(wait)
        last_request[0] = time.monotonic()
        return method(*args, **kwargs)
    def fetch_bars(symbols, start, end, timeframe):
        frames = []
        for offset in range(0, len(symbols), s.batch_size):
            result = call(data_client.get_stock_bars, StockBarsRequest(
                symbol_or_symbols=list(symbols[offset:offset+s.batch_size]), timeframe=timeframe,
                start=start.to_pydatetime(), end=end.to_pydatetime(), feed=DataFeed.SIP,
                adjustment=Adjustment.RAW, asof=c.test_date))
            if any(result.data.values()):
                frames.append(base.normalize_bars(result.df))
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=EMPTY_BARS)
    try:
        print('1/6 Referenztests und Scanner-Grenzfaelle …')
        old_tests, stops = base.self_tests()
        new_tests, reservations = scanner_tests()
        tests = pd.concat([old_tests, new_tests], ignore_index=True)
        base.csv_write(out / 'self_tests.csv', tests)
        base.csv_write(out / 'synthetic_stop_paths.csv', stops)
        base.csv_write(out / 'synthetic_reservations.csv', reservations)
        if not tests.Bestanden.all():
            raise ValueError('Referenztest fehlgeschlagen')
        print(f'✅ {len(tests)}/{len(tests)} Referenztests bestanden')
        account = call(broker.get_account)
        clock = call(broker.get_clock)
        positions = call(broker.get_all_positions)
        orders = call(broker.get_orders, filter=GetOrdersRequest(status=QueryOrderStatus.OPEN, limit=500))
        base.json_write(out / 'paper_account_snapshot.json', dict(
            read_at_utc=pd.Timestamp.now(tz='UTC').isoformat(), equity=str(account.equity), cash=str(account.cash),
            currency=account.currency, account_blocked=account.account_blocked,
            trading_blocked=account.trading_blocked, positions_count=len(positions),
            open_orders_returned=len(orders), orders_may_be_truncated=len(orders) >= 500,
            scope='Nur Bestandsaufnahme; kein operativer Brokerabgleich oder Backtest-Portfolio'))
        day = datetime.strptime(c.test_date, '%Y-%m-%d').date()
        calendar = base.calendar_rows(call(broker.get_calendar, GetCalendarRequest(
            start=day-timedelta(days=max(70, c.rvol_days*4)), end=day)))
        session_rows = [r for r in calendar if r['date'] == c.test_date]
        past = [r for r in calendar if r['date'] < c.test_date][-c.rvol_days:]
        if len(session_rows) != 1 or len(past) != c.rvol_days:
            raise ValueError('Handelstag oder 20 Vergleichssitzungen fehlen')
        session = session_rows[0]
        if session['close'] > base.utc(clock.timestamp)-pd.Timedelta(minutes=15):
            raise ValueError('Bitte einen abgeschlossenen Handelstag mit mindestens 15 Minuten Abstand waehlen')
        base.json_write(out / 'calendar.json', dict(session=session, references=past))
        print('2/6 Nasdaq-Mitglieder, Sektoren und aktuelle Alpaca-Handelbarkeit …')
        universe, provenance = load_universe(out, embedded)
        assets = call(broker.get_all_assets, filter=GetAssetsRequest(status=AssetStatus.ACTIVE, asset_class=AssetClass.US_EQUITY))
        tradeable = {a.symbol for a in assets if a.tradable}
        universe['tradable_now'] = universe.symbol.isin(tradeable)
        print(f'✅ {len(universe)} Symbole; Mitgliederstand: {provenance["retrieved_at_utc"]}')
        print('3/6 Liquiditaet aus 20 vorherigen Tageskerzen (VWAP × Volumen) …')
        daily = fetch_bars(list(universe.symbol), past[0]['open'].normalize(), session['open'].normalize(), TimeFrame.Day)
        base.csv_write(out / 'liquidity_daily_sip.csv', daily)
        audit = []
        dates = [r['date'] for r in past]
        for row in universe.itertuples():
            liquidity = liquidity_check(daily, row.symbol, dates, s)
            reason = ('SECTOR_UNKNOWN' if not row.sector else 'NOT_TRADABLE_NOW' if not row.tradable_now else liquidity['liquidity_result'])
            audit.append(dict(symbol=row.symbol, sector=row.sector, tradable_now=row.tradable_now,
                              **liquidity, universe_result=reason))
        audit = pd.DataFrame(audit)
        base.csv_write(out / 'universe_filters.csv', audit)
        eligible = list(audit.loc[audit.universe_result == 'PASS', 'symbol'])
        print(f'✅ {len(eligible)}/{len(universe)} Symbole bestehen den Vorfilter')
        print('4/6 Minutenkurse, Opening Ranges und Ausbrueche …')
        current = fetch_bars(sorted(set(eligible + ['QQQ'])), session['open'], session['close'], TimeFrame.Minute)
        base.csv_write(out / 'session_bars_sip.csv', current)
        range_rows, decision_frames, scan_status = [], [], []
        for symbol_number, symbol in enumerate(eligible, 1):
            if symbol_number == 1 or symbol_number % 10 == 0:
                print(f'  Opening Range und Minutenentscheidungen: {symbol_number}/{len(eligible)}', flush=True)
            try:
                opening_range, decisions = scan_prefix(current, symbol, session['open'], session['close'], c)
            except ValueError:
                scan_status.append(dict(symbol=symbol, scan_result='OPENING_RANGE_DATA_INVALID', price_candidates=0))
                continue
            range_rows.append(opening_range)
            decision_frames.append(decisions)
            scan_status.append(dict(symbol=symbol, scan_result='PREFIX_DATA_GAP' if (decisions.result == 'INCOMPLETE_DATA_PREFIX').any() else 'CHECKED',
                                    price_candidates=int((decisions.result == 'PRICE_CANDIDATE').sum())))
        decisions = pd.concat(decision_frames, ignore_index=True) if decision_frames else pd.DataFrame(columns=['symbol', 'result', 'decision_time_utc'])
        ranges = pd.DataFrame(range_rows, columns=['symbol', 'or_high', 'or_low', 'or_open', 'or_volume'])
        base.csv_write(out / 'opening_ranges.csv', ranges)
        base.csv_write(out / 'minute_decisions.csv', decisions)
        base.csv_write(out / 'scan_status.csv', pd.DataFrame(scan_status, columns=['symbol', 'scan_result', 'price_candidates']))
        candidates = decisions[decisions.result == 'PRICE_CANDIDATE'].copy()
        print(f'5/6 RVOL fuer {len(candidates)} Preisereignisse; Referenzdaten tageweise …')
        history_symbols = sorted(candidates.symbol.unique())
        history_frames = []
        history_dir = out / 'rvol_reference_bars'
        history_dir.mkdir()
        if history_symbols:
            for number, reference in enumerate(past, 1):
                frame = fetch_bars(history_symbols, reference['open'], reference['close'], TimeFrame.Minute)
                base.csv_write(history_dir / (reference['date'] + '.csv'), frame)
                history_frames.append(frame)
                print(f'  RVOL-Referenztag {number}/{len(past)} gespeichert', flush=True)
        history = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame(columns=EMPTY_BARS)
        volume_frames, reference_frames = [], []
        for symbol, events in candidates.groupby('symbol', sort=True):
            symbol_volumes, symbol_references = base.check_rvol(events,
                current[current.symbol == symbol], history[history.symbol == symbol], past, session['open'], c)
            volume_frames.append(symbol_volumes)
            reference_frames.append(symbol_references)
            print(f'  RVOL geprueft: {symbol} ({len(events)} Preisereignisse)', flush=True)
        volumes = pd.concat(volume_frames, ignore_index=True) if volume_frames else pd.DataFrame(columns=base.RVOL_COLUMNS)
        references = pd.concat(reference_frames, ignore_index=True) if reference_frames else pd.DataFrame(columns=['symbol', 'signal_time_utc', 'reference_date', 'volume', 'status'])
        base.csv_write(out / 'volume_candidates.csv', volumes)
        base.csv_write(out / 'volume_reference_checks.csv', references)
        print('6/6 Relative Staerke vs. QQQ, historische SIP-Quotes und Rangfolge …')
        volume_lookup = {(r.Aktie, r.decision_time_utc): r for r in volumes.itertuples()}
        sectors = dict(zip(universe.symbol, universe.sector))
        filtered = []
        for candidate in candidates.itertuples():
            decision = base.utc(candidate.decision_time_utc)
            row = candidate._asdict()
            row.update(sector=sectors[candidate.symbol],
                       signal_id=f'{c.test_date}|{candidate.symbol}|{decision.isoformat()}|ORB_LONG',
                       timing_status=timing_status(decision, started, started,
                           session['close']-pd.Timedelta(minutes=c.entry_cutoff_minutes), False, False, s),
                       rvol=None, relative_strength=None, breakout=candidate.close/candidate.or_high-1,
                       filter_result='PENDING', quote_result='NOT_REQUESTED', spread=None, ask=None)
            volume = volume_lookup[(candidate.symbol, candidate.decision_time_utc)]
            row['rvol'] = volume.RVOL
            if volume.Ergebnis != 'VOLUME_PASS':
                row['filter_result'] = volume.Ergebnis
            else:
                try:
                    qqq = base.checked_window(current, 'QQQ', session['open'], decision)
                    stock = base.checked_window(current, candidate.symbol, session['open'], decision)
                    row['relative_strength'] = float(stock.close.iloc[-1]/stock.open.iloc[0] - qqq.close.iloc[-1]/qqq.open.iloc[0])
                except ValueError:
                    row['filter_result'] = 'BENCHMARK_OR_STOCK_PREFIX_INVALID'
                else:
                    quotes = call(data_client.get_stock_quotes, StockQuotesRequest(
                        symbol_or_symbols=candidate.symbol, start=(decision-pd.Timedelta(seconds=s.quote_search_seconds)).to_pydatetime(),
                        end=decision.to_pydatetime(), sort=Sort.DESC, limit=1, feed=DataFeed.SIP, asof=c.test_date))
                    items = quotes.data.get(candidate.symbol, [])
                    quote = max(items, key=lambda q: base.utc(q.timestamp)) if items else None
                    row.update(quote_check(quote, decision, candidate.threshold, candidate.price_cap, s, c.minimum_price))
                    row['filter_result'] = row['quote_result']
            filtered.append(row)
        filters = pd.DataFrame(filtered) if filtered else pd.DataFrame(columns=['symbol', 'decision_time_utc', 'sector', 'rvol', 'relative_strength', 'breakout', 'spread', 'ask', 'filter_result'])
        base.csv_write(out / 'candidate_filters.csv', filters)
        ranked = rank_candidates(filters[filters.filter_result == 'PASS'], c, s)
        base.csv_write(out / 'ranked_candidates.csv', ranked)
        watchlist = ranked[ranked.watchlist == True].copy()
        base.csv_write(out / 'watchlist_by_minute.csv', watchlist)
        summary = audit.merge(pd.DataFrame(scan_status, columns=['symbol', 'scan_result', 'price_candidates']), on='symbol', how='left')
        counts = filters.loc[filters.filter_result == 'PASS'].groupby('symbol').size()
        summary['scanner_pass_count'] = summary.symbol.map(counts).fillna(0).astype(int)
        summary['price_candidates'] = summary.price_candidates.fillna(0).astype(int)
        base.csv_write(out / 'summary.csv', summary)
        base.json_write(out / 'completed.json', dict(status='completed', tests_passed=len(tests),
            universe_symbols=len(universe), prefilter_pass=len(eligible), price_events=len(candidates),
            scanner_pass=len(ranked), watchlist_events=len(watchlist), orders_sent=0,
            portfolio_simulated=False, finished_at_utc=pd.Timestamp.now(tz='UTC').isoformat()))
        return dict(folder=out, tests=tests, summary=summary, filters=filters, ranked=ranked,
                    watchlist=watchlist, provenance=provenance)
    except Exception as exc:
        status = getattr(exc, 'status_code', None)
        detail = str(exc) if type(exc) is ValueError else 'API-/Laufzeitfehler; siehe Typ und HTTP-Status'
        base.json_write(out / 'failed.json', dict(status='failed', error_type=type(exc).__name__,
            http_status=status, detail=detail, orders_sent=0))
        raise RuntimeError(f'Scanner gestoppt: {detail}; Typ {type(exc).__name__}; HTTP {status}. Protokoll: {out}') from None


In [ ]:
%%writefile /content/orb_portfolio_v15.py
"""Only virtual long positions on one data feed (SIP or IEX); no Alpaca trading client or order methods."""
from collections import Counter
from datetime import timezone
import math

import numpy as np
import pandas as pd
import us_orb_test_v02 as base

BAR_COLUMNS = ['open', 'high', 'low', 'close', 'volume']


def stamp(value):
    return pd.Timestamp(value).tz_convert('UTC')


def clean_bars(current, symbol, start, end):
    """Valid bars of one symbol in [start, end) and the minutes of invalid ones.

    Duplicated or contradictory bars are data errors: only that minute is
    dropped, the symbol stays usable. numpy instead of many small pandas
    operations: this runs for every symbol in every minute."""
    start, end = stamp(start), stamp(end)
    rows = current[(current.symbol == symbol) & (current.timestamp >= start) &
                   (current.timestamp < end)]
    if rows.empty:
        return rows.set_index('timestamp')[BAR_COLUMNS], set()
    o, h, l, c, v = (pd.to_numeric(rows[column], errors='coerce').to_numpy(dtype=float)
                     for column in BAR_COLUMNS)
    with np.errstate(invalid='ignore'):
        valid = (np.isfinite(o) & np.isfinite(h) & np.isfinite(l) & np.isfinite(c) &
                 np.isfinite(v) & (o > 0) & (h > 0) & (l > 0) & (c > 0) & (v >= 0) &
                 (h >= np.maximum(np.maximum(o, c), l)) & (l <= np.minimum(np.minimum(o, c), h)) &
                 ~rows.timestamp.duplicated(keep=False).to_numpy())
    stamps = pd.DatetimeIndex(rows.timestamp)
    good = pd.DataFrame(dict(open=o[valid], high=h[valid], low=l[valid], close=c[valid],
                             volume=v[valid]), index=stamps[valid]).sort_index()
    return good, {stamp(value) for value in stamps[~valid]}


def fresh_bid(quote, when, max_age_seconds=2):
    """Bid of a checked exit quote if it is at most `max_age_seconds` old."""
    if not quote or quote.get('quote_result') != 'PASS' or not quote.get('bid'):
        return None
    age = (stamp(when) - stamp(quote['quote_time_utc'])).total_seconds()
    return float(quote['bid']) if 0 <= age <= max_age_seconds else None


class ShadowPortfolio:
    def __init__(self, config, opening, planned_close, initial_cash=10000,
                 settle_seconds=120, max_gap_minutes=3, feed='sip', wall_clock=None):
        self.c = config
        self.feed = feed
        # market_event_time = `at_utc` (simulated/market time); processing_time = real time
        self.wall_clock = wall_clock or (lambda: pd.Timestamp.now(tz='UTC'))
        self.settle = pd.Timedelta(seconds=settle_seconds)
        self.stop_observation_grace = pd.Timedelta(seconds=15)
        self.max_gap_minutes = max_gap_minutes
        self.opening = stamp(opening)
        self.planned_close = stamp(planned_close)
        self.start_cash = float(initial_cash)
        self.cash = float(initial_cash)
        self.positions = {}
        self.closed = []
        self.events = []
        self.used_symbols = set()
        self.last_marks = {}
        self.last_decision = None
        self.locked = False
        self.data_degraded = False
        self.peak_equity = self.start_cash
        self.max_drawdown = 0.0
        self.last_equity = self.start_cash

    def log(self, kind, when, **fields):
        event = dict(kind=kind, at_utc=stamp(when).isoformat(),
                     processing_utc=self.wall_clock().isoformat(), **fields)
        self.events.append(event)
        return event

    def reject(self, when, symbol, reason):
        self.log('ENTRY_REJECTED', when, symbol=symbol, reason=reason)
        return reason

    def _mark(self, decision):
        if self.positions:
            stale = [s for s in self.positions if self.last_marks.get(s, (None, None))[1] is None]
            if stale:
                self.data_degraded = True
                return None
        equity = self.cash + sum(pos['qty'] * self.last_marks[s][1]
                                 for s, pos in self.positions.items())
        self.last_equity = equity
        self.peak_equity = max(self.peak_equity, equity)
        self.max_drawdown = max(self.max_drawdown, self.peak_equity - equity)
        if equity <= self.start_cash * (1 - float(self.c.daily_loss)):
            if not self.locked:
                self.log('DAILY_LOSS_LOCK', decision, equity=equity)
            self.locked = True
        return equity

    def update(self, decision, bars, now=None):
        """Process completed SIP bars; stop active from the next full minute.

        `last_bar` is the last *settled* minute. A minute without a bar stays
        pending (it may still be delivered) until a later bar exists or it is
        older than `settle`; then it counts as no observed SIP bar: the stop
        is unchanged and the minute is not stop-observed."""
        decision = stamp(decision)
        now = decision if now is None else stamp(now)
        if not bars.empty and ('feed' not in bars or not (bars.feed == self.feed).all()):
            raise ValueError(f'Virtuelles {self.feed.upper()}-Portfolio akzeptiert nur '
                             f'{self.feed.upper()}-Minutenkerzen.')
        if self.last_decision is not None and decision <= self.last_decision:
            raise ValueError('Virtuelle Entscheidungen dürfen nicht rückwärts laufen.')
        if self.last_decision is not None and decision - self.last_decision > pd.Timedelta(minutes=1):
            self.data_degraded = True
            self.log('MISSED_DECISIONS', decision,
                     count=int((decision-self.last_decision)/pd.Timedelta(minutes=1))-1)
        self.last_decision = decision
        minute = pd.Timedelta(minutes=1)
        for symbol, pos in list(self.positions.items()):
            if pos['force_gap_exit']:
                continue  # Await a fresh bid; do not use older candle prices.
            first = pos['last_bar'] + minute
            good, invalid = clean_bars(bars, symbol, first, decision)
            latest = good.index.max() if len(good) else None
            pos['pending_minutes'] = 0
            for when in pd.date_range(first, decision, freq='min', inclusive='left'):
                if when in good.index:
                    row = good.loc[when]
                    if now > when + minute + self.stop_observation_grace:
                        # A bar that surfaces after its decision cannot supply
                        # an executable stop fill at its historical OHLC.
                        self.data_degraded = True
                        pos['stop_coverage_unreliable'] = True
                        pos['needs_gap_quote'] = True
                        pos['unobserved_minutes'] += 1
                        pos['gap_run'] += 1
                        self.log('LATE_HELD_BAR_STOP_NOT_BACKDATED', now,
                                 symbol=symbol, bar_minute_utc=when.isoformat(),
                                 stop_before_gap=pos['stop'],
                                 reported_low=float(row.low))
                        pos['last_bar'] = when
                        if pos['gap_run'] > self.max_gap_minutes:
                            pos['force_gap_exit'] = True
                            break
                        continue
                    old_stop = pos['stop']
                    if float(row.open) <= old_stop:
                        fill, reason = float(row.open), 'STOP_GAP_OPEN_PROXY'
                    elif float(row.low) <= old_stop:
                        fill, reason = old_stop, 'TRAILING_STOP_PROXY'
                    else:
                        fill, reason = None, None
                    if fill is not None:
                        self.close(symbol, now, fill, reason,
                                   trigger_bar_minute=when)
                        break
                    pos['high_water'] = max(pos['high_water'], float(row.high))
                    pos['low_water'] = min(pos['low_water'], float(row.low))
                    pos['stop'] = max(old_stop, pos['high_water']*(1-float(self.c.trailing)))
                    pos['gap_run'] = 0
                    self.last_marks[symbol] = (when, float(row.close))
                    self.log('BAR_REVIEWED', when, symbol=symbol,
                             high_water=pos['high_water'], stop_next_bar=pos['stop'],
                             close_proxy=float(row.close))
                elif when in invalid:
                    self.data_degraded = True
                    pos['stop_coverage_unreliable'] = True
                    pos['needs_gap_quote'] = True
                    pos['unobserved_minutes'] += 1
                    pos['gap_run'] += 1
                    self.log('INVALID_HELD_BAR', when, symbol=symbol)
                elif (latest is not None and when < latest) or now >= when + minute + self.settle:
                    self.data_degraded = True
                    pos['stop_coverage_unreliable'] = True
                    pos['needs_gap_quote'] = True
                    pos['unobserved_minutes'] += 1
                    pos['gap_run'] += 1
                    self.log('NO_SIP_BAR_WHILE_HELD', when, symbol=symbol,
                             consecutive=pos['gap_run'])
                else:
                    pos['pending_minutes'] = int((decision - when) / minute)
                    break
                pos['last_bar'] = when
                if pos['gap_run'] > self.max_gap_minutes:
                    pos['force_gap_exit'] = True
                    break
        self._mark(decision)

    def gap_symbols(self):
        """Held symbols whose latest settled minute had no usable SIP bar."""
        return sorted(s for s, pos in self.positions.items()
                      if pos['gap_run'] > 0 or pos['needs_gap_quote'] or pos['force_gap_exit'])

    def entry_data_blocked(self):
        """Only unresolved stop observations block additional virtual entries.

        `data_degraded` remains the permanent audit flag, not a permanent
        entry lock after a fresh bid or a subsequent good minute resolves a gap.
        """
        return any(pos['pending_minutes'] > 0 or pos['gap_run'] > 0 or
                   pos['needs_gap_quote'] or pos['force_gap_exit']
                   for pos in self.positions.values())

    def check_gap_exits(self, when, bids):
        """Without a SIP bar the stop is checked against a fresh SIP bid; after
        more than `max_gap_minutes` unobserved minutes the position is left."""
        when = stamp(when)
        for symbol in self.gap_symbols():
            pos = self.positions[symbol]
            bid = fresh_bid(bids.get(symbol), when)
            if bid is not None and bid <= pos['stop']:
                self.close(symbol, when, bid, 'STOP_BID_QUOTE_PROXY')
            elif pos['force_gap_exit'] or pos['gap_run'] > self.max_gap_minutes:
                if bid is not None:
                    self.close(symbol, when, bid, 'EXIT_DATA_GAP_BID_PROXY')
                else:
                    self.data_degraded = True
                    self.log('DATA_GAP_EXIT_PRICE_UNAVAILABLE', when, symbol=symbol,
                             consecutive=pos['gap_run'])
            elif bid is not None:
                pos['needs_gap_quote'] = False

    def enter(self, when, signal, quote, signal_ttl_seconds):
        """Use a newly fetched SIP quote at this actual calculation time."""
        when = stamp(when)
        symbol = signal['symbol']
        cutoff = self.planned_close-pd.Timedelta(minutes=
            self.c.entry_cutoff_minutes-self.c.close_before_minutes)
        if when >= cutoff:
            return self.reject(when, symbol, 'ENTRY_CUTOFF')
        if (when-stamp(signal['decision_time_utc'])).total_seconds() >= signal_ttl_seconds:
            return self.reject(when, symbol, 'SIGNAL_EXPIRED')
        if symbol in self.used_symbols:
            return self.reject(when, symbol, 'SYMBOL_ALREADY_USED_TODAY')
        if self.locked:
            return self.reject(when, symbol, 'DAILY_LOSS_LOCK')
        if self.entry_data_blocked():
            return self.reject(when, symbol, 'HELD_POSITION_STOP_OBSERVATION_UNRESOLVED')
        if quote is None or quote.get('quote_result') != 'PASS':
            return self.reject(when, symbol, 'EXECUTION_QUOTE_' + str(
                (quote or {}).get('quote_result', 'MISSING')))
        qtime = stamp(quote['quote_time_utc'])
        if qtime > when or (when-qtime).total_seconds() > 2:
            return self.reject(when, symbol, 'EXECUTION_QUOTE_NOT_FRESH')
        ask = float(quote['ask'])
        if not math.isfinite(ask) or ask <= 0:
            return self.reject(when, symbol, 'INVALID_ENTRY_PRICE')
        if self._mark(when) is None:
            return self.reject(when, symbol, 'PORTFOLIO_MARK_MISSING')
        sector = signal['sector']
        occupied = [pos['sector'] for pos in self.positions.values()]
        gate = base.entry_gate(self.start_cash, self.last_equity, 0,
                               occupied, sector, self.locked, self.c)
        self.locked = self.locked or gate['day_locked']
        if not gate['allowed']:
            return self.reject(when, symbol, ';'.join(gate['reasons']))
        existing_risk = sum(p['qty']*max(0, p['entry']-p['stop'])
                            for p in self.positions.values())
        sizing = base.size_position(self.last_equity, self.cash,
                                    existing_risk, ask, self.c)
        qty = sizing['qty']
        if qty <= 0:
            return self.reject(when, symbol, 'NO_CAPITAL_OR_RISK_BUDGET')
        cost = qty*ask
        self.cash -= cost
        self.used_symbols.add(symbol)
        entry_minute = when.floor('min')
        pos = dict(symbol=symbol, sector=sector, qty=qty, entry=ask,
                   entry_at_utc=when.isoformat(), entry_quote_at_utc=qtime.isoformat(),
                   signal_at_utc=signal['decision_time_utc'], rank=signal['rank'],
                   score=signal['score'], rvol=signal['rvol'],
                   stop=ask*(1-float(self.c.trailing)), high_water=ask, low_water=ask,
                   last_bar=entry_minute, unobserved_minutes=0,
                   gap_run=0, pending_minutes=0, stop_coverage_unreliable=False,
                   needs_gap_quote=False, force_gap_exit=False,
                   entry_minute_not_observed=True)
        self.positions[symbol] = pos
        self.last_marks[symbol] = (entry_minute, ask)
        self.log('VIRTUAL_ENTRY', when, symbol=symbol, sector=sector,
                 qty=qty, ask_proxy=ask, notional=cost, initial_stop=pos['stop'],
                 initial_planned_risk=float(sizing['planned_risk']),
                 signal_rank=pos['rank'], signal_score=pos['score'],
                 quote_at_utc=qtime.isoformat())
        return 'VIRTUAL_ENTRY'

    def close(self, symbol, when, price, reason, trigger_bar_minute=None):
        when = stamp(when)
        pos = self.positions.pop(symbol)
        price = float(price)
        if not math.isfinite(price) or price <= 0:
            raise ValueError('Ungültiger virtueller Ausstiegspreis')
        proceeds = pos['qty']*price
        self.cash += proceeds
        pnl = pos['qty']*(price-pos['entry'])
        # Scenario: adverse 5 bp on both sides, separate from the bid/ask proxy.
        stress = pnl-.0005*pos['qty']*(price+pos['entry'])
        item = dict(**{k: v for k, v in pos.items()
                       if k not in ('last_bar', 'gap_run', 'pending_minutes',
                                    'needs_gap_quote', 'force_gap_exit')},
                    exit_at_utc=when.isoformat(), exit_price_proxy=price,
                    exit_reason=reason, pnl_usd=pnl, pnl_10bp_stress_usd=stress,
                    # excursions over the observed bars while held (entry minute excluded)
                    mfe_usd=pos['qty']*(pos['high_water']-pos['entry']),
                    mae_usd=pos['qty']*(min(pos['low_water'], price)-pos['entry']),
                    exit_trigger_bar_minute_utc=(stamp(trigger_bar_minute).isoformat()
                        if trigger_bar_minute is not None else None),
                    outcome_unreliable=pos['stop_coverage_unreliable'])
        self.closed.append(item)
        self.last_marks.pop(symbol, None)
        self.log('VIRTUAL_EXIT', when, symbol=symbol, qty=pos['qty'],
                 exit_price_proxy=price, reason=reason, pnl_usd=pnl,
                 pnl_10bp_stress_usd=stress,
                 trigger_bar_minute_utc=item['exit_trigger_bar_minute_utc'],
                 outcome_unreliable=item['outcome_unreliable'])
        self._mark(when)
        return item

    def finish(self, when, bids):
        """Close at fresh bid; otherwise a visibly flagged last-bar proxy."""
        when = stamp(when)
        for symbol, pos in list(self.positions.items()):
            bid = fresh_bid(bids.get(symbol), when)
            if bid is not None:
                self.close(symbol, when, bid, 'PLANNED_CLOSE_BID_PROXY')
                continue
            mark = self.last_marks.get(symbol)
            if mark and mark[0] > stamp(pos['entry_at_utc']).floor('min') and \
                    0 <= (when-mark[0]).total_seconds() <= 180:
                self.data_degraded = True
                self.close(symbol, when, mark[1], 'PLANNED_CLOSE_LAST_BAR_PROXY')
            else:
                self.data_degraded = True
                self.log('CLOSE_PRICE_UNAVAILABLE', when, symbol=symbol)
        self._mark(when)

    def report(self):
        rejected = Counter(event['reason'] for event in self.events
                           if event['kind'] == 'ENTRY_REJECTED')
        return dict(mode='VIRTUAL_ONLY_NO_BROKER_ORDERS', orders_sent=0, feed=self.feed,
                    starting_cash_usd=self.start_cash, ending_cash_usd=self.cash,
                    ending_equity_proxy_usd=self.last_equity,
                    realized_pnl_usd=sum(p['pnl_usd'] for p in self.closed),
                    pnl_proxy_from_unreliable_trades_usd=sum(
                        p['pnl_usd'] for p in self.closed if p['outcome_unreliable']),
                    realized_pnl_10bp_stress_usd=sum(p['pnl_10bp_stress_usd']
                                                      for p in self.closed),
                    max_drawdown_proxy_usd=self.max_drawdown,
                    virtual_trades=len(self.closed), open_symbols=sorted(self.positions),
                    unreliable_virtual_trades=sum(p['outcome_unreliable'] for p in self.closed),
                    entry_data_blocked=self.entry_data_blocked(),
                    rejected_signals=dict(rejected), data_degraded=self.data_degraded,
                    assumptions=['Long only; SIP quote ask/bid are hypothetical fills, no fills or partial fills verified.',
                                 'Stop uses the preceding bar stop; opening gap exits at next bar open; trailing high applies from next bar.',
                                 'The partial entry minute and minutes without SIP bars are not stop-observed; '
                                 'such a minute is modeled as no observed SIP trade once a later bar exists or it is settled.',
                                 'Without a SIP bar the stop is checked against a fresh SIP bid; after too many '
                                 'consecutive unobserved minutes the position is left at the fresh bid.',
                                 'A SIP bar first observed over 15 seconds after its decision minute never triggers a backdated stop; affected trades are outcome_unreliable.',
                                 'Separate 10 bp total adverse execution scenario; no commissions, latency or market impact modeled.'])


In [ ]:
%%writefile /content/orb_sim_v15.py
"""Read-only ORB simulation (SIP or IEX), live, delayed or replayed. Contains no order methods."""
from dataclasses import asdict, dataclass
from datetime import timedelta
from pathlib import Path
import hashlib
import json
import math
import sys
import time
import uuid

import numpy as np
import pandas as pd
import us_orb_test_v02 as base
import us_orb_scanner_v03 as scanner
import orb_portfolio_v15 as shadow

VERSION = '1.5-orb-simulation-sip-iex-no-orders'
FEEDS = ('sip', 'iex')
NY = 'America/New_York'


@dataclass(frozen=True)
class DryRunSettings:
    poll_seconds: int = 5
    feed: str = 'sip'                     # intraday bars, quotes and RVOL references: 'sip' or 'iex'
    initial_cash: float = 10000.0         # virtual start equity of the day
    batch_size: int = 20
    historical_control_date: str = '2026-09-22'
    # A missing SIP minute is modeled as no price print; an outage cannot be ruled out.
    min_opening_bars: int = 12            # of 15 opening-range minutes; first one required
    min_coverage: float = 0.85            # observed share of minutes since the open
    max_previous_age_minutes: int = 5     # previous real SIP close for a crossing
    benchmark_max_age_minutes: int = 2    # last real QQQ close before the decision
    bar_settle_seconds: int = 120         # a held minute stays pending this long
    max_held_gap_minutes: int = 3         # consecutive unobserved held minutes
    # RVOL: average over the valid reference sessions if at least this many of the
    # `rvol_days` (20) are valid; V1.4.3 required all 20 (user decision 2026-09-24).
    min_rvol_reference_days: int = 18
    print_every_minute: bool = True       # replay prints only a daily summary
    save_minute_snapshots: bool = True    # session_bars_sip.csv after every minute


class SystemClock:
    """Wall clock of the live run. The replay swaps in a simulated clock."""

    def now(self):
        return pd.Timestamp.now(tz='UTC')

    def sleep(self, seconds):
        time.sleep(seconds)


SYSTEM_CLOCK = SystemClock()


def utc(value):
    return pd.Timestamp(value).tz_convert('UTC')


def save_json(path, content):
    encoded = json.dumps(content, ensure_ascii=False, indent=2, allow_nan=False, default=str)
    temporary = path.with_name(path.name + '.incomplete')
    temporary.write_text(encoded + '\n', encoding='utf-8')
    temporary.replace(path)


def save_frame(path, frame):
    temporary = path.with_name(path.name + '.incomplete')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def restore_frame(path):
    frame = pd.read_csv(path)
    frame['timestamp'] = pd.to_datetime(frame.timestamp, utc=True)
    return frame


def completed_decision(now, delay_seconds=5):
    now = utc(now)
    if now.second < delay_seconds:
        return now.floor('min') - pd.Timedelta(minutes=1)
    return now.floor('min')


def session_start_window(started, opening, closing, c):
    """Return a same-session start state; never invent another trading date."""
    started, opening, closing = map(utc, (started, opening, closing))
    first_decision = opening + pd.Timedelta(minutes=c.opening_minutes + 1)
    cutoff = closing - pd.Timedelta(minutes=c.entry_cutoff_minutes)
    planned_close = closing - pd.Timedelta(minutes=c.close_before_minutes)
    if not first_decision < cutoff < planned_close < closing:
        raise ValueError('Handelssitzung ist für die Strategie zu kurz.')
    if started >= planned_close:
        raise ValueError('Geplanter virtueller Tagesabschluss ist bereits vorbei; '
                         'heute kein sinnvoller Start mehr.')
    if started < opening:
        return 'WAITING_FOR_OPEN'
    if started < first_decision:
        return 'OPENING_RANGE_IN_PROGRESS'
    if started < cutoff:
        return 'LATE_START_PARTIAL_DAY'
    return 'ENTRY_CUTOFF_PASSED_OBSERVATION_ONLY'


def minute_key(decision, symbol):
    return f'{utc(decision):%Y%m%dT%H%M%SZ}_{symbol}.json'


def fetch_bars(data, symbols, start, end, feed, timeframe, batch_size=20):
    if feed not in FEEDS:
        raise ValueError(f'Unbekannter Datenfeed: {feed}')
    from alpaca.data.requests import StockBarsRequest
    from alpaca.data.enums import DataFeed, Adjustment
    from alpaca.data.timeframe import TimeFrame
    mode = data_feed(feed)
    scale = TimeFrame.Minute if timeframe == 'minute' else TimeFrame.Day
    frames = []
    for offset in range(0, len(symbols), batch_size):
        chunk = symbols[offset:offset + batch_size]
        answer = data.get_stock_bars(StockBarsRequest(
            symbol_or_symbols=chunk, timeframe=scale,
            start=utc(start).to_pydatetime(), end=utc(end).to_pydatetime(),
            adjustment=Adjustment.RAW, feed=mode))
        if any(answer.data.values()):
            frame = base.normalize_bars(answer.df)
            # Alpaca may include a bar exactly at the requested end boundary.
            # All requested windows in this notebook are [start, end).
            frame = frame[(frame.timestamp >= utc(start)) &
                          (frame.timestamp < utc(end))].copy()
            if frame.empty:
                continue
            frame['feed'] = feed  # base.normalize_bars labels the older test's SIP source.
            frames.append(frame)
    columns = ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'vwap', 'feed']
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)


def data_feed(feed):
    from alpaca.data.enums import DataFeed
    if feed not in FEEDS:
        raise ValueError(f'Unbekannter Datenfeed: {feed}')
    return DataFeed.SIP if feed == 'sip' else DataFeed.IEX


def check_realtime_sip_access(data, now):
    return check_realtime_access(data, now, 'sip')


def check_realtime_access(data, now, feed='sip'):
    """Explicit recent requests on `feed`; a successful old query proves nothing."""
    now = utc(now)
    start = now - pd.Timedelta(minutes=2)
    recent = fetch_bars(data, ['AAPL'], start, now, feed, 'minute', batch_size=1)
    from alpaca.data.requests import StockQuotesRequest
    from alpaca.data.enums import DataFeed
    from alpaca.common.enums import Sort
    quotes = data.get_stock_quotes(StockQuotesRequest(
        symbol_or_symbols='AAPL', start=start.to_pydatetime(),
        end=now.to_pydatetime(), sort=Sort.DESC, limit=1, feed=data_feed(feed)))
    if not isinstance(quotes.data, dict):
        raise ValueError(f'{feed.upper()}-Quote-Antwort ist ungültig.')
    return dict(result=f'RECENT_{feed.upper()}_BARS_AND_QUOTES_ACCEPTED',
                checked_at_utc=now.isoformat(),
                request_end_within_last_15_minutes=True,
                recent_aapl_bars=len(recent),
                recent_aapl_quotes=len(quotes.data.get('AAPL', [])),
                feed=feed, orders_sent=0,
                note='Außerhalb der Handelszeit können beide Abfragen korrekt null Datensätze liefern.')


def fresh_universe(broker):
    from alpaca.trading.requests import GetAssetsRequest
    from alpaca.trading.enums import AssetStatus, AssetClass
    members = scanner.fetch_json(scanner.MEMBERS_URL)
    sectors = scanner.fetch_json(scanner.SECTORS_URL)
    frame = scanner.assemble_universe(members, sectors)
    assets = broker.get_all_assets(filter=GetAssetsRequest(
        status=AssetStatus.ACTIVE, asset_class=AssetClass.US_EQUITY))
    tradeable = {asset.symbol for asset in assets if asset.tradable}
    frame['tradable_now'] = frame.symbol.isin(tradeable)
    frame['universe_result'] = frame.apply(lambda r:
        'SECTOR_UNKNOWN' if not r.sector else
        'NOT_TRADABLE' if not r.tradable_now else 'PASS', axis=1)
    return frame


def review_opening(symbol, current, opening, decision, closing, c):
    """One historical prefix; do not infer a missing SIP bar from a trade elsewhere."""
    if decision < opening + pd.Timedelta(minutes=c.opening_minutes + 1):
        return 'OPENING_RANGE_IN_PROGRESS', None
    try:
        opening_range, rows = scanner.scan_prefix(current, symbol, opening, closing, c)
    except ValueError:
        return 'OPENING_RANGE_DATA_INVALID', None
    match = rows[rows.decision_time_utc == decision.isoformat()]
    if match.empty:
        return 'DECISION_NOT_RECONSTRUCTED', None
    row = match.iloc[0].to_dict()
    row.update(opening_range)
    return row['result'], row


def minute_series(current, symbol, opening, decision):
    """One row per minute in [opening, decision); missing minutes stay NaN.

    `observed` marks a real SIP bar, `volume` counts a missing minute as 0,
    `last_close`/`last_seen` carry the latest real SIP print forward."""
    grid = pd.date_range(utc(opening), utc(decision), freq='min', inclusive='left')
    good, invalid = shadow.clean_bars(current, symbol, opening, decision)
    size = len(grid)
    columns = {name: np.full(size, np.nan) for name in ('open', 'high', 'low', 'close', 'volume')}
    observed = np.zeros(size, dtype=bool)
    positions = grid.get_indexer(good.index)
    on_grid = positions >= 0          # off-grid timestamps are ignored, as before
    for name in columns:
        columns[name][positions[on_grid]] = good[name].to_numpy(dtype=float)[on_grid]
    observed[positions[on_grid]] = True
    flagged = np.zeros(size, dtype=bool)
    bad = grid.get_indexer(pd.DatetimeIndex(sorted(invalid))) if invalid else np.array([], dtype=int)
    flagged[bad[bad >= 0]] = True
    seen = np.maximum.accumulate(np.where(observed, np.arange(size), -1)) if size else np.array([], dtype=int)
    series = pd.DataFrame(columns, index=grid)
    series['observed'] = observed
    series['invalid'] = flagged
    series['volume'] = np.nan_to_num(columns['volume'], nan=0.0)
    series['last_close'] = np.where(seen >= 0, columns['close'][np.maximum(seen, 0)], np.nan)
    series['last_seen'] = grid[np.maximum(seen, 0)].where(seen >= 0) if size else grid
    series['coverage'] = np.cumsum(observed) / np.arange(1, size + 1)
    return series


def opening_range(series, opening, c, settings):
    """Opening range from the observed bars; None if coverage is too thin."""
    end_or = utc(opening) + pd.Timedelta(minutes=c.opening_minutes)
    part = series[series.index < end_or]
    observed = part[part.observed]
    if len(part) < c.opening_minutes or not part.observed.iloc[0] or \
            len(observed) < settings.min_opening_bars:
        return None
    high, low = base.D(observed.high.max()), base.D(observed.low.min())
    return dict(high=high, low=low, threshold=high * (1 + base.D(c.breakout_buffer)),
                cap=high * (1 + base.D(c.entry_cap)), bars=len(observed),
                last_close=base.D(observed.close.iloc[-1]), last_seen=observed.index[-1])


def split_by_symbol(frame):
    """Per-symbol views of a multi-symbol bar frame; avoids rescanning it per symbol."""
    empty = frame.iloc[0:0]
    parts = {symbol: part for symbol, part in frame.groupby('symbol', sort=False)}
    return lambda symbol: parts.get(symbol, empty)


def price_candidates_until(current, symbol, opening, decision, c, settings=DryRunSettings()):
    """Price crossings on the SIP minute grid; missing minutes carry the last print.

    A crossing needs a real bar in the signal minute, a previous real close no
    older than `max_previous_age_minutes` and enough coverage since the open.
    Returns (candidates, opening_invalid, later_data_issue)."""
    opening, decision = utc(opening), utc(decision)
    end_or = opening + pd.Timedelta(minutes=c.opening_minutes)
    if decision <= end_or:
        return [], False, False
    series = minute_series(current, symbol, opening, decision)
    rng = opening_range(series, opening, c, settings)
    if rng is None:
        return [], True, False
    later_issue = bool(series.invalid.any())
    max_age = pd.Timedelta(minutes=settings.max_previous_age_minutes)
    later = series[series.index >= end_or]
    observed = later.observed.to_numpy()
    moments = later.index[observed]
    closes = later.close.to_numpy()[observed]
    coverages = later.coverage.to_numpy()[observed]
    previous_closes = np.concatenate([[float(rng['last_close'])], closes[:-1]])
    # Exact Decimal rule below; this float pre-filter only skips minutes that are
    # clearly no crossing (margin far above float rounding).
    threshold = float(rng['threshold'])
    margin = abs(threshold) * 1e-9
    candidates = []
    for k in np.flatnonzero((closes > threshold - margin) & (previous_closes <= threshold + margin)):
        moment = moments[k]
        close = base.D(closes[k])
        previous = rng['last_close'] if k == 0 else base.D(closes[k - 1])
        previous_seen = rng['last_seen'] if k == 0 else moments[k - 1]
        if previous <= rng['threshold'] and close > rng['threshold'] and \
                close >= base.D(c.minimum_price) and close <= rng['cap']:
            if moment - previous_seen > max_age or coverages[k] < settings.min_coverage:
                later_issue = True
            else:
                candidates.append(dict(symbol=symbol,
                    decision_time_utc=(moment + pd.Timedelta(minutes=1)).isoformat(),
                    or_high=float(rng['high']), or_low=float(rng['low']),
                    threshold=float(rng['threshold']), price_cap=float(rng['cap']),
                    close=float(close), opening_bars=rng['bars'],
                    coverage=float(coverages[k]),
                    previous_close_age_minutes=int((moment - previous_seen) / pd.Timedelta(minutes=1))))
    return candidates, False, later_issue


def exit_quotes(data, symbols, clock=SYSTEM_CLOCK, feed='sip'):
    """Fresh SIP bid per symbol, each checked at its own request time."""
    bids = {}
    for symbol in sorted(symbols):
        quote_now = clock.now()
        try:
            bids[symbol] = exit_quote_quality(fresh_quote(data, symbol, quote_now, feed=feed), quote_now)
        except Exception as exc:
            bids[symbol] = dict(quote_result='ERROR', error_type=type(exc).__name__)
    return bids


def check_gap_exits(data, portfolio, clock=SYSTEM_CLOCK, feed='sip'):
    """Held symbols whose last minute had no SIP bar: stop via the fresh bid,
    or leave after too many consecutive unobserved minutes."""
    symbols = portfolio.gap_symbols()
    if symbols:
        bids = exit_quotes(data, symbols, clock, feed)
        portfolio.check_gap_exits(clock.now(), bids)


def account_snapshot(broker, clock=SYSTEM_CLOCK):
    from alpaca.trading.requests import GetOrdersRequest
    from alpaca.trading.enums import QueryOrderStatus
    account = broker.get_account()
    positions = broker.get_all_positions()
    orders = broker.get_orders(filter=GetOrdersRequest(status=QueryOrderStatus.OPEN, limit=500))
    if len(orders) == 500:
        raise ValueError('Offene Orderliste könnte abgeschnitten sein.')
    return dict(equity=float(account.equity), cash=float(account.cash),
        currency=account.currency, account_blocked=bool(account.account_blocked),
        trading_blocked=bool(account.trading_blocked),
        position_symbols=[p.symbol for p in positions],
        open_order_ids=[str(o.id) for o in orders],
        checked_at_utc=clock.now().isoformat())


def fresh_quote(data, symbol, at, search_seconds=10, feed='sip'):
    from alpaca.data.requests import StockQuotesRequest
    from alpaca.data.enums import DataFeed
    from alpaca.common.enums import Sort
    at = utc(at)
    response = data.get_stock_quotes(StockQuotesRequest(
        symbol_or_symbols=symbol,
        start=(at-pd.Timedelta(seconds=search_seconds)).to_pydatetime(),
        end=at.to_pydatetime(), sort=Sort.DESC, limit=1, feed=data_feed(feed)))
    quotes = response.data.get(symbol, [])
    return max(quotes, key=lambda q: utc(q.timestamp)) if quotes else None


def exit_quote_quality(quote, at):
    if quote is None:
        return dict(quote_result='MISSING')
    try:
        bid, ask = float(quote.bid_price), float(quote.ask_price)
        bid_size = float(quote.bid_size)
        at = utc(at)
        qtime = utc(quote.timestamp)
        if not all(math.isfinite(x) for x in (bid, ask, bid_size)) or \
                min(bid, ask, bid_size) <= 0 or bid > ask or \
                qtime > at or (at-qtime).total_seconds() > 2:
            return dict(quote_result='INVALID_OR_STALE')
        return dict(quote_result='PASS', bid=bid, ask=ask,
                    quote_time_utc=qtime.isoformat())
    except (AttributeError, TypeError, ValueError):
        return dict(quote_result='INVALID_OR_STALE')


def prepare_reference(data, symbol, past, out, settings):
    """Fetch the 20-day SIP history through the data client, then cache each session."""
    folder = out / f'{settings.feed}_reference_minutes' / symbol
    folder.mkdir(parents=True, exist_ok=True)
    dates = {session['date'] for session in past}
    cached = {path.stem for path in folder.glob('*.csv')}
    if cached != dates:
        # A partial cache is never treated as complete. alpaca-py pages the
        # request itself, so live and replay use the same data client.
        frame = fetch_bars(data, [symbol], past[0]['open'], past[-1]['close'],
                           settings.feed, 'minute', settings.batch_size)
        if not frame.empty and frame.timestamp.duplicated().any():
            raise ValueError('Doppelte historische SIP-Kerzen.')
        prepared = {}
        for session in past:
            day = frame[(frame.timestamp >= session['open']) &
                        (frame.timestamp < session['close'])].copy()
            prepared[session['date']] = day
        for session in past:
            save_frame(folder / (session['date'] + '.csv'), prepared[session['date']])
        save_json(out / f'{settings.feed}_reference_minutes' / f'{symbol}_download.json',
                  dict(symbol=symbol, date_first=past[0]['date'],
                       date_last=past[-1]['date'], feed=settings.feed,
                       returned_bars=len(frame), cached_regular_bars=sum(map(len, prepared.values()))))
    frames = []
    for session in past:
        frame = restore_frame(folder / (session['date'] + '.csv'))
        if not frame.empty and (set(frame.symbol) != {symbol} or
                (frame.timestamp < session['open']).any() or
                (frame.timestamp >= session['close']).any() or
                frame.timestamp.duplicated().any()):
            raise ValueError('Gespeicherter SIP-Vergleichstag ist widersprüchlich.')
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def missing_odd_lot_volume(data, symbol, minute, out, feed='sip'):
    """Fetch one absent SIP bar's trade volume; never fabricate OHLC."""
    from alpaca.data.requests import StockTradesRequest
    from alpaca.data.enums import DataFeed
    folder = out / f'odd_lot_reference_minutes_{feed}' / symbol
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f'{utc(minute):%Y%m%dT%H%M%SZ}.json'
    if path.exists():
        saved = json.loads(path.read_text())
        if saved['symbol'] != symbol or saved['minute_utc'] != utc(minute).isoformat():
            raise ValueError('Odd-Lot-Cache gehört zu anderem Symbol/Minute.')
        return saved['volume']
    start, end = utc(minute), utc(minute) + pd.Timedelta(minutes=1)
    response = data.get_stock_trades(StockTradesRequest(
        symbol_or_symbols=symbol, start=start.to_pydatetime(),
        end=end.to_pydatetime(), feed=data_feed(feed)))
    trades = response.data.get(symbol, [])
    volume = 0
    for trade in trades:
        stamp = utc(trade.timestamp)
        size = float(trade.size)
        if not start <= stamp < end or not size > 0 or size != int(size):
            raise ValueError(f'Trade außerhalb Minute oder ungültige Stückzahl: fehlende Minute '
                             f'{start:%Y-%m-%d %H:%M} UTC, Trade {stamp.isoformat()} Stück {size}.')
        if set(trade.conditions or []) != {'@', 'I'}:
            raise ValueError(f'Fehlende Kerze {start:%Y-%m-%d %H:%M} UTC nicht aus Odd-Lots erklärbar: '
                             f'Trade {stamp.isoformat()} Stück {size:g} Bedingungen '
                             f'{sorted(trade.conditions or [])} (erwartet nur @,I).')
        volume += int(size)
    save_json(path, dict(symbol=symbol, minute_utc=start.isoformat(),
                         volume=volume, trade_count=len(trades),
                         method=f'{feed.upper()} odd-lot trade sizes for absent reference bar'))
    return volume


def checked_reference_volume(data, frame, symbol, start, end, out, feed='sip'):
    expected = pd.date_range(start, end, freq='min', inclusive='left')
    observed = frame[(frame.symbol == symbol) & (frame.timestamp >= start) &
                     (frame.timestamp < end)].sort_values('timestamp')
    if observed.timestamp.duplicated().any():
        twice = observed.timestamp[observed.timestamp.duplicated()].iloc[0]
        raise ValueError(f'Doppelte Referenzkerzen, z. B. {twice:%Y-%m-%d %H:%M} UTC.')
    if not observed.empty:
        base.checked_window(observed, symbol, observed.timestamp.iloc[0],
                            observed.timestamp.iloc[0] + pd.Timedelta(minutes=1))
        values = observed[['open', 'high', 'low', 'close', 'volume']]
        valid = (values.notna().all(axis=1) & (values.volume >= 0) &
                 (values[['open','high','low','close']] > 0).all(axis=1) &
                 (values.high >= values[['open','close','low']].max(axis=1)) &
                 (values.low <= values[['open','close','high']].min(axis=1)))
        if not valid.all():
            bad = observed[~valid].iloc[0]
            raise ValueError(f'Ungültige Referenzkerze {bad.timestamp:%Y-%m-%d %H:%M} UTC: '
                             f'O {bad.open} H {bad.high} L {bad.low} C {bad.close} V {bad.volume} '
                             f'({int((~valid).sum())} ungültige Kerzen im Fenster).')
    gaps = expected.difference(pd.DatetimeIndex(observed.timestamp))
    if len(gaps) > 40:
        raise ValueError('Mehr als 40 Referenzlücken; keine automatische Rekonstruktion.')
    extra = sum(missing_odd_lot_volume(data, symbol, minute, out, feed)
                for minute in gaps)
    return float(observed.volume.sum() + extra), len(gaps), float(extra)


def check_sip_rvol(data, symbol, current, history, past, opening, decision,
                   config, out, feed='sip', min_reference_days=None):
    """RVOL against references of the same feed (named for SIP, used for both)."""
    elapsed = decision - opening
    # Model an absent SIP bar as zero observed volume; distinguish outages separately.
    actual = float(minute_series(current, symbol, opening, decision).volume.sum())
    rows, volumes = [], []
    for session in past:
        start, end = session['open'], session['open'] + elapsed
        value, gaps, extra, status, detail = None, None, None, 'OK', None
        if end > session['close']:
            status = 'REFERENCE_SESSION_TOO_SHORT'
        else:
            try:
                value, gaps, extra = checked_reference_volume(
                    data, history, symbol, start, end, out, feed)
                if gaps:
                    status = 'ODD_LOT_VOLUME_RECOVERED'
            except Exception as exc:
                status = 'REFERENCE_UNVERIFIED_' + type(exc).__name__
                detail = str(exc)[:500]
        rows.append(dict(symbol=symbol, signal_time_utc=decision.isoformat(),
                         reference_date=session['date'], volume=value,
                         recovered_gap_minutes=gaps, recovered_volume=extra,
                         status=status, error_detail=detail))
        if value is not None:
            volumes.append(value)
    needed = config.rvol_days if min_reference_days is None else min(min_reference_days, config.rvol_days)
    average = sum(volumes) / len(volumes) if len(volumes) >= needed and volumes else None
    rvol = actual / average if average is not None and average > 0 else None
    verdict = ('NOT_CHECKABLE' if rvol is None else
               'VOLUME_PASS' if rvol >= config.minimum_rvol else 'VOLUME_REJECT')
    return dict(Ergebnis=verdict, RVOL=rvol, Vollständige_Vergleichstage=len(volumes),
                Volumen_bis_Signal=actual, Durchschnitt_20_Tage=average), pd.DataFrame(rows)


def series_endpoints(current, symbol, opening, decision, max_age_minutes):
    """Opening open and latest real close; the close may be up to
    `max_age_minutes` old when the last SIP minutes had no trade."""
    series = minute_series(current, symbol, opening, decision)
    if series.empty or not series.observed.iloc[0]:
        raise ValueError(f'{symbol}: erste SIP-Kerze fehlt oder ist ungültig.')
    # A bar starting at t closes at t + 1 min; the age is measured from that close.
    last_closed = series.last_seen.iloc[-1] + pd.Timedelta(minutes=1)
    if utc(decision) - last_closed > pd.Timedelta(minutes=max_age_minutes):
        raise ValueError(f'{symbol}: letzter SIP-Kurs ist älter als {max_age_minutes} Minuten.')
    return (float(series.open.iloc[0]), float(series.last_close.iloc[-1]),
            int((~series.observed).sum()))


def checked_qqq_endpoints(current, opening, decision, max_age_minutes=2):
    """Relative strength uses only the opening open and last real close."""
    return series_endpoints(current, 'QQQ', opening, decision, max_age_minutes)


def prefix_diagnostic(current, symbol, opening, decision):
    """Persist the data actually seen at a decision; no later backfill."""
    window = current[(current.symbol == symbol) &
                     (current.timestamp >= opening) &
                     (current.timestamp < decision)].sort_values('timestamp')
    expected = pd.date_range(opening, decision, freq='min', inclusive='left')
    missing = expected.difference(pd.DatetimeIndex(window.timestamp))
    return dict(symbol=symbol, bars_seen=len(window), expected_bars=len(expected),
                missing_minutes_utc=[stamp.isoformat() for stamp in missing[:20]],
                total_missing=len(missing),
                duplicate_timestamps=int(window.timestamp.duplicated().sum()),
                first_time_utc=window.timestamp.iloc[0].isoformat() if len(window) else None,
                last_time_utc=window.timestamp.iloc[-1].isoformat() if len(window) else None)


def quote_or_ranking_expired(item, decision, ttl, stage, clock=SYSTEM_CLOCK):
    age = (clock.now() - decision).total_seconds()
    item['signal_age_seconds'] = age
    if age >= ttl:
        item['status'] = f'EXPIRED_{stage}'
        return True
    return False


def inspect_signal(data, symbol, candidate, current, past, opening,
                   out, c, s, settings, refs, last_live_decision, clock=SYSTEM_CLOCK):
    decision = utc(candidate['decision_time_utc'])
    item = dict(symbol=symbol, decision_time_utc=decision.isoformat(),
                decision_time_ny=decision.tz_convert(NY).isoformat(),
                feed=settings.feed, decision_data_complete=True,
                processing_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                or_high=candidate['or_high'], or_low=candidate['or_low'],
                threshold=candidate['threshold'], price_cap=candidate['price_cap'],
                close=candidate['close'], status='PENDING', rvol=None,
                relative_strength=None, spread=None, bid=None, ask=None,
                score=None, planned_qty=None, orders_sent=0)
    if decision <= last_live_decision:
        item['status'] = 'HISTORICAL_ONLY_STARTUP_OR_RESTART'
        return item
    now = clock.now()
    if now - decision >= pd.Timedelta(seconds=s.signal_ttl_seconds):
        item['status'] = 'EXPIRED_LIVE_DECISION'
        return item
    item.update(opening_bars=candidate.get('opening_bars'),
                coverage=candidate.get('coverage'),
                previous_close_age_minutes=candidate.get('previous_close_age_minutes'))
    try:
        stock_open, stock_close, stock_missing = series_endpoints(
            current, symbol, opening, decision, 0)  # signal bar must be real
        qqq_open, qqq_close, qqq_missing = checked_qqq_endpoints(
            current, opening, decision, settings.benchmark_max_age_minutes)
        item['stock_missing_minutes'] = stock_missing
        item['benchmark_missing_internal_minutes'] = qqq_missing
    except ValueError as exc:
        item['status'] = 'BENCHMARK_OR_STOCK_PREFIX_INVALID'
        item['decision_data_complete'] = False
        item['data_error'] = str(exc)
        item['data_diagnostic'] = {name: prefix_diagnostic(current, name, opening, decision)
                                   for name in (symbol, 'QQQ')}
        snapshot = current[(current.symbol.isin([symbol, 'QQQ'])) &
                           (current.timestamp >= opening) & (current.timestamp < decision)]
        save_frame(out / 'signal_market_snapshots' /
                   minute_key(decision, symbol).replace('.json', '.csv'), snapshot)
        return item
    try:
        if symbol not in refs:
            refs[symbol] = prepare_reference(data, symbol, past, out, settings)
        result, ref_check = check_sip_rvol(data, symbol, current,
            refs[symbol], past, opening, decision, c, out, settings.feed,
            settings.min_rvol_reference_days)
    except Exception as exc:
        item.update(status='RVOL_DATA_ERROR', error_type=type(exc).__name__)
        return item
    item.update(rvol_status=result['Ergebnis'], rvol=float(result['RVOL']) if pd.notna(result['RVOL']) else None,
                complete_reference_days=int(result['Vollständige_Vergleichstage']),
                current_volume=float(result['Volumen_bis_Signal']),
                reference_average=float(result['Durchschnitt_20_Tage']) if pd.notna(result['Durchschnitt_20_Tage']) else None)
    save_frame(out / 'reference_checks' / minute_key(decision, symbol).replace('.json', '.csv'), ref_check)
    if result['Ergebnis'] != 'VOLUME_PASS':
        item['status'] = result['Ergebnis']
        return item
    item['relative_strength'] = float(stock_close / stock_open - qqq_close / qqq_open)
    if quote_or_ranking_expired(item, decision, s.signal_ttl_seconds, 'DURING_RVOL_CHECK', clock):
        return item
    try:
        from alpaca.data.requests import StockQuotesRequest
        from alpaca.data.enums import DataFeed
        from alpaca.common.enums import Sort
        response = data.get_stock_quotes(StockQuotesRequest(
            symbol_or_symbols=symbol,
            start=(decision - pd.Timedelta(seconds=s.quote_search_seconds)).to_pydatetime(),
            end=decision.to_pydatetime(), sort=Sort.DESC, limit=1, feed=data_feed(settings.feed)))
        quotes = response.data.get(symbol, [])
        quote = max(quotes, key=lambda q: base.utc(q.timestamp)) if quotes else None
        quality = scanner.quote_check(quote, decision, item['threshold'],
                                      item['price_cap'], s, c.minimum_price)
    except Exception as exc:
        item.update(status='QUOTE_DATA_ERROR', error_type=type(exc).__name__)
        return item
    item.update(quality)
    if quote_or_ranking_expired(item, decision, s.signal_ttl_seconds, 'DURING_QUOTE_CHECK', clock):
        return item
    item['status'] = quality['quote_result']
    if quality['quote_result'] == 'PASS':
        item['breakout'] = item['close'] / item['or_high'] - 1
        item['status'] = 'SCANNER_PASS_NO_ORDER'
    return item


def run_dryrun(broker, data, project_root, c, s, settings=DryRunSettings(),
               clock=SYSTEM_CLOCK, universe_loader=fresh_universe, replay=None,
               cockpit=None, order_hook=None, output_folder=None, cockpit_mode=None):
    """One trading day on `settings.feed`. Live: wall clock and real-time data.
    Replay/delayed: `clock`, `broker` and `data` come from orb_replay_v15 and
    serve only data already published at the simulated time."""
    feed = settings.feed
    if feed not in FEEDS:
        raise ValueError(f'Unbekannter Datenfeed: {feed}')
    if not 1 <= settings.min_rvol_reference_days <= c.rvol_days:
        raise ValueError('min_rvol_reference_days muss zwischen 1 und rvol_days liegen.')
    if order_hook is not None and replay:
        # Replayed or delayed decisions must never become broker orders.
        raise ValueError('Broker-Orders sind nur im Echtzeitbetrieb zulässig.')
    from alpaca.trading.requests import GetCalendarRequest
    if not 1 <= settings.poll_seconds <= 15:
        raise ValueError('Ungültige Abfragefrequenz.')
    scanner.validate_scanner(c, s)
    started = clock.now()
    broker_clock = broker.get_clock()
    broker_time = utc(broker_clock.timestamp)
    if abs((broker_time - started).total_seconds()) > 120:
        raise ValueError('Broker-Uhr und Colab-Uhr weichen zu stark ab.')
    date_ny = broker_time.tz_convert(NY).date()
    calendar = base.calendar_rows(broker.get_calendar(GetCalendarRequest(
        start=date_ny-timedelta(days=70), end=date_ny)))
    dates = [r for r in calendar if r['date'] == str(date_ny)]
    past = [r for r in calendar if r['date'] < str(date_ny)][-c.rvol_days:]
    if len(dates) != 1 or len(past) != c.rvol_days:
        raise ValueError('Handelstag oder 20 vorherige Sitzungen fehlen.')
    session = dates[0]
    opening, closing = session['open'], session['close']
    cutoff = closing - pd.Timedelta(minutes=c.entry_cutoff_minutes)
    start_window = session_start_window(started, opening, closing, c)

    out = Path(project_root) / (output_folder or (replay or {}).get('output_folder', 'shadow_replay_v1_5')
                                if (output_folder or replay) else 'shadow_sim_v1_5') / session['date'] / (
        started.strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
    out.mkdir(parents=True, exist_ok=False)
    (out / 'signals').mkdir()
    (out / 'reference_checks').mkdir()
    (out / 'signal_market_snapshots').mkdir()
    portfolio = shadow.ShadowPortfolio(c, opening,
        closing - pd.Timedelta(minutes=c.close_before_minutes), initial_cash=settings.initial_cash,
        feed=feed,
        settle_seconds=settings.bar_settle_seconds,
        max_gap_minutes=settings.max_held_gap_minutes)
    broker_start = None
    eligible_count = 0
    cockpit_error = None
    last_price_bar = None
    def paint(stage, decision=None, signals=()):
        nonlocal cockpit_error
        if cockpit is None or cockpit_error is not None:
            return
        try:
            cockpit.update(out, portfolio, clock.now(), decision=decision,
                stage=stage, mode=cockpit_mode or (replay.get('cockpit_mode', 'replay') if replay else 'live'),
                clock_lag_seconds=getattr(clock, 'lag_seconds', 0),
                eligible_count=eligible_count, last_price_bar=last_price_bar,
                signals=signals, broker_start=broker_start)
        except Exception as exc:
            cockpit_error = type(exc).__name__
            portfolio.log('COCKPIT_ERROR', clock.now(), error_type=cockpit_error)
            print('Cockpit konnte nicht aktualisiert werden:', cockpit_error)
    portfolio_closed = False
    print('Simulations-Ergebnisordner:', out)
    status = 'INCOMPLETE'
    session_scope = 'NOT_READY'
    ready_at = None
    try:
        save_json(out / 'manifest.json', dict(version=VERSION, created_at_utc=started.isoformat(),
            mode=(replay.get('mode', 'REPLAY_VIRTUAL_PORTFOLIO') if replay
                  else 'FLEXIBLE_START_VIRTUAL_PORTFOLIO'),
            replay=replay, orders_enabled=False, orders_sent=0,
            start_window=start_window, session_open_utc=opening.isoformat(),
            session_close_utc=closing.isoformat(),
            live_feed=feed, reference_volume_feed=feed,
            prior_day_liquidity_feed='sip_delayed_historical_only',
            strategy=asdict(c), scanner=asdict(s), settings=asdict(settings),
            code_sha256={Path(module.__file__).name: hashlib.sha256(
                Path(module.__file__).read_bytes()).hexdigest()
                for module in ((base, scanner, shadow, sys.modules[__name__]) +
                    tuple(module for module in (sys.modules.get('orb_cockpit_v15'),)
                          if module is not None))},
            virtual_start_cash_usd=settings.initial_cash,
            important='Virtuelle Transaktionen: keine Brokerorders oder echten Ausführungen.'))
        snapshot = account_snapshot(broker, clock)
        broker_start = snapshot
        save_json(out / 'broker_snapshot_start.json', snapshot)
        print('Paper-Kontoequity:', snapshot['equity'], 'USD; Cash:', snapshot['cash'], 'USD')
        if snapshot['currency'] != 'USD' or snapshot['account_blocked'] or snapshot['trading_blocked']:
            raise ValueError('Paper-Konto ist gesperrt oder nicht in USD.')
        if order_hook is not None:
            order_hook.on_start(out, clock)   # own reconciliation; may disable itself
        elif snapshot['position_symbols'] or snapshot['open_order_ids']:
            raise ValueError('Paper-Konto hat bestehende Positionen/Orders; Dry-Run setzt einen leeren Brokerstatus voraus.')
        if replay:
            # Historical SIP is served by the replay store; the real-time check and
            # the fixed control day are live-only (the control day may lie in the
            # simulated future).
            save_json(out / f'{feed}_access_check.json', dict(
                result=replay.get('access_result', 'REPLAY_HISTORICAL_SIP'), feed=feed, orders_sent=0))
        else:
            try:
                sip_access = check_realtime_access(data, clock.now(), feed)
            except Exception as exc:
                status = f'{feed.upper()}_ACCESS_CHECK_FAILED_NO_ORDERS'
                save_json(out / f'{feed}_access_check.json', dict(
                    result=f'RECENT_{feed.upper()}_UNAVAILABLE_OR_API_ERROR',
                    error_type=type(exc).__name__, feed=feed, orders_sent=0,
                    note='Kein Rückgriff auf einen anderen oder verzögerten Feed. Datenberechtigung und Verbindung prüfen.'))
                raise RuntimeError(f'Echtzeit-{feed.upper()} für aktuelle Minutenkerzen und Quotes '
                    'nicht bestätigt. Simulation sicher gestoppt; keine Brokerorders.') from exc
            save_json(out / f'{feed}_access_check.json', sip_access)
            print(f'Aktuelle {feed.upper()}-Minutenkerzen und Quotes: Zugriff bestätigt.')
            control_day = pd.Timestamp(settings.historical_control_date + ' 09:30', tz=NY).tz_convert('UTC')
            control = fetch_bars(data, ['AAPL', 'MSFT', 'NVDA'], control_day,
                control_day + pd.Timedelta(minutes=15), feed, 'minute', settings.batch_size)
            control_count = {stock: int((control.symbol == stock).sum()) for stock in ('AAPL', 'MSFT', 'NVDA')}
            save_json(out / f'{feed}_control.json', dict(date=settings.historical_control_date,
                opening_minutes=15, bars_per_symbol=control_count, feed=feed))
            print(f'{feed.upper()}-Kontrollfenster 15 Min:', control_count)
            if any(count != 15 for count in control_count.values()):
                raise ValueError('15-Minuten-Kontrollfenster ist nicht exakt vollständig.')
        universe = universe_loader(broker)  # Live: no undated or cached index-member fallback.
        save_frame(out / 'universe_snapshot.csv', universe)
        print('Nasdaq-100-Kandidaten:', len(universe))
        last_past_close = past[-1]['close'] + pd.Timedelta(minutes=1)
        first_past_start = past[0]['open'].normalize()
        daily = fetch_bars(data, list(universe.symbol), first_past_start,
                           last_past_close, 'sip', 'day', settings.batch_size)
        # Prior-day liquidity prefilter stays on historical SIP for every feed:
        # both bots start from the same stock universe.
        save_frame(out / 'liquidity_daily_historical_sip.csv', daily)
        eligible = []
        reasons = []
        for stock in universe.itertuples():
            liquid = scanner.liquidity_check(daily, stock.symbol, [p['date'] for p in past], s)
            verdict = stock.universe_result if stock.universe_result != 'PASS' else liquid['liquidity_result']
            reasons.append(dict(symbol=stock.symbol, sector=stock.sector,
                                average_dollar_volume=liquid['average_dollar_volume'], result=verdict))
            if verdict == 'PASS':
                eligible.append(stock.symbol)
        save_frame(out / 'universe_filter_results.csv', pd.DataFrame(reasons))
        print(f'{len(eligible)} Aktien passieren Universums- und historischen Liquiditätsfilter')
        if not eligible:
            raise ValueError('Keine zulässigen Aktien; Dry-Run stoppt sicher.')
        eligible_count = len(eligible)
        symbols = sorted(set(eligible + ['QQQ']))
        refs = {}
        seen_decisions = set()
        ready_at = clock.now()
        if ready_at >= portfolio.planned_close:
            session_scope = 'NO_SIMULATION_WINDOW_AFTER_STARTUP'
            raise ValueError('Startprüfungen endeten nach dem geplanten virtuellen '
                             'Tagesabschluss; keine Handelssimulation mehr möglich.')
        first_decision = opening + pd.Timedelta(minutes=c.opening_minutes + 1)
        session_scope = ('FULL_SIGNAL_WINDOW' if ready_at < first_decision else
                         'PARTIAL_DAY_LATE_START' if ready_at < cutoff else
                         'PARTIAL_DAY_AFTER_ENTRY_CUTOFF')
        # Decisions before readiness are historical. The current decision
        # may still be processed while its 60-second TTL remains valid.
        startup_decision = ready_at.floor('min') - pd.Timedelta(minutes=1)
        first_expected_decision = max(first_decision,
            completed_decision(ready_at + pd.Timedelta(seconds=5)))
        save_json(out / 'startup.json', dict(started_at_utc=started.isoformat(),
            market_data_ready_at_utc=ready_at.isoformat(),
            market_open_utc=opening.isoformat(), first_signal_utc=first_decision.isoformat(),
            entry_cutoff_utc=cutoff.isoformat(), planned_close_utc=portfolio.planned_close.isoformat(),
            session_close_utc=closing.isoformat(), start_window=start_window,
            session_scope=session_scope, orders_sent=0))
        last_decision = None
        missed_decision_minutes = 0
        data_error_minutes = 0
        after_cutoff_logged = False
        planned_close_checked = False
        planned_close_error = None
        end_at = closing
        print('Startstatus:', start_window, '| Auswertung:', session_scope)
        print('Virtuelle Simulation; reguläres Ende:', end_at.tz_convert(NY),
              '(NY). Keine Brokerorders möglich.')
        paint('WARTE_AUF_SIGNALFENSTER')
        historical_price_events = 0
        last_current = None
        next_wait_message = clock.now()
        while clock.now() < end_at:
            now = clock.now()
            if now < first_decision:
                if now >= next_wait_message and settings.print_every_minute:
                    stage = 'Börsenöffnung' if now < opening else 'erste Opening Range'
                    target = opening if now < opening else first_decision
                    print('Warte auf', stage, 'bis', target.tz_convert(NY),
                          '(NY); geplanter Börsenschluss:', closing.tz_convert(NY), '(NY)')
                    next_wait_message = now + pd.Timedelta(minutes=10)
                paint('WARTE_AUF_SIGNALFENSTER')   # the cockpit repaints at most once a minute
                clock.sleep(settings.poll_seconds)
                continue
            decision = completed_decision(now)
            if decision >= cutoff and last_decision is None:
                if ready_at < cutoff:
                    session_scope = 'PARTIAL_DAY_RUNTIME_INTERRUPTED'
                    portfolio.data_degraded = True
                    portfolio.log('MISSED_ENTRY_WINDOW_DURING_RUNTIME', now,
                                  ready_at_utc=ready_at.isoformat())
                # No past signal becomes a live virtual entry.
                try:
                    history = fetch_bars(data, symbols, opening, cutoff,
                        feed, 'minute', settings.batch_size)
                    save_frame(out / f'historical_only_bars_{feed}.csv', history)
                    invalid_openings = invalid_later = 0
                    history_of = split_by_symbol(history)
                    for symbol in eligible:
                        matches, opening_bad, later_bad = price_candidates_until(
                            history_of(symbol), symbol, opening, cutoff, c, settings)
                        invalid_openings += int(opening_bad)
                        invalid_later += int(later_bad)
                        for details in matches:
                            stamp = utc(details['decision_time_utc'])
                            if stamp >= cutoff:
                                continue
                            seen_decisions.add((symbol, stamp.isoformat()))
                            save_json(out / 'signals' / minute_key(stamp, symbol),
                                dict(symbol=symbol, decision_time_utc=stamp.isoformat(),
                                     status='HISTORICAL_ONLY_STARTUP_OR_RESTART',
                                     feed=feed, or_high=details['or_high'],
                                     processing_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                                     or_low=details['or_low'], orders_sent=0))
                            historical_price_events += 1
                    if invalid_openings or invalid_later:
                        portfolio.data_degraded = True
                    save_json(out / 'historical_backfill_quality.json',
                        dict(eligible_stocks=len(eligible),
                             opening_invalid_stocks=invalid_openings,
                             later_prefix_invalid_stocks=invalid_later,
                             historical_price_events=historical_price_events))
                    print('Nach Einstiegsschluss gestartet:', historical_price_events,
                          'frühere Kursereignisse nur historisch dokumentiert.')
                except Exception as exc:
                    data_error_minutes += 1
                    portfolio.data_degraded = True
                    portfolio.log('HISTORICAL_BACKFILL_ERROR', clock.now(),
                                  error_type=type(exc).__name__)
                    save_json(out / 'historical_backfill_error.json',
                        dict(error_type=type(exc).__name__, orders_sent=0))
            if decision >= cutoff:
                if not after_cutoff_logged:
                    after_cutoff_logged = True
                    print('15:15 NY: Keine neuen Einstiege; virtuelle Stops laufen bis 15:30 NY weiter.')
                if decision != last_decision and decision <= portfolio.planned_close:
                    held_bars = pd.DataFrame(columns=['symbol','timestamp','open','high',
                        'low','close','volume','vwap','feed'])
                    requested_at = clock.now()
                    if portfolio.positions:
                        try:
                            held_bars = fetch_bars(data, sorted(portfolio.positions), opening,
                                decision+pd.Timedelta(seconds=1), feed, 'minute', settings.batch_size)
                        except Exception as exc:
                            # A failed request is no evidence of "no SIP trade":
                            # keep the held minutes pending and retry next minute.
                            held_bars = None
                            data_error_minutes += 1
                            portfolio.data_degraded = True
                            portfolio.log('HELD_BAR_FETCH_ERROR', decision,
                                          error_type=type(exc).__name__)
                    if held_bars is not None:
                        # A bar returned by this request was visible when it started;
                        # a slow response must not make timely bars look late.
                        portfolio.update(decision, held_bars, requested_at)
                        check_gap_exits(data, portfolio, clock, feed)
                        if not held_bars.empty:
                            last_price_bar = held_bars.timestamp.max()
                    if order_hook is not None:
                        order_hook.on_minute(decision, clock.now())
                    last_decision = decision
                    save_json(out / 'shadow_state.json', portfolio.report())
                    save_json(out / 'shadow_events.json', portfolio.events)
                    paint('POSITIONEN_UEBERWACHEN', decision)
                if now >= portfolio.planned_close + pd.Timedelta(seconds=5) and not portfolio_closed:
                    bids = exit_quotes(data, portfolio.positions, clock, feed)
                    portfolio.finish(clock.now(), bids)
                    portfolio_closed = True
                    if order_hook is not None:
                        order_hook.on_planned_close(clock.now())
                    save_json(out / 'shadow_events.json', portfolio.events)
                    save_json(out / 'shadow_trades.json', portfolio.closed)
                    save_json(out / 'shadow_state.json', portfolio.report())
                    planned_close_checked = True
                    try:
                        save_json(out / 'broker_snapshot_planned_close.json', account_snapshot(broker, clock))
                    except Exception as exc:
                        planned_close_error = type(exc).__name__
                        save_json(out / 'broker_snapshot_planned_close_error.json',
                                  dict(status='CHECK_FAILED', error_type=planned_close_error))
                    print('15:30 NY: Virtuelle Positionen geschlossen oder als ungelöst markiert;',
                          len(portfolio.closed), 'virtuelle Trades; keine Brokerorders.')
                    paint('GEPLANTER_TAGESABSCHLUSS', decision)
                elif portfolio_closed:
                    paint('GEPLANTER_TAGESABSCHLUSS', last_decision)   # keep "Stand" moving until 16:00
                clock.sleep(settings.poll_seconds)
                continue
            if decision == last_decision or decision < first_decision:
                clock.sleep(settings.poll_seconds)
                continue
            if last_decision is None and decision > first_expected_decision:
                missed = int((decision-first_expected_decision) / pd.Timedelta(minutes=1))
                missed_decision_minutes += missed
                portfolio.data_degraded = True
                save_json(out / 'signals' / f'{decision:%Y%m%dT%H%M%SZ}_START_SKIPPED.json',
                    dict(status='MISSED_DECISION_MINUTES_AFTER_READY', count=missed,
                         first_expected_utc=first_expected_decision.isoformat(),
                         first_actual_utc=decision.isoformat(), orders_sent=0))
            if last_decision is not None and decision - last_decision > pd.Timedelta(minutes=1):
                missed = int((decision - last_decision) / pd.Timedelta(minutes=1)) - 1
                missed_decision_minutes += missed
                save_json(out / 'signals' / f'{decision:%Y%m%dT%H%M%SZ}_SKIPPED.json',
                          dict(status='MISSED_DECISION_MINUTES', count=missed,
                               after_utc=last_decision.isoformat(),
                               before_utc=decision.isoformat(), orders_sent=0))
            last_decision = decision
            if decision - now > pd.Timedelta(seconds=2):
                raise ValueError('Lokale Uhrzeit weicht von berechneter Entscheidung ab.')
            requested_at = clock.now()
            try:
                current = fetch_bars(data, symbols, opening,
                    decision + pd.Timedelta(seconds=1), feed, 'minute', settings.batch_size)
            except Exception as exc:
                data_error_minutes += 1
                portfolio.data_degraded = True
                portfolio.log('MARKET_DATA_FETCH_ERROR', decision,
                              error_type=type(exc).__name__)
                save_json(out / 'signals' / f'{decision:%Y%m%dT%H%M%SZ}_DATA_ERROR.json',
                    dict(decision_time_utc=decision.isoformat(), status='DATA_ERROR_NO_SIGNAL',
                         error_type=type(exc).__name__, orders_sent=0))
                paint('DATENFEHLER', decision)
                continue
            if settings.save_minute_snapshots:
                save_frame(out / f'session_bars_{feed}.csv', current)
            last_current = current
            if not current.empty:
                last_price_bar = current.timestamp.max()
            portfolio.update(decision, current, requested_at)   # see held-bar comment above
            check_gap_exits(data, portfolio, clock, feed)
            if order_hook is not None:
                order_hook.on_minute(decision, clock.now())
            events = []
            invalid_or = 0
            invalid_later = 0
            current_of = split_by_symbol(current)
            for symbol in eligible:
                # Detect already elapsed opportunities, but never turn them into new signals.
                matches, opening_invalid, later_invalid = price_candidates_until(
                    current_of(symbol), symbol, opening, decision, c, settings)
                if opening_invalid:
                    invalid_or += 1
                    continue
                if later_invalid:
                    invalid_later += 1
                for details in matches:
                    stamp = utc(details['decision_time_utc'])
                    pair = (symbol, stamp.isoformat())
                    if pair in seen_decisions:
                        continue
                    seen_decisions.add(pair)
                    filename = out / 'signals' / minute_key(stamp, symbol)
                    if filename.exists():
                        continue
                    if stamp != decision or stamp <= startup_decision:
                        result = dict(symbol=symbol, decision_time_utc=stamp.isoformat(),
                            status='HISTORICAL_ONLY_STARTUP_OR_RESTART', feed=feed,
                            processing_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                            or_high=details['or_high'], or_low=details['or_low'],
                            orders_sent=0)
                        historical_price_events += 1
                    else:
                        result = inspect_signal(data, symbol, details, current, past,
                            opening, out, c, s, settings, refs, startup_decision, clock)
                    save_json(filename, result)
                    events.append(result)
            for item in events:
                if item['status'] == 'SCANNER_PASS_NO_ORDER' and quote_or_ranking_expired(
                        item, utc(item['decision_time_utc']), s.signal_ttl_seconds,
                        'BEFORE_RANKING', clock):
                    save_json(out / 'signals' / minute_key(decision, item['symbol']), item)
            live_passes = [item for item in events if item['status'] == 'SCANNER_PASS_NO_ORDER']
            if live_passes:
                frame = pd.DataFrame([dict(symbol=item['symbol'],
                    decision_time_utc=item['decision_time_utc'],
                    sector=str(universe.set_index('symbol').at[item['symbol'], 'sector']),
                    rvol=item['rvol'], relative_strength=item['relative_strength'],
                    breakout=item['breakout'], spread=item['spread'], ask=item['ask'])
                    for item in live_passes])
                ranked = scanner.rank_candidates(frame, c, s)
                for record in ranked.itertuples():
                    path = out / 'signals' / minute_key(decision, record.symbol)
                    item = json.loads(path.read_text())
                    item.update(rank=int(record.rank), score=float(record.score),
                        previous_status=item['status'],
                        ranked_at_utc=clock.now().isoformat(),
                        ranking_result=record.ranking_result,
                        rvol_component=float(record.rvol_component),
                        strength_component=float(record.strength_component),
                        breakout_component=float(record.breakout_component),
                        spread_component=float(record.spread_component),
                        status='WATCHLIST_NO_ORDER' if record.watchlist else 'OUTSIDE_TOP_FIVE_NO_ORDER')
                    if record.watchlist:
                        now_entry = clock.now()
                        if (now_entry-decision).total_seconds() >= s.signal_ttl_seconds:
                            quality = dict(quote_result='SIGNAL_EXPIRED')
                        else:
                            try:
                                latest = fresh_quote(data, record.symbol, now_entry,
                                                    s.quote_search_seconds, feed)
                                quality = scanner.quote_check(latest, now_entry,
                                    item['threshold'], item['price_cap'], s, c.minimum_price)
                            except Exception as exc:
                                quality = dict(quote_result='ERROR',
                                               error_type=type(exc).__name__)
                        item['virtual_entry_quote'] = quality
                        item['shadow_status'] = portfolio.enter(clock.now(),
                            dict(symbol=record.symbol, sector=record.sector,
                                 rank=int(record.rank), score=float(record.score),
                                 rvol=float(record.rvol), decision_time_utc=item['decision_time_utc']),
                            quality, s.signal_ttl_seconds)
                        if order_hook is not None:
                            item['paper_order'] = order_hook.on_signal(
                                decision, clock.now(), dict(
                                    symbol=record.symbol, sector=record.sector,
                                    rank=int(record.rank), score=float(record.score),
                                    rvol=float(record.rvol), threshold=item['threshold'],
                                    price_cap=item['price_cap'],
                                    decision_time_utc=item['decision_time_utc']), quality)
                    save_json(path, item)
                    for event in events:   # the cockpit shows these in-memory items
                        if event['symbol'] == record.symbol and \
                                event['decision_time_utc'] == item['decision_time_utc']:
                            event.update(status=item['status'],
                                         shadow_status=item.get('shadow_status'))
            save_json(out / 'shadow_events.json', portfolio.events)
            save_json(out / 'shadow_trades.json', portfolio.closed)
            save_json(out / 'shadow_state.json', portfolio.report())
            save_json(out / 'signals' / f'{decision:%Y%m%dT%H%M%SZ}_SUMMARY.json',
                dict(decision_time_utc=decision.isoformat(), feed=feed,
                     eligible_stocks=len(eligible), opening_invalid_stocks=invalid_or,
                     later_prefix_invalid_stocks=invalid_later,
                     new_price_events=len(events), live_scanner_passes=len(live_passes),
                     missed_decision_minutes=missed_decision_minutes,
                     data_error_minutes=data_error_minutes,
                     virtual_open_positions=len(portfolio.positions),
                     virtual_closed_trades=len(portfolio.closed), orders_sent=0))
            if settings.print_every_minute:
                print(decision.tz_convert(NY).strftime('%H:%M'), 'NY:',
                      len(events), 'Preisereignisse,', len(live_passes), 'Scanner-Treffer;',
                      invalid_or, 'Opening Ranges unvollständig;',
                      invalid_later, 'spätere Kursreihen unvollständig')
            paint('SIGNALPRUEFUNG', decision, events)
            clock.sleep(settings.poll_seconds)
        status = ('SHADOW_SIM_INCOMPLETE_OR_DATA_DEGRADED_NO_ORDERS'
                  if missed_decision_minutes or data_error_minutes or planned_close_error or
                  not planned_close_checked or portfolio.positions or portfolio.data_degraded
                  else 'SHADOW_SIM_COMPLETED_NO_ORDERS' if session_scope == 'FULL_SIGNAL_WINDOW'
                  else 'SHADOW_SIM_PARTIAL_DAY_COMPLETED_NO_ORDERS')
        save_json(out / 'coverage.json', dict(missed_decision_minutes=missed_decision_minutes,
                    data_error_minutes=data_error_minutes,
                    cockpit_error=cockpit_error,
                    session_scope=session_scope, historical_price_events_total=historical_price_events,
                    started_at_utc=started.isoformat(),
                    market_data_ready_at_utc=ready_at.isoformat(),
                    planned_close_checked=planned_close_checked,
                    planned_close_error=planned_close_error,
                    session_close_utc=closing.isoformat(),
                    note='Ab 15:15 NY nur bestehende virtuelle Positionen überwacht; keine Brokerorders.'))
        if last_current is not None and not settings.save_minute_snapshots:
            save_frame(out / f'session_bars_{feed}.csv', last_current)
    except Exception as exc:
        # Persist why the run stopped; the exception still propagates.
        save_json(out / 'failed.json', dict(status='failed', error_type=type(exc).__name__,
            detail=str(exc)[:2000], http_status=getattr(exc, 'status_code', None),
            failed_at_utc=clock.now().isoformat(), orders_sent=0))
        raise
    finally:
        if order_hook is not None:
            try:   # stopped early or finished: never leave unmanaged paper positions
                order_hook.on_stop(clock.now())
            except Exception as exc:
                status = 'PAPER_FLATTEN_FAILED_MANUAL_REVIEW_REQUIRED'
                print('ACHTUNG: Paper-Glattstellung fehlgeschlagen:', type(exc).__name__)
        if out.exists():
            save_json(out / 'shadow_events.json', portfolio.events)
            save_json(out / 'shadow_trades.json', portfolio.closed)
            save_json(out / 'shadow_state.json', portfolio.report())
            try:
                ending = account_snapshot(broker, clock)
                save_json(out / 'broker_snapshot_end.json', ending)
                if ending['position_symbols'] or ending['open_order_ids']:
                    status = 'BROKER_HAS_OPEN_ITEMS_MANUAL_REVIEW_REQUIRED'
                    print('ACHTUNG: Broker meldet offene Positionen/Orders. Im Paper-Dashboard manuell prüfen.')
            except Exception as exc:
                status = 'BROKER_END_SNAPSHOT_FAILED_MANUAL_REVIEW_REQUIRED'
                print('Abschlussabgleich fehlgeschlagen:', type(exc).__name__)
            if order_hook is not None and order_hook.orders_sent and status.endswith('_NO_ORDERS'):
                status = status[:-len('_NO_ORDERS')] + '_WITH_PAPER_ORDERS'
            save_json(out / 'status.json', dict(status=status, finished_at_utc=
                clock.now().isoformat(),
                orders_sent=order_hook.orders_sent if order_hook is not None else 0,
                session_scope=session_scope,
                warning='Virtuelle Ergebnisse sind Modellwerte, keine Brokerfills.'))
            paint('FAILED' if status in ('INCOMPLETE',
                    'BROKER_HAS_OPEN_ITEMS_MANUAL_REVIEW_REQUIRED',
                    'BROKER_END_SNAPSHOT_FAILED_MANUAL_REVIEW_REQUIRED',
                    f'{feed.upper()}_ACCESS_CHECK_FAILED_NO_ORDERS') else 'FINISHED', last_decision if
                    'last_decision' in locals() else None)
    print('Fertig:', out, '– virtuelle Trades:', len(portfolio.closed),
          '; keine Brokerorders gesendet.')
    return out


In [ ]:
%%writefile /content/orb_cockpit_v15.py
"""Read-only Colab cockpit for virtual ORB positions, backed by run-folder snapshots."""
from collections import deque
from html import escape
from pathlib import Path
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from types import SimpleNamespace
import json
import math
import threading

import pandas as pd


NY = 'America/New_York'


def utc(value):
    return pd.Timestamp(value).tz_convert('UTC')


def safe(value):
    return escape(str(value if value is not None else '—'), quote=True)


def dollars(value):
    return f'{float(value):,.2f} USD' if value is not None and math.isfinite(float(value)) else '—'


def ny_time(value):
    return utc(value).tz_convert(NY).strftime('%H:%M:%S') if value is not None else '—'


def table(headings, rows, empty):
    header = ''.join(f'<th>{safe(h)}</th>' for h in headings)
    body = ''.join('<tr>' + ''.join(f'<td>{cell}</td>' for cell in row) + '</tr>'
                   for row in rows)
    if not body:
        body = f'<tr><td colspan="{len(headings)}" class="muted">{safe(empty)}</td></tr>'
    return f'<div class="scroll"><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>'


def snapshot(portfolio, at, decision, stage, mode, eligible_count,
             last_price_bar=None, signals=(), broker_start=None, clock_lag_seconds=0):
    """Use state already computed by the scanner; never request market data."""
    report = portfolio.report()
    positions = []
    for symbol, pos in sorted(portfolio.positions.items()):
        mark_time, mark = portfolio.last_marks.get(symbol, (None, None))
        entry, stop, qty = float(pos['entry']), float(pos['stop']), int(pos['qty'])
        entry_mark = (mark_time is not None and
                      mark_time <= utc(pos['entry_at_utc']).floor('min'))
        positions.append(dict(symbol=symbol, qty=qty,
            entry_at_utc=pos['entry_at_utc'], entry=entry, last_close_proxy=mark,
            last_close_minute_utc=mark_time.isoformat() if mark_time is not None else None,
            mark_is_entry_proxy=entry_mark,
            stop=stop, stop_distance_usd=stop-entry,
            stop_distance_pct=100*(stop/entry-1),
            pnl_open_proxy_usd=qty*(mark-entry) if mark is not None else None,
            planned_risk_usd=qty*max(0, entry-stop),
            stop_coverage_unreliable=bool(pos.get('stop_coverage_unreliable')),
            unobserved_minutes=pos.get('unobserved_minutes', 0),
            pending_minutes=pos.get('pending_minutes', 0)))
    closed = [dict(symbol=p['symbol'], qty=p['qty'],
                   entry_at_utc=p['entry_at_utc'], exit_at_utc=p['exit_at_utc'],
                   entry=p['entry'], exit_price_proxy=p['exit_price_proxy'],
                   pnl_usd=p['pnl_usd'], reason=p['exit_reason'],
                   outcome_unreliable=bool(p.get('outcome_unreliable')))
              for p in portfolio.closed[-12:]]
    return dict(version='1.4.3', mode=mode, stage=stage, clock_lag_seconds=clock_lag_seconds,
        updated_at_utc=utc(at).isoformat(),
        checked_decision_utc=utc(decision).isoformat() if decision is not None else None,
        last_price_bar_utc=utc(last_price_bar).isoformat() if last_price_bar is not None else None,
        broker_start_equity_usd=(broker_start or {}).get('equity'),
        eligible_symbols=eligible_count,
        closed_count=len(portfolio.closed),
        wins=sum(1 for p in portfolio.closed if p['pnl_usd'] > 0),
        losses=sum(1 for p in portfolio.closed if p['pnl_usd'] < 0),
        start_equity_usd=report.get('starting_cash_usd'),
        orders_sent=0, equity_proxy_usd=report['ending_equity_proxy_usd'],
        cash_proxy_usd=report['ending_cash_usd'],
        invested_usd=sum(float(p['qty'])*float(p['entry']) for p in portfolio.positions.values()),
        planned_open_risk_usd=sum(p['planned_risk_usd'] for p in positions),
        realized_pnl_proxy_usd=report['realized_pnl_usd'],
        realized_pnl_10bp_stress_usd=report['realized_pnl_10bp_stress_usd'],
        open_pnl_proxy_usd=sum(p['pnl_open_proxy_usd'] or 0 for p in positions),
        data_degraded=report['data_degraded'],
        entry_data_blocked=report['entry_data_blocked'],
        unreliable_closed_trades=report['unreliable_virtual_trades'],
        open_positions=positions, closed_trades=closed,
        signals=[dict(symbol=x.get('symbol'), status=x.get('status'),
                      shadow_status=x.get('shadow_status'),
                      time=x.get('decision_time_utc')) for x in signals],
        # Routine per-minute bar reviews would push entries, exits and gaps out of view.
        recent_events=[dict(kind=e['kind'], time=e['at_utc'],
                            symbol=e.get('symbol'), reason=e.get('reason'))
                       for e in [e for e in portfolio.events if e['kind'] != 'BAR_REVIEWED'][-8:]])


def render_html(data):
    """A single self-contained HTML surface; all external data is escaped."""
    lag_minutes = round(data.get('clock_lag_seconds', 0) / 60)
    mode = ('Historischer Replay' if data['mode'] == 'replay' else
            f'Verzögerte Live-Beobachtung · SIP {lag_minutes} Min. versetzt' if data['mode'] == 'delayed'
            else f'BOT 2 – SIP-Delayed-Bot · SIMULIERT · SIP {lag_minutes} Min. versetzt'
            if data['mode'] == 'sip_shadow'
            else 'BOT 1 – IEX-Echtzeit-Bot · IEX-Simulation' if data['mode'] == 'iex_live'
            else 'Live-Beobachtung')
    status = ('Lauf mit Fehler beendet' if data['stage'] == 'FAILED' else
              'Datenqualität eingeschränkt' if data['data_degraded'] else
              'Beobachtung läuft' if data['stage'] not in ('FINISHED', 'FAILED') else
              'Lauf beendet')
    status_class = 'bad' if data['data_degraded'] or data['stage'] == 'FAILED' else 'ok'
    entry_status = ('Neue Einstiege: ausstehende Stop-Beobachtung' if
                    data['entry_data_blocked'] else 'Neue Einstiege: keine Datenlückensperre')
    pos_rows = []
    for p in data['open_positions']:
        distance = p['stop_distance_usd']
        tone = 'bad' if distance < 0 else 'ok' if distance > 0 else 'neutral'
        stop = (f'<span class="{tone}">{dollars(p["stop"])} '
                f'({distance:+.2f} / {p["stop_distance_pct"]:+.2f} %)</span>')
        quality = 'Stop-Verlauf unsicher' if p['stop_coverage_unreliable'] else 'beobachtet'
        if p['unobserved_minutes']:
            quality += f' · {p["unobserved_minutes"]} Min. ohne Kerze'
        if p['pending_minutes']:
            quality += f' · {p["pending_minutes"]} Min. ausstehend'
        last_label = ('Einstiegs-Ask · Folgekerze ausstehend' if p['mark_is_entry_proxy'] else
                      f'Kerze {ny_time(p["last_close_minute_utc"])} NY')
        last = (f'{dollars(p["last_close_proxy"])}<br><small>{safe(last_label)}</small>')
        pos_rows.append((safe(p['symbol']), safe(p['qty']),
            f'{dollars(p["entry"])}<br><small>{ny_time(p["entry_at_utc"])} NY</small>',
            last, stop, dollars(p['pnl_open_proxy_usd']),
            f'{dollars(p["planned_risk_usd"])}<br><small>{safe(quality)}</small>'))
    trade_rows = [(safe(t['symbol']),
                   f'{ny_time(t["entry_at_utc"])} → {ny_time(t["exit_at_utc"])} NY',
                   safe(t['qty']), dollars(t['entry']), dollars(t['exit_price_proxy']),
                   dollars(t['pnl_usd']), safe(t['reason'])+
                   (' · <span class="bad">unsicher</span>' if t['outcome_unreliable'] else ''))
                  for t in reversed(data['closed_trades'])]
    signal_rows = [(safe(s['symbol']), ny_time(s['time']), safe(s['status']),
                    safe(s['shadow_status'])) for s in reversed(data['signals'])]
    event_rows = [(ny_time(e['time']), safe(e['symbol']), safe(e['kind']),
                   safe(e['reason'])) for e in reversed(data['recent_events'])]
    bar_note = (' (nach 15:15 NY Kursabruf nur für offene Positionen)'
                if data['checked_decision_utc'] and data['last_price_bar_utc'] and
                utc(data['checked_decision_utc']) - utc(data['last_price_bar_utc']) > pd.Timedelta(minutes=2)
                and not data['open_positions'] else '')
    account_note = ('Replay: kein Alpaca-Konto abgefragt; virtuelles Startkapital 10,000.00 USD.'
                    if data['mode'] == 'replay' else
                    'SIMULIERT: eigenes virtuelles Konto von BOT 2; keinerlei Alpaca-Orders möglich.'
                    if data['mode'] == 'sip_shadow' else
                    f'Start-Snapshot des Alpaca-Paper-Kontos: {dollars(data["broker_start_equity_usd"])}; '
                    'niemals mit dem virtuellen Portfolio-Ergebnis verrechnen.')
    def tile(label, value):
        return f'<div class="tile"><span>{safe(label)}</span><strong>{value}</strong></div>'
    cards = ''.join((tile('Virtuelles Equity · Modell', dollars(data['equity_proxy_usd'])),
                     tile('Virtuelles Cash', dollars(data['cash_proxy_usd'])),
                     tile('In offenen Positionen', dollars(data['invested_usd'])),
                     tile('Geplantes offenes Risiko', dollars(data['planned_open_risk_usd'])),
                     tile('Offenes Ergebnis · letzter Kerzenschluss', dollars(data['open_pnl_proxy_usd'])),
                     tile('Realisierter Modell-PnL', dollars(data['realized_pnl_proxy_usd'])),
                     tile('Mit 10 bp Kostenstress', dollars(data['realized_pnl_10bp_stress_usd'])),
                     tile('Offen / geschlossen', f'{len(data["open_positions"])} / {data["closed_count"]}')))
    return f'''<!doctype html><html lang="de"><head><meta charset="utf-8"><style>
.orb-cockpit{{font:14px/1.42 system-ui,-apple-system,sans-serif;color:#182338;background:#f5f8fc;
padding:18px;border-radius:14px;max-width:1260px;margin:0 auto}}
.orb-cockpit h2{{margin:0 0 6px;font-size:22px}} .orb-cockpit h3{{font-size:16px;margin:18px 0 7px}}
.orb-cockpit .subtitle,.orb-cockpit small,.orb-cockpit .muted{{color:#58677a}}
.orb-cockpit .ok{{color:#177245;font-weight:700}} .orb-cockpit .bad{{color:#ba3042;font-weight:700}}
.orb-cockpit .neutral{{color:#344e75;font-weight:700}}
.orb-cockpit .tiles{{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:9px;margin:14px 0}}
.orb-cockpit .tile{{background:white;border:1px solid #d9e1ed;border-radius:10px;padding:10px 12px}}
.orb-cockpit .tile span{{display:block;color:#58677a;font-size:12px}} .orb-cockpit .tile strong{{font-size:17px}}
.orb-cockpit .scroll{{overflow-x:auto}} .orb-cockpit table{{border-collapse:collapse;width:100%;background:white}}
.orb-cockpit th,.orb-cockpit td{{border-bottom:1px solid #e1e7f0;text-align:left;padding:8px 9px;vertical-align:top;white-space:nowrap}}
.orb-cockpit th{{background:#eaf0fa;font-size:12px}} .orb-cockpit .note{{margin-top:13px;font-size:12px;color:#47566d}}
</style></head><body><div class="orb-cockpit">
<h2>US-Aktien-Bot · virtuelles Cockpit</h2>
<div class="subtitle" id="orb-stand" data-updated-ms="{int(utc(data['updated_at_utc']).timestamp() * 1000)}" data-quiet="{'1' if data['stage'] in QUIET_STAGES else '0'}" data-replay="{'1' if data['mode'] == 'replay' else '0'}" data-lag-ms="{int(data.get('clock_lag_seconds', 0) * 1000)}">{safe(mode)} · {safe(data['stage'])} · Stand {ny_time(data['updated_at_utc'])} NY ·
letzte Strategieentscheidung {ny_time(data['checked_decision_utc'])} NY ·
letzte verfügbare Kursminute {ny_time(data['last_price_bar_utc'])} NY{bar_note}</div>
<p><span class="{status_class}">{safe(status)}</span> · SIP · {data['eligible_symbols']} Aktien im Filter ·
Alpaca-Orders: 0 · offene Trades sind ausschließlich virtuell.<br>
<span class="{'bad' if data['entry_data_blocked'] else 'neutral'}">{safe(entry_status)}</span></p>
<div class="tiles">{cards}</div>
<h3>Offene virtuelle Positionen</h3>
{table(('Aktie','Stück','Einstieg','Letzter Kerzenschluss','Trailing-Stop','Offener PnL','Risiko / Qualität'), pos_rows, 'Keine offene virtuelle Position.')}
<h3>Abgeschlossene virtuelle Deals · letzte 12</h3>
{table(('Aktie','Zeit','Stück','Kauf-Proxy','Verkauf-Proxy','PnL','Ausstieg / Qualität'), trade_rows, 'Noch kein virtueller Abschluss.')}
<h3>Letzte Signale</h3>
{table(('Aktie','Signal NY','Scannerstatus','Virtueller Einstieg'), signal_rows, 'Noch kein auswertbares Signal.')}
<h3>Letzte Portfolioereignisse</h3>
{table(('Zeit NY','Aktie','Ereignis','Grund'), event_rows, 'Noch keine Portfolioereignisse.')}
<div class="note">Kurse und PnL sind Modellwerte aus abgeschlossenen SIP-Minuten und SIP-Quotes, keine Alpaca-Fills.
Das offene Ergebnis ist eine Bewertung zum letzten beobachteten Kerzenschluss; bei Datenlücken kann sie veraltet sein.
Virtuelle Trades erscheinen nicht im Alpaca-Paper-Konto. Datum/Handelszeit America/New_York.
Scanner prüft abgeschlossene Minuten einmal pro Minute; neue Einstiege bis 15:15 NY, virtuelle Glattstellung 15:30 NY.
<br>{account_note} Nach simulierten Kosten positive Stops sind gesondert zu prüfen.</div>
</div></body></html>'''


BROWSER_REFRESH_SECONDS = 15
# No minute is computed in these stages (waiting for the signal window, after the
# planned close, finished): an old timestamp there is expected, not a stall.
QUIET_STAGES = ('WARTE_AUF_SIGNALFENSTER', 'GEPLANTER_TAGESABSCHLUSS', 'FINISHED', 'FAILED')


def browser_page(page, refresh_seconds=BROWSER_REFRESH_SECONDS):
    """Same view for a normal browser tab: reloads itself and shows its age."""
    head = (f'<meta charset="utf-8"><meta http-equiv="refresh" content="{int(refresh_seconds)}">'
            '<meta name="viewport" content="width=device-width,initial-scale=1">')
    age = ('<p class="muted" id="orb-age"></p><script>(function(){'
           'var s=document.getElementById("orb-stand");if(!s)return;'
           'var e=document.getElementById("orb-age");'
           'if(s.dataset.replay==="1"){e.textContent="Browseransicht lädt alle '
           f'{int(refresh_seconds)}'
           ' s neu · Replay: Uhrzeiten sind simuliert";return;}'
           'var m=Math.floor((Date.now()-Number(s.dataset.lagMs||0)-Number(s.dataset.updatedMs))/60000);if(!(m>=0))return;'
           f'e.textContent="Browseransicht lädt alle {int(refresh_seconds)} s neu · Stand ist "+m+" Min. alt";'
           'if(m>=3&&s.dataset.quiet!=="1"){e.className="bad";'
           'e.textContent+=" – Lauf läuft evtl. nicht mehr (Colab prüfen)";}})();</script>')
    return page.replace('<meta charset="utf-8">', head, 1).replace(
        '<div class="tiles">', age + '<div class="tiles">', 1)


WAITING_PAGE = browser_page('<!doctype html><html lang="de"><head><meta charset="utf-8"></head>'
    '<body style="font:15px system-ui,sans-serif;padding:24px">'
    '<h2>US-Aktien-Bot · virtuelles Cockpit</h2><p>Warte auf den ersten Cockpit-Stand …</p>'
    '<div class="tiles"></div></body></html>')


def write_atomic(path, payload):
    temp = path.with_name(path.name + '.incomplete')
    temp.write_text(payload, encoding='utf-8')
    temp.replace(path)


class Cockpit:
    def __init__(self, display_enabled=True, latest_path=None, latest_json_path=None):
        """`latest_path`: one fixed, self-refreshing HTML file (e.g. on Drive)
        that always shows the newest view, across replay days."""
        self.display_enabled = display_enabled
        self.latest_path = Path(latest_path) if latest_path is not None else None
        self.latest_json_path = Path(latest_json_path) if latest_json_path is not None else None
        self.latest_browser_page = WAITING_PAGE
        self.latest_snapshot_json = '{}'
        self.write_errors = 0
        self.handle = None
        self.last_signals = deque(maxlen=8)
        self.signal_total = 0
        self.closed_count = 0
        self.last_render_key = None
        self.last_run_folder = None

    def update(self, folder, portfolio, at, decision=None, stage='RUNNING',
               mode='live', eligible_count=0, last_price_bar=None,
               signals=(), broker_start=None, clock_lag_seconds=0):
        if str(folder) != self.last_run_folder:
            self.last_run_folder = str(folder)
            self.last_signals.clear()
            self.signal_total = 0
            self.closed_count = 0
            self.last_render_key = None
        for item in signals:
            self.last_signals.append(item)
        self.signal_total += len(signals)
        now = utc(at)
        changed = self.closed_count != len(portfolio.closed)
        # Replay paints every 5 simulated minutes, and on position/signal changes.
        if (mode == 'replay' and stage in ('SIGNALPRUEFUNG', 'POSITIONEN_UEBERWACHEN',
                                           'WARTE_AUF_SIGNALFENSTER', 'GEPLANTER_TAGESABSCHLUSS')
                and not changed and not signals
                and now.minute % 5 != 0):
            return None
        key = (str(folder), now.floor('min'), stage, len(portfolio.closed))
        if key == self.last_render_key and not changed and not signals:
            return None
        self.last_render_key = key
        self.closed_count = len(portfolio.closed)
        data = snapshot(portfolio, now, decision, stage, mode, eligible_count,
                        last_price_bar, self.last_signals, broker_start, clock_lag_seconds)
        data['signal_total'] = self.signal_total
        page = render_html(data)
        snapshot_json = json.dumps(data, ensure_ascii=False, indent=2, default=str)
        self.latest_browser_page = browser_page(page)   # served by the live link
        self.latest_snapshot_json = snapshot_json
        folder = Path(folder)
        targets = [(folder / 'cockpit_snapshot.json', snapshot_json), (folder / 'cockpit.html', page)]
        if self.latest_path is not None:
            targets.append((self.latest_path, self.latest_browser_page))
        if self.latest_json_path is not None:
            targets.append((self.latest_json_path, snapshot_json))
        for path, payload in targets:
            try:
                path.parent.mkdir(parents=True, exist_ok=True)
                write_atomic(path, payload)
            except OSError as exc:
                # A Drive hiccup must not switch the cockpit off for the rest of the day;
                # the next update writes the file again.
                self.write_errors += 1
                print('Cockpit-Datei nicht geschrieben:', path.name, type(exc).__name__)
        if self.display_enabled:
            from IPython.display import HTML, display
            if self.handle is None:
                self.handle = display(HTML(page), display_id=True)
            else:
                self.handle.update(HTML(page))
        return data


def refresh_saved_cockpit(folder, cockpit, ui_hash):
    """Refresh a cached day's final view without querying or recomputing trades."""
    if cockpit is None:
        return
    folder = Path(folder)
    marker = folder / 'cockpit_render.json'
    if marker.exists() and (folder / 'cockpit.html').exists() and \
            json.loads(marker.read_text()).get('cockpit_sha256') == ui_hash:
        return
    state = json.loads((folder / 'shadow_state.json').read_text())
    if state.get('open_symbols'):
        # No intraday position state is available after a runtime restart.
        return
    closed = json.loads((folder / 'shadow_trades.json').read_text())
    events = json.loads((folder / 'shadow_events.json').read_text())
    previous = json.loads((folder / 'cockpit_snapshot.json').read_text()) if \
        (folder / 'cockpit_snapshot.json').exists() else {}
    status = json.loads((folder / 'status.json').read_text())
    view = SimpleNamespace(positions={}, closed=closed, events=events,
                           last_marks={}, report=lambda: state)
    signals = [dict(symbol=s.get('symbol'), status=s.get('status'),
                    shadow_status=s.get('shadow_status'),
                    decision_time_utc=s.get('time'))
               for s in previous.get('signals', [])]
    cockpit.update(folder, view, utc(status['finished_at_utc']),
        decision=previous.get('checked_decision_utc'),
        stage='FINISHED', mode='replay',
        eligible_count=previous.get('eligible_symbols', 0),
        last_price_bar=previous.get('last_price_bar_utc'),
        signals=signals, broker_start={'equity': 10000})
    temporary = marker.with_name(marker.name + '.incomplete')
    temporary.write_text(json.dumps(dict(cockpit_sha256=ui_hash,
        rendered_from_saved_state=True, strategy_replayed=False), indent=2))
    temporary.replace(marker)


class _Handler(BaseHTTPRequestHandler):
    cockpit = None

    def do_GET(self):
        path = self.path.split('?', 1)[0]
        if path in ('/', '/index.html'):
            body, kind = self.cockpit.latest_browser_page, 'text/html; charset=utf-8'
        elif path == '/snapshot.json':
            body, kind = self.cockpit.latest_snapshot_json, 'application/json; charset=utf-8'
        else:
            self.send_error(404)
            return
        payload = body.encode('utf-8')
        self.send_response(200)
        self.send_header('Content-Type', kind)
        self.send_header('Cache-Control', 'no-store')
        self.send_header('Content-Length', str(len(payload)))
        self.end_headers()
        self.wfile.write(payload)

    def log_message(self, *args):   # keep the notebook output clean
        pass


def serve_cockpit(cockpit, port=8765, attempts=20):
    """Serve the newest cockpit view over HTTP from a daemon thread.

    Read-only: only the page/snapshot already rendered by Cockpit.update are
    returned; no market data, no orders, no file system access."""
    handler = type('CockpitHandler', (_Handler,), dict(cockpit=cockpit))
    for candidate in range(port, port + attempts):
        try:
            server = ThreadingHTTPServer(('', candidate), handler)
        except OSError:
            continue   # e.g. a previous run of the cell still holds the port
        server.daemon_threads = True
        threading.Thread(target=server.serve_forever, daemon=True,
                         name='orb-cockpit-server').start()
        return server
    raise OSError('Kein freier Port für die Cockpit-Browseransicht.')


def open_browser_view(cockpit, port=8765):
    """Start the server and, inside Colab, show a link that opens it in a browser tab."""
    server = serve_cockpit(cockpit, port)
    port = server.server_address[1]
    try:
        from google.colab import output
    except ImportError:
        print(f'Cockpit im Browser: http://localhost:{port}/')
        return server
    output.serve_kernel_port_as_window(port, path='/', anchor_text='Cockpit im Browser öffnen')
    print('Der Link funktioniert, solange diese Colab-Sitzung läuft; '
          f'die Seite lädt sich alle {BROWSER_REFRESH_SECONDS} s selbst neu.')
    return server


def page_parts(page):
    """(style, body) of a rendered cockpit page, to compose several sections."""
    style = page.split('<style>', 1)[1].split('</style>', 1)[0]
    body = page.split('<body>', 1)[1].rsplit('</body>', 1)[0]
    return style, body


In [ ]:
%%writefile /content/orb_replay_v15.py
"""Replay historical SIP sessions through the unchanged live code path. No orders.

The live loop (sip_shadow_sim_v14.run_dryrun) runs against a simulated clock.
ReplayData serves no future timestamps: a final historical minute bar becomes
visible from `bar_delay_seconds` after its minute, quotes/trades up to now.
The actual arrival of later trades and bar revisions is not in the archive.
Every data request advances the clock by `request_latency_seconds`.
Historical SIP data is fetched once from Alpaca and cached on disk (Drive).
"""
from dataclasses import asdict, dataclass, replace
from datetime import date, datetime, timedelta
from pathlib import Path
from types import SimpleNamespace
import hashlib
import json
import math
import sys
import time
import uuid

import pandas as pd
import us_orb_test_v02 as base
import us_orb_scanner_v03 as scanner
import orb_portfolio_v15 as shadow
import orb_sim_v15 as sim
import orb_cockpit_v15 as cockpit_module

VERSION = '1.4.3-sip-replay-no-orders'
NY = 'America/New_York'
BAR_COLUMNS = ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'vwap']


@dataclass(frozen=True)
class ReplaySettings:
    start_before_open_minutes: int = 10
    bar_delay_seconds: float = 2.0         # assumed availability, not actual historical arrival time
    request_latency_seconds: float = 0.3   # simulated time consumed by each data request
    source_spacing_seconds: float = 0.35   # real pause between Alpaca requests (rate limit)
    source_batch_size: int = 50


def as_utc(value):
    """alpaca-py stores request datetimes as naive UTC; never read them as NY."""
    stamp = pd.Timestamp(value)
    return stamp.tz_localize('UTC') if stamp.tzinfo is None else stamp.tz_convert('UTC')


def ny_day(value):
    return as_utc(value).tz_convert(NY).strftime('%Y-%m-%d')


def request_key(*parts):
    return hashlib.sha256(json.dumps(parts, default=str).encode()).hexdigest()[:24]


def write_gz(path, frame):
    temporary = path.with_name(path.name + '.incomplete')
    frame.to_csv(temporary, index=False, compression='gzip')
    temporary.replace(path)


def read_gz(path):
    frame = pd.read_csv(path, compression='gzip')
    frame['timestamp'] = pd.to_datetime(frame.timestamp, utc=True)
    return frame


def bars_frame(answer):
    """Alpaca BarSet (or a replay answer) -> flat frame with BAR_COLUMNS."""
    if not any(answer.data.values()):
        return pd.DataFrame(columns=BAR_COLUMNS)
    frame = base.normalize_bars(answer.df)
    if 'vwap' not in frame:
        frame['vwap'] = float('nan')
    return frame[BAR_COLUMNS]


class ReplayClock:
    def __init__(self, start):
        self.current = as_utc(start)

    def now(self):
        return self.current

    def sleep(self, seconds):
        self.advance(seconds)

    def advance(self, seconds):
        self.current += pd.Timedelta(seconds=seconds)


class ReplayStore:
    """Disk cache of historical SIP data; `source_data`/`source_broker` fill gaps.

    Without a source every missing item raises: a replay never silently runs
    on incomplete data."""

    def __init__(self, folder, source_data=None, source_broker=None,
                 spacing_seconds=0.35, batch_size=50):
        self.folder = Path(folder)
        self.folder.mkdir(parents=True, exist_ok=True)
        self.source_data, self.source_broker = source_data, source_broker
        self.spacing, self.batch_size = spacing_seconds, batch_size
        self.last_call = 0.0
        self.minutes = {}       # NY date -> frame of all cached symbols
        self.calendars = {}     # year -> list of calendar dicts
        self.source_calls = 0

    def _call(self, client, method, *args, **kwargs):
        if client is None:
            raise ValueError(f'Replay-Daten fehlen im Cache ({method}) und keine Datenquelle angegeben.')
        wait = self.spacing - (time.monotonic() - self.last_call)
        if wait > 0:
            time.sleep(wait)
        self.last_call = time.monotonic()
        self.source_calls += 1
        return getattr(client, method)(*args, **kwargs)

    # ---- calendar ----
    def _calendar_year(self, year):
        if year not in self.calendars:
            path = self.folder / 'calendar' / f'{year}.json'
            if not path.exists():
                from alpaca.trading.requests import GetCalendarRequest
                rows = self._call(self.source_broker, 'get_calendar', GetCalendarRequest(
                    start=date(year, 1, 1), end=date(year, 12, 31)))
                path.parent.mkdir(parents=True, exist_ok=True)
                sim.save_json(path, [dict(date=str(r.date), open=str(r.open), close=str(r.close))
                                     for r in rows])
            self.calendars[year] = json.loads(path.read_text())
        return self.calendars[year]

    def calendar(self, start, end):
        """Alpaca-shaped calendar entries (naive NY open/close) in [start, end]."""
        start, end = str(start)[:10], str(end)[:10]
        rows = []
        for year in range(int(start[:4]), int(end[:4]) + 1):
            rows.extend(r for r in self._calendar_year(year) if start <= r['date'] <= end)
        return [SimpleNamespace(date=date.fromisoformat(r['date']),
                                open=datetime.fromisoformat(r['open']),
                                close=datetime.fromisoformat(r['close'])) for r in rows]

    def sessions(self, start, end):
        return base.calendar_rows(self.calendar(start, end))

    # ---- universe ----
    def universe(self):
        """One dated Nasdaq-100 snapshot for all replay days (survivorship bias!)."""
        path = self.folder / 'universe_snapshot.csv'
        if not path.exists():
            frame = sim.fresh_universe(self.source_broker)
            self.set_universe(frame, dict(source='nasdaq_api_current_members'))
        return pd.read_csv(path, keep_default_na=False).assign(
            tradable_now=lambda f: f.tradable_now.astype(str) == 'True')

    def set_universe(self, frame, meta):
        sim.save_frame(self.folder / 'universe_snapshot.csv', frame)
        sim.save_json(self.folder / 'universe_provenance.json', dict(
            meta, retrieved_at_utc=pd.Timestamp.now(tz='UTC').isoformat(),
            historical_membership_verified=False,
            note='Aktuelle Indexmitglieder für alle Replay-Tage: Survivorship-Bias.'))

    def universe_meta(self):
        path = self.folder / 'universe_provenance.json'
        return json.loads(path.read_text()) if path.exists() else None

    # ---- minute bars ----
    def day_minutes(self, symbols, day):
        """Regular-hours SIP minute bars of one NY date, fetched once per symbol."""
        folder = self.folder / 'minute'
        path, index_path = folder / f'{day}.csv.gz', folder / f'{day}.symbols.json'
        if day not in self.minutes:
            if path.exists() and index_path.exists():
                self.minutes[day] = (read_gz(path), set(json.loads(index_path.read_text())))
            else:
                self.minutes[day] = (pd.DataFrame(columns=BAR_COLUMNS), set())
        frame, known = self.minutes[day]
        missing = sorted(set(symbols) - known)
        if missing:
            from alpaca.data.requests import StockBarsRequest
            from alpaca.data.enums import DataFeed, Adjustment
            from alpaca.data.timeframe import TimeFrame
            start = pd.Timestamp(f'{day} 09:30', tz=NY).tz_convert('UTC')
            end = pd.Timestamp(f'{day} 16:00', tz=NY).tz_convert('UTC')
            parts = [frame] if len(frame) else []
            for offset in range(0, len(missing), self.batch_size):
                answer = self._call(self.source_data, 'get_stock_bars', StockBarsRequest(
                    symbol_or_symbols=missing[offset:offset + self.batch_size],
                    timeframe=TimeFrame.Minute, start=start.to_pydatetime(),
                    end=end.to_pydatetime(), feed=DataFeed.SIP, adjustment=Adjustment.RAW))
                fetched = bars_frame(answer)
                parts.append(fetched[(fetched.timestamp >= start) & (fetched.timestamp < end)])
            parts = [part for part in parts if len(part)]
            frame = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=BAR_COLUMNS)
            known = known | set(missing)
            folder.mkdir(parents=True, exist_ok=True)
            write_gz(path, frame)
            sim.save_json(index_path, sorted(known))
            self.minutes[day] = (frame, known)
        return frame[frame.symbol.isin(symbols)]

    def keep_days(self, days):
        """Drop in-memory minute data outside `days` (disk cache stays)."""
        for day in list(self.minutes):
            if day not in days:
                del self.minutes[day]

    # ---- other requests, cached per request ----
    def _cached(self, kind, key, fetch):
        path = self.folder / kind / f'{key}.json'
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            sim.save_json(path, fetch())
        return json.loads(path.read_text())

    def daily(self, symbols, start, end):
        def fetch():
            from alpaca.data.requests import StockBarsRequest
            from alpaca.data.enums import DataFeed, Adjustment
            from alpaca.data.timeframe import TimeFrame
            frames = []
            for offset in range(0, len(symbols), self.batch_size):
                answer = self._call(self.source_data, 'get_stock_bars', StockBarsRequest(
                    symbol_or_symbols=symbols[offset:offset + self.batch_size],
                    timeframe=TimeFrame.Day, start=start.to_pydatetime(),
                    end=end.to_pydatetime(), feed=DataFeed.SIP, adjustment=Adjustment.RAW))
                frames.append(bars_frame(answer))
            frames = [f for f in frames if len(f)]
            frame = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=BAR_COLUMNS)
            return json.loads(frame.to_json(orient='records', date_format='iso'))
        rows = self._cached('daily', request_key('daily', sorted(symbols), start, end), fetch)
        frame = pd.DataFrame(rows, columns=BAR_COLUMNS)
        frame['timestamp'] = pd.to_datetime(frame.timestamp, utc=True)
        return frame

    def quotes(self, symbol, start, end, limit, sort):
        def fetch():
            from alpaca.data.requests import StockQuotesRequest
            from alpaca.data.enums import DataFeed
            answer = self._call(self.source_data, 'get_stock_quotes', StockQuotesRequest(
                symbol_or_symbols=symbol, start=start.to_pydatetime(), end=end.to_pydatetime(),
                limit=limit, sort=sort, feed=DataFeed.SIP))
            return [dict(timestamp=as_utc(q.timestamp).isoformat(), bid_price=q.bid_price,
                         ask_price=q.ask_price, bid_size=q.bid_size, ask_size=q.ask_size)
                    for q in answer.data.get(symbol, [])]
        key = request_key('quotes', symbol, start, end, limit, str(sort))
        return [SimpleNamespace(**dict(q, timestamp=pd.Timestamp(q['timestamp'])))
                for q in self._cached(f'quotes/{ny_day(start)}', key, fetch)]

    def trades(self, symbol, start, end):
        def fetch():
            from alpaca.data.requests import StockTradesRequest
            from alpaca.data.enums import DataFeed
            answer = self._call(self.source_data, 'get_stock_trades', StockTradesRequest(
                symbol_or_symbols=symbol, start=start.to_pydatetime(),
                end=end.to_pydatetime(), feed=DataFeed.SIP))
            return [dict(timestamp=as_utc(t.timestamp).isoformat(), size=t.size,
                         conditions=list(t.conditions or []))
                    for t in answer.data.get(symbol, [])]
        key = request_key('trades', symbol, start, end)
        return [SimpleNamespace(**dict(t, timestamp=pd.Timestamp(t['timestamp'])))
                for t in self._cached(f'trades/{ny_day(start)}', key, fetch)]


class ReplayData:
    """Never returns future timestamps; final historical bars may contain revisions."""

    def __init__(self, store, clock, settings=ReplaySettings()):
        self.store, self.clock, self.settings = store, clock, settings

    def _symbols(self, request):
        feed = getattr(request, 'feed', None)
        if feed is not None and getattr(feed, 'value', feed) != 'sip':
            raise ValueError('Der Replay-Speicher enthält nur SIP-Daten.')
        value = request.symbol_or_symbols
        return [value] if isinstance(value, str) else list(value)

    def _done(self, answer):
        self.clock.advance(self.settings.request_latency_seconds)
        return answer

    def published_until(self):
        """Latest minute-bar timestamp that is already published."""
        return self.clock.now() - pd.Timedelta(seconds=60 + self.settings.bar_delay_seconds)

    def get_stock_bars(self, request):
        symbols = self._symbols(request)
        start, end = as_utc(request.start), as_utc(request.end)
        now = self.clock.now()
        if getattr(request.timeframe, 'value', str(request.timeframe)) == '1Day':
            frame = self.store.daily(symbols, start, end)
            today = now.tz_convert(NY).strftime('%Y-%m-%d')
            frame = frame[frame.timestamp.dt.tz_convert(NY).dt.strftime('%Y-%m-%d') < today]
        else:
            days = [s['date'] for s in self.store.sessions(ny_day(start), ny_day(min(end, now)))]
            parts = [part for part in (self.store.day_minutes(symbols, day) for day in days) if len(part)]
            frame = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=BAR_COLUMNS)
            frame = frame[frame.timestamp <= self.published_until()]
        frame = frame[(frame.timestamp >= start) & (frame.timestamp <= end)]
        frame = frame.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
        data = {symbol: [True] * int((frame.symbol == symbol).sum())
                for symbol in frame.symbol.unique()}
        return self._done(SimpleNamespace(data=data, df=frame))

    def _clamped(self, request):
        start, end = as_utc(request.start), min(as_utc(request.end), self.clock.now())
        return start, end

    def get_stock_quotes(self, request):
        (symbol,) = self._symbols(request)
        start, end = self._clamped(request)
        quotes = [] if end < start else self.store.quotes(
            symbol, start, end, request.limit, request.sort)
        quotes = [q for q in quotes if q.timestamp <= self.clock.now()]
        return self._done(SimpleNamespace(data={symbol: quotes} if quotes else {}))

    def get_stock_trades(self, request):
        (symbol,) = self._symbols(request)
        start, end = self._clamped(request)
        trades = [] if end < start else self.store.trades(symbol, start, end)
        return self._done(SimpleNamespace(data={symbol: trades} if trades else {}))


class ReplayBroker:
    """Read-only broker view of the replay: calendar and an empty virtual account."""

    def __init__(self, store, clock):
        self.store, self.clock = store, clock

    def get_clock(self):
        return SimpleNamespace(timestamp=self.clock.now().to_pydatetime(), is_open=None)

    def get_calendar(self, request):
        return self.store.calendar(request.start, request.end)

    def get_account(self):
        return SimpleNamespace(equity='10000', cash='10000', currency='USD',
                               account_blocked=False, trading_blocked=False)

    def get_all_positions(self):
        return []

    def get_orders(self, filter=None):
        return []


def replay_settings_default():
    return replace(sim.DryRunSettings(), print_every_minute=False, save_minute_snapshots=False)


def code_hashes(include_cockpit=False):
    """Replay identity follows trading logic; preserve UI provenance separately."""
    modules = (base, scanner, shadow, sim, sys.modules[__name__])
    if include_cockpit:
        modules += (cockpit_module,)
    return {Path(m.__file__).name: hashlib.sha256(Path(m.__file__).read_bytes()).hexdigest()
            for m in modules}


def completed_run(project_root, day, fingerprint):
    """An earlier replay of `day` with identical code and settings, if any."""
    for manifest in sorted((Path(project_root) / 'shadow_replay_v1_5' / day).glob('*/manifest.json')):
        folder = manifest.parent
        status = folder / 'status.json'
        coverage = folder / 'coverage.json'
        if not status.exists() or not coverage.exists() or (folder / 'failed.json').exists():
            continue
        state = json.loads(status.read_text())
        quality = json.loads(coverage.read_text())
        if state.get('status') not in ('SHADOW_SIM_COMPLETED_NO_ORDERS',
                                       'SHADOW_SIM_INCOMPLETE_OR_DATA_DEGRADED_NO_ORDERS') or \
                not quality.get('planned_close_checked') or \
                quality.get('data_error_minutes', 0) or quality.get('missed_decision_minutes', 0):
            continue
        replay = json.loads(manifest.read_text()).get('replay') or {}
        if replay.get('fingerprint') == fingerprint:
            return folder
    return None


def replay_day(store, day, project_root, c, s, settings, rs=ReplaySettings(),
               fingerprint=None, cockpit=None):
    """Replay one NY trading day; returns the run folder."""
    (session,) = store.sessions(day, day)
    past = store.sessions(date.fromisoformat(day) - timedelta(days=70), day)[:-1][-c.rvol_days:]
    universe = store.universe()
    symbols = sorted(set(universe.symbol) | {'QQQ'})
    for reference in past + [session]:   # warm the cache in real time, not simulated time
        store.day_minutes(symbols, reference['date'])
    store.keep_days({r['date'] for r in past} | {day})
    clock = ReplayClock(session['open'] - pd.Timedelta(minutes=rs.start_before_open_minutes))
    info = dict(version=VERSION, fingerprint=fingerprint, replay_settings=asdict(rs),
                universe=store.universe_meta(), cache_folder=str(store.folder),
                cockpit_sha256=code_hashes(include_cockpit=True).get(
                    Path(cockpit_module.__file__).name),
                note='Keine späteren Minuten-Zeitstempel. Historische SIP-Kerzen können '
                     'später eingearbeitete Trades/Korrekturen enthalten; reale '
                     'Veröffentlichungszeitpunkte sind nicht rekonstruierbar.')
    return sim.run_dryrun(ReplayBroker(store, clock), ReplayData(store, clock, rs), project_root,
                          replace(c, test_date=day), s, settings, clock=clock,
                          universe_loader=lambda broker: universe, replay=info,
                          cockpit=cockpit)


def refresh_saved_cockpit(folder, cockpit):
    """Delegate presentation-only rendering to the separately hashed UI."""
    if cockpit is not None:
        ui_hash = code_hashes(include_cockpit=True)[Path(cockpit_module.__file__).name]
        cockpit_module.refresh_saved_cockpit(folder, cockpit, ui_hash)


def run_replay(store, project_root, c, s, start, end, settings=None, rs=ReplaySettings(),
               reuse_completed=True, cockpit=None):
    """Replay all trading days in [start, end]; resumable, then summarize."""
    settings = settings or replay_settings_default()
    fingerprint = request_key(code_hashes(), asdict(c), asdict(s), asdict(settings), asdict(rs))
    days = [r['date'] for r in store.sessions(start, end)]
    folders, failures = [], []
    for number, day in enumerate(days, 1):
        folder = completed_run(project_root, day, fingerprint) if reuse_completed else None
        if folder is None:
            try:
                folder = replay_day(store, day, project_root, c, s, settings, rs,
                                    fingerprint, cockpit)
            except Exception as exc:
                failures.append(dict(date=day, error_type=type(exc).__name__, detail=str(exc)[:500]))
                print(f'{day}: FEHLER {type(exc).__name__}: {str(exc)[:200]}')
                continue
        elif cockpit is not None:
            try:
                refresh_saved_cockpit(folder, cockpit)
            except Exception as exc:
                print(f'{day}: Cockpit-Ansicht konnte nicht erneuert werden: {type(exc).__name__}')
        folders.append(folder)
        state = json.loads((folder / 'shadow_state.json').read_text())
        print(f'Replay {number}/{len(days)} {day}: {state["virtual_trades"]} Trades, '
              f'PnL {state["realized_pnl_usd"]:.2f} USD')
    return summarize(folders, Path(project_root) / 'shadow_replay_v1_5' / 'summaries' / (
        pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]),
        failures, dict(start=str(start), end=str(end), fingerprint=fingerprint,
                       replay_settings=asdict(rs), version=VERSION))


def summarize(folders, out, failures=(), meta=None):
    """Aggregate day folders into days.csv, trades.csv and summary.json."""
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    days, trades = [], []
    for folder in folders:
        folder = Path(folder)
        state = json.loads((folder / 'shadow_state.json').read_text())
        status = json.loads((folder / 'status.json').read_text())
        day = folder.parent.name
        days.append(dict(date=day, status=status['status'], trades=state['virtual_trades'],
                         pnl_usd=state['realized_pnl_usd'],
                         pnl_10bp_stress_usd=state['realized_pnl_10bp_stress_usd'],
                         max_drawdown_proxy_usd=state['max_drawdown_proxy_usd'],
                         open_symbols=len(state['open_symbols']),
                         data_degraded=state['data_degraded'],
                         unreliable_trades=state['unreliable_virtual_trades'],
                         unreliable_pnl_proxy_usd=state['pnl_proxy_from_unreliable_trades_usd'],
                         folder=str(folder)))
        for trade in json.loads((folder / 'shadow_trades.json').read_text()):
            notional = trade['qty'] * trade['entry']
            trades.append(dict(date=day, **{k: trade[k] for k in (
                'symbol', 'sector', 'qty', 'entry', 'exit_price_proxy', 'exit_reason',
                'entry_at_utc', 'exit_at_utc', 'pnl_usd', 'pnl_10bp_stress_usd',
                'rank', 'score', 'rvol', 'unobserved_minutes',
                'outcome_unreliable', 'exit_trigger_bar_minute_utc')},
                return_pct=100 * trade['pnl_usd'] / notional,
                return_10bp_stress_pct=100 * trade['pnl_10bp_stress_usd'] / notional))
    day_frame = pd.DataFrame(days)
    trade_frame = pd.DataFrame(trades)
    sim.save_frame(out / 'days.csv', day_frame)
    sim.save_frame(out / 'trades.csv', trade_frame)
    summary = dict(meta or {}, days_replayed=len(days), days_failed=len(failures),
                   failures=list(failures), orders_sent=0)
    if len(day_frame):
        cumulative = day_frame.pnl_usd.cumsum()
        summary.update(
            days_with_trades=int((day_frame.trades > 0).sum()),
            days_data_degraded=int(day_frame.data_degraded.sum()),
            unreliable_trades=int(day_frame.unreliable_trades.sum()),
            pnl_proxy_from_unreliable_trades_usd=float(day_frame.unreliable_pnl_proxy_usd.sum()),
            days_with_unresolved_positions=int((day_frame.open_symbols > 0).sum()),
            total_pnl_usd=float(day_frame.pnl_usd.sum()),
            total_pnl_10bp_stress_usd=float(day_frame.pnl_10bp_stress_usd.sum()),
            max_drawdown_of_daily_pnl_usd=float((cumulative.cummax().clip(lower=0) - cumulative).max()),
            best_day_usd=float(day_frame.pnl_usd.max()), worst_day_usd=float(day_frame.pnl_usd.min()))
    if len(trade_frame):
        wins = trade_frame[trade_frame.pnl_usd > 0].pnl_usd.sum()
        losses = -trade_frame[trade_frame.pnl_usd < 0].pnl_usd.sum()
        summary.update(
            trades=len(trade_frame),
            win_rate=float((trade_frame.pnl_usd > 0).mean()),
            average_trade_usd=float(trade_frame.pnl_usd.mean()),
            average_return_pct=float(trade_frame.return_pct.mean()),
            average_return_10bp_stress_pct=float(trade_frame.return_10bp_stress_pct.mean()),
            profit_factor=float(wins / losses) if losses > 0 else None,
            exit_reasons=trade_frame.exit_reason.value_counts().to_dict())
    summary['warning'] = ('Modellergebnis mit historischen SIP-Daten, aktuellem Index-Universum '
                          '(Survivorship-Bias) und virtuellen Ausführungen; historische '
                          'Kerzen können spätere Korrekturen enthalten; kein Rentabilitätsnachweis.')
    sim.save_json(out / 'summary.json', {k: (None if isinstance(v, float) and not math.isfinite(v) else v)
                                         for k, v in summary.items()})
    print('Replay-Auswertung:', out)
    return out


# ---------------------------------------------------------------------------
# Delayed live run: the live loop on today's SIP data, `lag_minutes` behind
# real time, for accounts without real-time SIP (free plan: SIP >= 15 min old).
# ---------------------------------------------------------------------------

class DelayedClock:
    """Real time minus `lag_minutes`. Sleeps and requests take real time."""

    def __init__(self, lag_minutes=16, base=None):
        if lag_minutes < 16:
            raise ValueError('Mindestens 16 Minuten Versatz: SIP ohne Abo erst nach 15 Minuten.')
        self.base = base or sim.SYSTEM_CLOCK
        self.lag = pd.Timedelta(minutes=lag_minutes)

    @property
    def lag_seconds(self):
        return self.lag.total_seconds()

    def now(self):
        return self.base.now() - self.lag

    def sleep(self, seconds):
        self.base.sleep(seconds)

    def advance(self, seconds):
        pass   # real requests already take real time


class DelayedData:
    """Alpaca data client limited to what was published at the delayed time.

    Every request end is cut to the delayed clock, so nothing younger than the
    lag is ever requested; minute bars appear `bar_delay_seconds` after their
    minute, quotes and trades up to the delayed now."""

    def __init__(self, source, clock, bar_delay_seconds=2.0):
        self.source, self.clock, self.bar_delay_seconds = source, clock, bar_delay_seconds

    def _clamped(self, request):
        start, end = as_utc(request.start), min(as_utc(request.end), self.clock.now())
        return request.model_copy(update={'end': end.to_pydatetime()}), start, end

    def get_stock_bars(self, request):
        clamped, start, end = self._clamped(request)
        now = self.clock.now()
        frame = pd.DataFrame(columns=BAR_COLUMNS)
        if end > start:
            frame = bars_frame(self.source.get_stock_bars(clamped))
        if getattr(request.timeframe, 'value', str(request.timeframe)) == '1Day':
            today = now.tz_convert(NY).strftime('%Y-%m-%d')
            frame = frame[frame.timestamp.dt.tz_convert(NY).dt.strftime('%Y-%m-%d') < today]
        else:
            published = now - pd.Timedelta(seconds=60 + self.bar_delay_seconds)
            frame = frame[frame.timestamp <= published]
        frame = frame[(frame.timestamp >= start) & (frame.timestamp <= end)]
        frame = frame.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
        data = {symbol: [True] * int((frame.symbol == symbol).sum())
                for symbol in frame.symbol.unique()}
        return SimpleNamespace(data=data, df=frame)

    def _items(self, method, request):
        clamped, start, end = self._clamped(request)
        (symbol,) = [request.symbol_or_symbols] if isinstance(request.symbol_or_symbols, str) \
            else list(request.symbol_or_symbols)
        if end < start:
            return SimpleNamespace(data={})
        items = [item for item in getattr(self.source, method)(clamped).data.get(symbol, [])
                 if as_utc(item.timestamp) <= self.clock.now()]
        return SimpleNamespace(data={symbol: items} if items else {})

    def get_stock_quotes(self, request):
        return self._items('get_stock_quotes', request)

    def get_stock_trades(self, request):
        return self._items('get_stock_trades', request)


class DelayedBroker:
    """Read-only view of the real paper account; the clock is the delayed one.

    Only read methods exist here - there is nothing to place an order with."""

    def __init__(self, broker, clock):
        self._broker, self.clock = broker, clock

    def get_clock(self):
        return SimpleNamespace(timestamp=self.clock.now().to_pydatetime(), is_open=None)

    def get_calendar(self, request):
        return self._broker.get_calendar(request)

    def get_account(self):
        return self._broker.get_account()

    def get_all_positions(self):
        return self._broker.get_all_positions()

    def get_orders(self, filter=None):
        return self._broker.get_orders(filter=filter)

    def get_all_assets(self, filter=None):
        return self._broker.get_all_assets(filter=filter)


def run_delayed(broker, data, project_root, c, s, settings=None, cockpit=None,
                lag_minutes=16, rs=ReplaySettings(), base_clock=None,
                universe_loader=sim.fresh_universe):
    """Today's session through the unchanged live loop, `lag_minutes` behind."""
    clock = DelayedClock(lag_minutes, base_clock)
    settings = settings or replace(sim.DryRunSettings(), print_every_minute=False)
    info = dict(kind='delayed_sip', version=VERSION, lag_minutes=lag_minutes,
                mode='DELAYED_SIP_VIRTUAL_PORTFOLIO', output_folder='shadow_delayed_v1_5',
                cockpit_mode='delayed', access_result='DELAYED_HISTORICAL_SIP',
                bar_delay_seconds=rs.bar_delay_seconds,
                note=f'Heutige SIP-Daten, {lag_minutes} Minuten hinter Echtzeit; '
                     'gleiche Live-Schleife, keine Echtzeit-Lieferung getestet.')
    print(f'Verzögerter Lauf: Bot-Uhr {lag_minutes} Min. hinter Echtzeit '
          f'(jetzt {clock.now().tz_convert(NY):%H:%M} NY).')
    return sim.run_dryrun(DelayedBroker(broker, clock), DelayedData(data, clock, rs.bar_delay_seconds),
                          project_root, c, s, settings, clock=clock,
                          universe_loader=universe_loader, replay=info, cockpit=cockpit)


In [ ]:
%%writefile /content/bot_accounts_v15.py
"""Persistent per-bot virtual accounts and logs (V1.5). Pure bookkeeping, no network.

Each bot has its own folder `<project>/v1_5/<BOT_ID>/` with
- `account.json`: start capital and every booked trading day (booked once per day),
- `<BOT_ID>_trades.csv`, `<BOT_ID>_signals.csv`, `<BOT_ID>_daily_performance.csv`:
  derived views, rebuilt from the run folders of the booked days.
Run folders are never modified; a restarted day gets a new run folder and only
the first complete run of a day is booked (no double booking)."""
from pathlib import Path
import json

import pandas as pd

TRADE_FIELDS = ('symbol', 'sector', 'qty', 'entry', 'entry_at_utc', 'signal_at_utc',
                'entry_quote_at_utc', 'exit_price_proxy', 'exit_at_utc', 'exit_reason',
                'exit_trigger_bar_minute_utc', 'pnl_usd', 'pnl_10bp_stress_usd',
                'mfe_usd', 'mae_usd', 'rank', 'score', 'rvol', 'stop',
                'unobserved_minutes', 'outcome_unreliable')
SIGNAL_FIELDS = ('symbol', 'decision_time_utc', 'processing_utc', 'feed', 'status',
                 'previous_status', 'shadow_status', 'paper_order', 'or_high', 'or_low',
                 'threshold', 'price_cap', 'close', 'rvol', 'rvol_status',
                 'complete_reference_days', 'current_volume', 'reference_average',
                 'relative_strength', 'spread', 'bid', 'ask', 'score', 'rank',
                 'rvol_component', 'strength_component', 'breakout_component',
                 'spread_component', 'ranking_result', 'opening_bars', 'coverage',
                 'previous_close_age_minutes', 'error_type')


def read_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else default


def write_json(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.incomplete')
    temporary.write_text(json.dumps(content, ensure_ascii=False, indent=2, default=str) + '\n',
                         encoding='utf-8')
    temporary.replace(path)


def write_csv(path, frame):
    path = Path(path)
    temporary = path.with_name(path.name + '.incomplete')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def run_is_complete(folder):
    """A run can be booked when the close was processed and nothing stayed open."""
    folder = Path(folder)
    state = read_json(folder / 'shadow_state.json')
    coverage = read_json(folder / 'coverage.json')
    if not state or not coverage or (folder / 'failed.json').exists():
        return False
    return bool(coverage.get('planned_close_checked')) and not state.get('open_symbols')


class BotLedger:
    """Virtual account of one bot across days (e.g. SIP_DELAYED_SHADOW)."""

    def __init__(self, root, bot_id, start_cash=10000.0):
        self.bot_id = bot_id
        self.folder = Path(root) / bot_id
        self.folder.mkdir(parents=True, exist_ok=True)
        self.path = self.folder / 'account.json'
        state = read_json(self.path)
        if state is None:
            state = dict(bot_id=bot_id, start_cash=float(start_cash),
                         created_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                         days={}, rejected_bookings=[])
            write_json(self.path, state)
        if state['bot_id'] != bot_id:
            raise ValueError('Kontodatei gehört zu einem anderen Bot.')
        self.state = state

    def save(self):
        write_json(self.path, self.state)

    def booked_days(self):
        return sorted(self.state['days'])

    def is_booked(self, day):
        return day in self.state['days']

    def start_equity(self, day):
        """Start capital plus the P&L of all booked days before `day`."""
        return self.state['start_cash'] + sum(
            entry['pnl_usd'] for d, entry in self.state['days'].items() if d < day)

    def equity(self):
        return self.state['start_cash'] + sum(e['pnl_usd'] for e in self.state['days'].values())

    def book(self, day, folder):
        """Book a complete run of `day` once. Returns True if booked now."""
        folder = Path(folder)
        if self.is_booked(day):
            self.state['rejected_bookings'].append(dict(
                date=day, folder=str(folder), reason='DAY_ALREADY_BOOKED',
                at_utc=pd.Timestamp.now(tz='UTC').isoformat()))
            self.save()
            return False
        if not run_is_complete(folder):
            self.state['rejected_bookings'].append(dict(
                date=day, folder=str(folder), reason='RUN_NOT_COMPLETE',
                at_utc=pd.Timestamp.now(tz='UTC').isoformat()))
            self.save()
            return False
        state = read_json(folder / 'shadow_state.json')
        status = read_json(folder / 'status.json', {})
        start = self.start_equity(day)
        self.state['days'][day] = dict(
            run_folder=str(folder), start_equity_usd=start,
            pnl_usd=float(state['realized_pnl_usd']),
            pnl_10bp_stress_usd=float(state['realized_pnl_10bp_stress_usd']),
            end_equity_usd=start + float(state['realized_pnl_usd']),
            trades=int(state['virtual_trades']),
            unreliable_trades=int(state.get('unreliable_virtual_trades', 0)),
            max_drawdown_proxy_usd=float(state['max_drawdown_proxy_usd']),
            data_degraded=bool(state['data_degraded']),
            status=status.get('status'), session_scope=status.get('session_scope'),
            booked_utc=pd.Timestamp.now(tz='UTC').isoformat())
        self.save()
        return True

    # ---- derived CSV logs ----
    def export_logs(self):
        trades, signals, daily = [], [], []
        for day in self.booked_days():
            entry = self.state['days'][day]
            folder = Path(entry['run_folder'])
            daily.append(dict(bot_id=self.bot_id, date=day, **{k: v for k, v in entry.items()
                                                                if k != 'run_folder'},
                              run_folder=entry['run_folder'],
                              return_pct=100 * entry['pnl_usd'] / entry['start_equity_usd']))
            for number, trade in enumerate(read_json(folder / 'shadow_trades.json', [])):
                trades.append(dict(bot_id=self.bot_id, date=day,
                                   trade_id=f'{self.bot_id}|{day}|{trade["symbol"]}|{number}',
                                   **{k: trade.get(k) for k in TRADE_FIELDS}))
            signals.extend(dict(bot_id=self.bot_id, date=day, **row)
                           for row in signal_rows(folder, self.bot_id, day))
        write_csv(self.folder / f'{self.bot_id}_trades.csv',
                  pd.DataFrame(trades, columns=['bot_id', 'date', 'trade_id', *TRADE_FIELDS]))
        write_csv(self.folder / f'{self.bot_id}_signals.csv',
                  pd.DataFrame(signals, columns=['bot_id', 'date', 'signal_id', *SIGNAL_FIELDS]))
        write_csv(self.folder / f'{self.bot_id}_daily_performance.csv', pd.DataFrame(daily))
        return self.folder


def signal_rows(folder, bot_id, day):
    """All detected signals of a run (historical, rejected, ranked, entered)."""
    rows = []
    for path in sorted((Path(folder) / 'signals').glob('*.json')):
        item = read_json(path)
        if not isinstance(item, dict) or 'symbol' not in item:
            continue   # per-minute summaries and skip markers
        if isinstance(item.get('paper_order'), dict):
            item['paper_order'] = json.dumps(item['paper_order'], ensure_ascii=False)
        rows.append(dict(signal_id=f'{bot_id}|{day}|{item["symbol"]}|{item.get("decision_time_utc")}',
                         **{k: item.get(k) for k in SIGNAL_FIELDS}))
    return rows


In [ ]:
%%writefile /content/sip_shadow_bot_v15.py
"""BOT 2 – SIP-Delayed-Bot (SIP_DELAYED_SHADOW). Virtual only, by construction.

This module never imports `alpaca.trading` and never receives a trading client.
It gets a market-data client (SIP, historical: >= 16 minutes old), the exchange
calendar and the stock universe as plain data, and a VirtualBroker that has no
network access and no order methods. All transactions are booked internally.

A (re)start re-simulates the session chronologically from the open ("catch-up")
and then follows real time with the lag; the day is booked into the virtual
account only once, from the first complete run.
"""
from dataclasses import replace
from datetime import date, datetime
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
import us_orb_test_v02 as base
import orb_sim_v15 as sim
import orb_replay_v15 as replay
import orb_cockpit_v15 as cockpit_ui
import bot_accounts_v15 as accounts

BOT_ID = 'SIP_DELAYED_SHADOW'
DISPLAY_NAME = 'BOT 2 – SIP-Delayed-Bot'
LAG_MINUTES = 16
NY = 'America/New_York'


def calendar_payload(rows):
    """Alpaca calendar entries -> plain dicts (picklable, no client needed)."""
    return [dict(date=str(r.date), open=str(r.open), close=str(r.close)) for r in rows]


def calendar_objects(payload, start=None, end=None):
    start, end = str(start or '0000')[:10], str(end or '9999')[:10]
    return [SimpleNamespace(date=date.fromisoformat(r['date']),
                            open=datetime.fromisoformat(r['open']),
                            close=datetime.fromisoformat(r['close']))
            for r in payload if start <= r['date'] <= end]


class VirtualBroker:
    """Calendar and the bot's own virtual account. No network, no order methods."""

    def __init__(self, calendar, clock, equity):
        self._calendar, self.clock, self._equity = list(calendar), clock, float(equity)

    def get_clock(self):
        return SimpleNamespace(timestamp=self.clock.now().to_pydatetime(), is_open=None)

    def get_calendar(self, request):
        return calendar_objects(self._calendar, request.start, request.end)

    def get_account(self):
        return SimpleNamespace(equity=str(self._equity), cash=str(self._equity), currency='USD',
                               account_blocked=False, trading_blocked=False)

    def get_all_positions(self):
        return []

    def get_orders(self, filter=None):
        return []


class CatchUpClock(replay.DelayedClock):
    """Delayed clock that first replays the elapsed session quickly.

    While `now()` is behind (real time - lag) it runs simulated (sleep advances
    instantly); once it has caught up it follows real time minus the lag."""

    def __init__(self, lag_minutes=LAG_MINUTES, start=None, base=None):
        super().__init__(lag_minutes, base)
        self.simulated = None
        if start is not None and replay.as_utc(start) < self.target():
            self.simulated = replay.as_utc(start)

    def target(self):
        return self.base.now() - self.lag

    def catching_up(self):
        return self.simulated is not None

    @property
    def lag_seconds(self):
        return (self.base.now() - self.now()).total_seconds()

    def now(self):
        return self.simulated if self.simulated is not None else self.target()

    def _step(self, seconds):
        self.simulated += pd.Timedelta(seconds=seconds)
        if self.simulated >= self.target():
            self.simulated = None      # caught up: from now on real time minus lag

    def sleep(self, seconds):
        if self.simulated is None:
            self.base.sleep(seconds)
        else:
            self._step(seconds)

    def advance(self, seconds):
        if self.simulated is not None:
            self._step(seconds)


def session_for(calendar, day):
    rows = base.calendar_rows(calendar_objects(calendar, day, day))
    if len(rows) != 1:
        raise ValueError(f'{day} ist laut Börsenkalender kein Handelstag.')
    return rows[0]


def run_sip_shadow_day(data_client, project_root, c, s, calendar, universe,
                       cockpit=None, lag_minutes=LAG_MINUTES, base_clock=None,
                       rs=replay.ReplaySettings(), start_cash=10000.0):
    """Today's session for BOT 2; returns the run folder (or the booked one)."""
    root = Path(project_root) / 'v1_5'
    ledger = accounts.BotLedger(root, BOT_ID, start_cash)
    probe = replay.DelayedClock(lag_minutes, base_clock)
    day = probe.now().tz_convert(NY).strftime('%Y-%m-%d')
    session = session_for(calendar, day)
    if ledger.is_booked(day):
        print(f'{DISPLAY_NAME}: {day} ist bereits vollständig gebucht; keine Doppelbuchung.')
        return Path(ledger.state['days'][day]['run_folder'])
    clock = CatchUpClock(lag_minutes, session['open'] - pd.Timedelta(minutes=10), base_clock)
    if clock.catching_up():
        print(f'{DISPLAY_NAME}: Neustart/Spätstart – der Tag wird ab Börsenöffnung '
              'chronologisch nachsimuliert, dann folgt der Bot mit '
              f'{lag_minutes} Min. Versatz.')
    equity = ledger.start_equity(day)
    settings = replace(sim.DryRunSettings(), feed='sip', initial_cash=equity,
                       print_every_minute=False)
    info = dict(kind='sip_delayed_shadow', bot_id=BOT_ID, display_name=DISPLAY_NAME,
                version=replay.VERSION, lag_minutes=lag_minutes,
                mode='SIP_DELAYED_SHADOW_VIRTUAL', output_folder=f'v1_5/{BOT_ID}/runs',
                cockpit_mode='sip_shadow', access_result='DELAYED_HISTORICAL_SIP',
                bar_delay_seconds=rs.bar_delay_seconds, catch_up_from_open=clock.catching_up(),
                note='SIMULIERT: historische SIP-Daten mit Versatz; keine Broker-Orders.')
    out = sim.run_dryrun(VirtualBroker(calendar, clock, equity),
                         replay.DelayedData(data_client, clock, rs.bar_delay_seconds),
                         project_root, replace(c, test_date=day), s, settings, clock=clock,
                         universe_loader=lambda broker: universe, replay=info, cockpit=cockpit)
    if ledger.book(day, out):
        print(f'{DISPLAY_NAME}: Tag {day} im virtuellen Konto gebucht.')
    ledger.export_logs()
    return out


def alpaca_market_data(api_key, secret_key):
    """Market-data client only (StockHistoricalDataClient cannot place orders)."""
    from alpaca.data.historical import StockHistoricalDataClient
    return StockHistoricalDataClient(api_key=api_key, secret_key=secret_key)


def child_main(data_factory, factory_kwargs, project_root, c, s, calendar, universe_records,
               cockpit_folder, lag_minutes=LAG_MINUTES, clock_factory=None):
    """Entry point of BOT 2 in its own process (DUAL_MODE); only picklable inputs.

    `data_factory` ('module:function') builds the market-data client;
    `clock_factory` (tests only) builds the base clock."""
    import importlib

    def load(spec):
        module, name = spec.split(':')
        return getattr(importlib.import_module(module), name)
    data_client = load(data_factory)(**factory_kwargs)
    base_clock = load(clock_factory)() if clock_factory else None
    cockpit = cockpit_ui.Cockpit(display_enabled=False,
                                 latest_path=Path(cockpit_folder) / 'cockpit_latest.html',
                                 latest_json_path=Path(cockpit_folder) / 'cockpit_latest.json')
    run_sip_shadow_day(data_client, project_root, c, s, calendar,
                       pd.DataFrame(universe_records), cockpit=cockpit, lag_minutes=lag_minutes,
                       base_clock=base_clock)


In [ ]:
%%writefile /content/paper_orders_v15.py
"""Alpaca PAPER order gateway for BOT 1 (IEX_REALTIME_PAPER). Only BOT 1 imports this.

Every order path runs through `_guard()`:
- the client must point at https://paper-api.alpaca.markets (never live),
- the account must be ACTIVE, USD, not blocked and match the approved account,
- the approval must be exactly `approval_token(account_number, day)` for today,
- positions/orders are reconciled with Alpaca right before each order.
Without a valid approval the gateway stays disabled and sends nothing.

Orders: IOC limit buy at the fresh ask (partial fills allowed), then a broker-side
DAY trailing stop for the filled quantity; flatten at the planned close and
confirm position 0. Client order ids are deterministic per day and symbol, so a
restart can never submit a second entry for the same signal. Missed signals are
never executed later.
"""
from pathlib import Path
import json
import time

import pandas as pd
import us_orb_test_v02 as base

PAPER_URL = 'https://paper-api.alpaca.markets'
PREFIX = 'ORB15'
TERMINAL = {'filled', 'canceled', 'expired', 'rejected', 'done_for_day', 'stopped', 'suspended'}


def approval_token(account_number, day):
    """The exact text a user must set to allow paper orders on `day` for this account."""
    return f'PAPER-ORDERS-OK {account_number} {day}'


def client_id(day, symbol, kind):
    return f'{PREFIX}-{day}-{symbol}-{kind}'


def value(item):
    return getattr(item, 'value', item)


class PaperOrderGateway:
    """Order hook for sim.run_dryrun (BOT 1 only)."""

    def __init__(self, trading_client, approval, day, config, scanner_config, sectors,
                 clock, log_folder, fill_timeout_seconds=10, sleep=time.sleep):
        self.client, self.approval, self.day = trading_client, approval, day
        self.c, self.s, self.sectors, self.clock = config, scanner_config, dict(sectors), clock
        self.log_path = Path(log_folder) / 'IEX_REALTIME_PAPER_orders.jsonl'
        self.fill_timeout_seconds, self.sleep = fill_timeout_seconds, sleep
        self.enabled, self.disabled_reason = False, 'NOT_STARTED'
        self.orders_sent, self.start_equity, self.flat_confirmed = 0, None, None
        self.account_number = None

    # ---------- logging ----------
    def log(self, kind, **fields):
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        record = dict(kind=kind, processing_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                      market_time_utc=self.clock.now().isoformat(), day=self.day, **fields)
        with self.log_path.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')
        return record

    def disable(self, reason):
        self.enabled, self.disabled_reason = False, reason
        self.log('ORDERS_DISABLED', reason=reason)

    # ---------- safety ----------
    def _paper_endpoint(self):
        url = str(value(getattr(self.client, '_base_url', ''))).rstrip('/')
        return url == PAPER_URL and getattr(self.client, '_sandbox', False) is True

    def _guard(self):
        """Raise unless every precondition for a paper order holds right now."""
        if not self._paper_endpoint():
            raise PermissionError('Kein Alpaca-Paper-Endpunkt – Orders gesperrt.')
        account = self.client.get_account()
        if value(account.status) != 'ACTIVE' or account.account_blocked or \
                account.trading_blocked or account.currency != 'USD':
            raise PermissionError('Paper-Konto nicht aktiv/gesperrt/nicht USD.')
        if self.account_number is not None and account.account_number != self.account_number:
            raise PermissionError('Anderes Paper-Konto als freigegeben.')
        if self.approval != approval_token(account.account_number, self.day):
            raise PermissionError('Keine gültige Order-Freigabe für heute und dieses Konto.')
        return account

    def _own(self, order):
        return str(getattr(order, 'client_order_id', '') or '').startswith(f'{PREFIX}-{self.day}-')

    def reconcile(self):
        """Current Alpaca state; every own position must have an own open stop."""
        from alpaca.trading.requests import GetOrdersRequest
        from alpaca.trading.enums import QueryOrderStatus
        positions = {p.symbol: p for p in self.client.get_all_positions()}
        orders = self.client.get_orders(filter=GetOrdersRequest(status=QueryOrderStatus.OPEN, limit=500))
        if len(orders) >= 500:
            raise PermissionError('Offene Orderliste möglicherweise abgeschnitten.')
        return positions, orders

    # ---------- run_dryrun hook interface ----------
    def on_start(self, out, clock):
        self.clock = clock
        try:
            if not self.approval:
                raise PermissionError('Keine Order-Freigabe gesetzt.')
            account = self._guard()
            self.account_number = account.account_number
            positions, orders = self.reconcile()
            foreign = [p for p in positions if not any(
                self._own(o) and o.symbol == p for o in orders)]
            foreign_orders = [o for o in orders if not self._own(o)]
            if foreign or foreign_orders:
                raise PermissionError('Paper-Konto hat fremde Positionen/Orders; bitte zuerst bereinigen.')
            self.start_equity = float(account.equity)
            self.enabled, self.disabled_reason = True, None
            self.log('ORDERS_ENABLED', account_number=account.account_number,
                     equity=self.start_equity, cash=float(account.cash),
                     adopted_positions=sorted(positions), open_own_orders=len(orders))
            self.ensure_stops(positions, orders)
        except Exception as exc:
            self.disable(f'{type(exc).__name__}: {exc}')

    def on_signal(self, decision, now, signal, quality):
        """Place a paper entry for a fresh, ranked watchlist signal (or say why not)."""
        symbol = signal['symbol']
        if not self.enabled:
            return dict(status='NOT_SENT', reason=self.disabled_reason)
        try:
            if (pd.Timestamp(now) - pd.Timestamp(decision)).total_seconds() >= self.s.signal_ttl_seconds:
                return self._skip(symbol, 'SIGNAL_EXPIRED')
            if quality.get('quote_result') != 'PASS':
                return self._skip(symbol, 'QUOTE_' + str(quality.get('quote_result')))
            quote_age = (pd.Timestamp(now) - pd.Timestamp(quality['quote_time_utc'])).total_seconds()
            if not 0 <= quote_age <= 2:
                return self._skip(symbol, 'QUOTE_NOT_FRESH')
            account = self._guard()
            positions, orders = self.reconcile()
            entry_id = client_id(self.day, symbol, 'entry')
            if self._order_exists(entry_id):
                return self._skip(symbol, 'ENTRY_ALREADY_SENT_TODAY')
            if symbol in positions or any(o.symbol == symbol for o in orders):
                return self._skip(symbol, 'SYMBOL_ALREADY_HELD_OR_PENDING')
            sectors = [self.sectors.get(p) for p in positions]
            gate = base.entry_gate(self.start_equity, float(account.equity), 0, sectors,
                                   signal['sector'], False, self.c)
            if not gate['allowed']:
                return self._skip(symbol, ';'.join(gate['reasons']))
            risk = sum(abs(float(p.qty)) * float(p.current_price) * float(self.c.trailing)
                       for p in positions.values())
            ask = float(quality['ask'])
            size = base.size_position(float(account.equity), float(account.cash), risk, ask, self.c)
            if size['qty'] <= 0:
                return self._skip(symbol, 'NO_CAPITAL_OR_RISK_BUDGET')
            return self._enter(symbol, int(size['qty']), ask, entry_id, signal)
        except Exception as exc:
            self.log('ENTRY_ERROR', symbol=symbol, error=f'{type(exc).__name__}: {exc}')
            return dict(status='ERROR', reason=type(exc).__name__)

    def on_minute(self, decision, now):
        if not self.enabled:
            return
        try:
            positions, orders = self.reconcile()
            self.ensure_stops(positions, orders)
        except Exception as exc:
            self.log('RECONCILE_ERROR', error=f'{type(exc).__name__}: {exc}')

    def on_planned_close(self, now):
        if self.account_number is not None:
            self.flatten('PLANNED_CLOSE')

    def on_stop(self, now):
        """End of run (normal or abort): never leave own paper positions behind."""
        if self.account_number is None:
            return
        positions, _ = self.reconcile()
        if positions:
            self.flatten('RUN_STOPPED')
        elif self.flat_confirmed is None:
            self.flat_confirmed = True
            self.log('POSITION_ZERO_CONFIRMED', reason='RUN_STOPPED')

    # ---------- order mechanics ----------
    def _skip(self, symbol, reason):
        self.log('ENTRY_SKIPPED', symbol=symbol, reason=reason)
        return dict(status='NOT_SENT', reason=reason)

    def _order_exists(self, client_order_id):
        try:
            self.client.get_order_by_client_id(client_order_id)
            return True
        except Exception:
            return False

    def _wait_terminal(self, client_order_id):
        deadline = time.monotonic() + self.fill_timeout_seconds
        while True:
            order = self.client.get_order_by_client_id(client_order_id)
            if value(order.status) in TERMINAL or time.monotonic() >= deadline:
                return order
            self.sleep(0.5)

    def _enter(self, symbol, qty, ask, entry_id, signal):
        from alpaca.trading.requests import LimitOrderRequest
        from alpaca.trading.enums import OrderSide, TimeInForce
        self._guard()
        self.client.submit_order(LimitOrderRequest(
            symbol=symbol, qty=qty, side=OrderSide.BUY, time_in_force=TimeInForce.IOC,
            limit_price=round(ask, 2), client_order_id=entry_id))
        self.orders_sent += 1
        order = self._wait_terminal(entry_id)
        filled = int(float(order.filled_qty or 0))
        price = float(order.filled_avg_price) if filled else None
        self.log('ENTRY_ORDER', symbol=symbol, qty=qty, limit=round(ask, 2), client_order_id=entry_id,
                 status=value(order.status), filled_qty=filled, filled_avg_price=price,
                 signal=signal)
        if filled:
            self._place_stop(symbol, filled)
        return dict(status=value(order.status), filled_qty=filled, filled_avg_price=price,
                    client_order_id=entry_id)

    def _place_stop(self, symbol, qty):
        from alpaca.trading.requests import TrailingStopOrderRequest
        from alpaca.trading.enums import OrderSide, TimeInForce
        stop_id = client_id(self.day, symbol, 'stop')
        if self._order_exists(stop_id):
            stop_id = client_id(self.day, symbol, f'stop{int(time.time())}')
        self._guard()
        self.client.submit_order(TrailingStopOrderRequest(
            symbol=symbol, qty=qty, side=OrderSide.SELL, time_in_force=TimeInForce.DAY,
            trail_percent=float(self.c.trailing) * 100, client_order_id=stop_id))
        self.orders_sent += 1
        self.log('TRAILING_STOP_ORDER', symbol=symbol, qty=qty, client_order_id=stop_id,
                 trail_percent=float(self.c.trailing) * 100)

    def ensure_stops(self, positions, orders):
        """Every own long position needs a broker-side sell stop for its quantity."""
        for symbol, position in positions.items():
            covered = sum(float(o.qty) for o in orders if o.symbol == symbol and
                          value(o.side) == 'sell' and self._own(o))
            missing = int(float(position.qty) - covered)
            if missing > 0:
                self.log('STOP_MISSING', symbol=symbol, missing_qty=missing)
                self._place_stop(symbol, missing)

    def flatten(self, reason):
        """Cancel own orders, close all positions, confirm position 0."""
        self._guard()
        self.client.cancel_orders()
        self.client.close_all_positions(cancel_orders=True)
        self.orders_sent += 1
        deadline = time.monotonic() + max(self.fill_timeout_seconds, 30)
        while True:
            positions = self.client.get_all_positions()
            if not positions:
                self.flat_confirmed = True
                self.log('POSITION_ZERO_CONFIRMED', reason=reason)
                return True
            if time.monotonic() >= deadline:
                self.flat_confirmed = False
                self.log('FLATTEN_NOT_CONFIRMED', reason=reason,
                         remaining=[p.symbol for p in positions])
                raise RuntimeError('Paper-Positionen nach Glattstellung nicht 0.')
            self.sleep(1)


In [ ]:
%%writefile /content/iex_paper_bot_v15.py
"""BOT 1 – IEX-Echtzeit-Bot (IEX_REALTIME_PAPER).

Runs the live loop on real-time IEX data. Always: an internal IEX simulation
with the same execution/cost model as BOT 2 (performance series 2, own virtual
account IEX_REALTIME_PAPER_SIM). Only with a valid approval for today and the
paper account: real Alpaca PAPER orders through paper_orders_v15 (series 1).
Decisions are never taken from delayed or replayed data (run_dryrun refuses an
order hook there), and missed signals are never ordered later.
"""
from dataclasses import replace
from pathlib import Path
import json

import pandas as pd
import orb_sim_v15 as sim
import bot_accounts_v15 as accounts
import paper_orders_v15 as paper

BOT_ID = 'IEX_REALTIME_PAPER'
SIM_ID = 'IEX_REALTIME_PAPER_SIM'
DISPLAY_NAME = 'BOT 1 – IEX-Echtzeit-Bot'
NY = 'America/New_York'


def run_iex_day(trading_client, data_client, project_root, c, s, universe, approval=None,
                cockpit=None, clock=sim.SYSTEM_CLOCK, gateway_factory=paper.PaperOrderGateway,
                start_cash=10000.0):
    """Today's IEX session; returns (run folder, gateway or None)."""
    root = Path(project_root) / 'v1_5'
    sim_ledger = accounts.BotLedger(root, SIM_ID, start_cash)
    day = clock.now().tz_convert(NY).strftime('%Y-%m-%d')
    settings = replace(sim.DryRunSettings(), feed='iex', print_every_minute=False,
                       initial_cash=sim_ledger.start_equity(day))
    gateway = None
    if approval:   # without any approval text the order gateway is not even created
        gateway = gateway_factory(trading_client, approval, day, c, s,
                                  dict(zip(universe.symbol, universe.sector)), clock, root / BOT_ID)
    out = sim.run_dryrun(trading_client, data_client, project_root, replace(c, test_date=day), s,
                         settings, clock=clock, universe_loader=lambda broker: universe,
                         cockpit=cockpit, order_hook=gateway,
                         output_folder=f'v1_5/{BOT_ID}/runs', cockpit_mode='iex_live')
    sim_ledger.book(day, out)
    sim_ledger.export_logs()
    record_paper_day(trading_client, gateway, root / BOT_ID, day, out)
    return out, gateway


def record_paper_day(trading_client, gateway, folder, day, run_folder):
    """Series 1 (actual paper fills) and the BOT 1 signal log."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    signals = pd.DataFrame(accounts.signal_rows(run_folder, BOT_ID, day))
    previous = folder / f'{BOT_ID}_signals.csv'
    if previous.exists() and previous.stat().st_size > 1:
        old = pd.read_csv(previous)
        signals = pd.concat([old[old.date != day], signals.assign(date=day, bot_id=BOT_ID)],
                            ignore_index=True)
    else:
        signals = signals.assign(date=day, bot_id=BOT_ID)
    accounts.write_csv(previous, signals)
    days_path = folder / 'paper_days.json'
    days = accounts.read_json(days_path, {})
    if gateway is None or gateway.account_number is None:
        days.setdefault(day, dict(orders_enabled=False, reason=getattr(gateway, 'disabled_reason',
                                                                       'NO_APPROVAL'),
                                  run_folder=str(run_folder)))
    else:
        account = trading_client.get_account()
        days[day] = dict(orders_enabled=True, run_folder=str(run_folder),
                         start_equity_usd=gateway.start_equity, end_equity_usd=float(account.equity),
                         pnl_usd=float(account.equity) - gateway.start_equity,
                         orders_sent=gateway.orders_sent, flat_confirmed=gateway.flat_confirmed)
    accounts.write_json(days_path, days)
    rows = [dict(bot_id=BOT_ID, date=d, **{k: v for k, v in e.items()}) for d, e in sorted(days.items())]
    accounts.write_csv(folder / f'{BOT_ID}_daily_performance.csv', pd.DataFrame(rows))
    fills = []
    log = folder / f'{BOT_ID}_orders.jsonl'
    if log.exists():
        for line in log.read_text(encoding='utf-8').splitlines():
            record = json.loads(line)
            if record['kind'] in ('ENTRY_ORDER', 'TRAILING_STOP_ORDER', 'POSITION_ZERO_CONFIRMED',
                                  'FLATTEN_NOT_CONFIRMED', 'ENTRY_SKIPPED', 'ORDERS_DISABLED'):
                fills.append({k: (json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v)
                              for k, v in record.items()})
    if gateway is not None and gateway.account_number is not None:
        fills.extend(paper_fills(trading_client, day, gateway))
    accounts.write_csv(folder / f'{BOT_ID}_trades.csv', pd.DataFrame(fills))


def paper_fills(trading_client, day, gateway):
    """Actual Alpaca paper fills of today for the symbols BOT 1 traded (entries,
    trailing stops and the closing orders of close_all_positions)."""
    from alpaca.trading.requests import GetOrdersRequest
    from alpaca.trading.enums import QueryOrderStatus
    traded = {line['symbol'] for line in map(json.loads,
              gateway.log_path.read_text(encoding='utf-8').splitlines())
              if line['kind'] == 'ENTRY_ORDER' and line.get('filled_qty')} if gateway.log_path.exists() else set()
    if not traded:
        return []
    after = pd.Timestamp(f'{day} 09:00', tz=NY).tz_convert('UTC').to_pydatetime()
    orders = trading_client.get_orders(filter=GetOrdersRequest(
        status=QueryOrderStatus.CLOSED, after=after, limit=500, symbols=sorted(traded)))
    return [dict(kind='ALPACA_FILL', day=day, symbol=o.symbol, side=paper.value(o.side),
                 order_type=paper.value(getattr(o, 'order_type', getattr(o, 'type', None))),
                 filled_qty=float(o.filled_qty or 0),
                 filled_avg_price=float(o.filled_avg_price) if o.filled_avg_price else None,
                 filled_at=str(getattr(o, 'filled_at', None)), client_order_id=o.client_order_id,
                 status=paper.value(o.status))
            for o in orders if float(o.filled_qty or 0) > 0]


In [ ]:
%%writefile /content/dual_bot_v15.py
"""V1.5 – Dual-Bot-System (IEX + SIP): modes, orchestration, dual cockpit, comparison.

Modes (delivery state: SIP_ONLY):
    SIP_ONLY   BOT 1 off,                    BOT 2 active (virtual)
    IEX_ONLY   BOT 1 active (orders only with a valid approval), BOT 2 off
    DUAL_MODE  both; BOT 2 runs in its own process so it can never block BOT 1

BOT 2 never receives a trading client: the main process passes it the exchange
calendar and the stock universe as plain data and the market-data keys only.
"""
from pathlib import Path
import json
import multiprocessing
import time

import pandas as pd
import orb_sim_v15 as sim
import orb_cockpit_v15 as cockpit_ui
import bot_accounts_v15 as accounts
import sip_shadow_bot_v15 as bot2

VERSION = 'V1.5 – Dual-Bot-System (IEX + SIP)'
BOT1_ID, BOT1_NAME = 'IEX_REALTIME_PAPER', 'BOT 1 – IEX-Echtzeit-Bot'
BOT2_ID, BOT2_NAME = bot2.BOT_ID, bot2.DISPLAY_NAME
MODES = {'SIP_ONLY': (False, True), 'IEX_ONLY': (True, False), 'DUAL_MODE': (True, True)}
DELIVERY_MODE = 'SIP_ONLY'
NY = 'America/New_York'


def resolve_mode(mode):
    """Unknown or missing modes fall back to the non-trading delivery state."""
    if mode not in MODES:
        print(f'Unbekannter Modus {mode!r} – verwende {DELIVERY_MODE}.')
        mode = DELIVERY_MODE
    bot1_active, bot2_active = MODES[mode]
    return dict(mode=mode, bot1_active=bot1_active, bot2_active=bot2_active)


def prepare_shared(trading_client, day):
    """Read-only preparation in the main process: calendar and one common universe."""
    from datetime import date, timedelta
    from alpaca.trading.requests import GetCalendarRequest
    today = date.fromisoformat(day)
    calendar = bot2.calendar_payload(trading_client.get_calendar(GetCalendarRequest(
        start=today - timedelta(days=90), end=today + timedelta(days=7))))
    universe = sim.fresh_universe(trading_client)
    return calendar, universe


# ---------------------------------------------------------------- dual cockpit
class DualCockpit:
    """One page with three areas: BOT 1, BOT 2 (SIMULIERT) and the comparison."""

    def __init__(self, root, mode_info, display=True, latest_path=None):
        self.root, self.info, self.display = Path(root), mode_info, display
        self.latest_path = Path(latest_path) if latest_path is not None else None
        self.cockpits = {bot: cockpit_ui.Cockpit(
            display_enabled=False, latest_json_path=self.root / bot / 'cockpit_latest.json')
            for bot in (BOT1_ID, BOT2_ID)}
        self.snapshots, self.gateway = {}, None
        self.latest_browser_page = cockpit_ui.WAITING_PAGE
        self.latest_snapshot_json = '{}'
        self.handle, self.write_errors = None, 0

    def view(self, bot_id):
        return BotView(self, bot_id)

    def read_other(self, bot_id):
        """Snapshot written by a bot in another process (DUAL_MODE)."""
        path = self.root / bot_id / 'cockpit_latest.json'
        try:
            data = json.loads(path.read_text(encoding='utf-8'))
            if data.get('updated_at_utc'):
                self.snapshots[bot_id] = data
        except (OSError, ValueError):
            pass

    def refresh(self, read_files=()):
        for bot_id in read_files:
            self.read_other(bot_id)
        page = self.render()
        self.latest_browser_page = cockpit_ui.browser_page(page)
        self.latest_snapshot_json = json.dumps(self.snapshots, ensure_ascii=False, default=str)
        if self.latest_path is not None:
            try:
                cockpit_ui.write_atomic(self.latest_path, self.latest_browser_page)
            except OSError:
                self.write_errors += 1
        if self.display:
            from IPython.display import HTML, display
            if self.handle is None:
                self.handle = display(HTML(page), display_id=True)
            else:
                self.handle.update(HTML(page))
        return page

    def render(self):
        from html import escape
        style, sections = None, []
        for bot_id, name, active, badge in (
                (BOT1_ID, BOT1_NAME, self.info['bot1_active'], 'IEX · Echtzeit'),
                (BOT2_ID, BOT2_NAME, self.info['bot2_active'], 'SIMULIERT · SIP verzögert')):
            data = self.snapshots.get(bot_id)
            head = (f'<h2 class="dual-head">{escape(name)} <small>({bot_id}) · {escape(badge)}'
                    f'</small></h2>')
            if not active:
                sections.append(f'<div class="orb-cockpit">{head}<p class="muted">Deaktiviert im '
                                f'Modus {escape(self.info["mode"])} · keine Alpaca-Orders.</p></div>')
                continue
            if data is None:
                sections.append(f'<div class="orb-cockpit">{head}<p class="muted">Warte auf den '
                                'ersten Stand …</p></div>')
                continue
            section_style, body = cockpit_ui.page_parts(cockpit_ui.render_html(data))
            style = style or section_style
            body = body.replace('id="orb-stand"', 'class="orb-stand-section"')
            body = body.replace('<h2>US-Aktien-Bot · virtuelles Cockpit</h2>', head, 1)
            if bot_id == BOT1_ID:
                body = body.replace('<div class="tiles">', self.order_status() + '<div class="tiles">', 1)
            sections.append(body)
        style = style or base_style()
        now = pd.Timestamp.now(tz='UTC')
        quiet = all(d.get('stage') in cockpit_ui.QUIET_STAGES for d in self.snapshots.values()) \
            if self.snapshots else True
        header = (f'<div class="orb-cockpit"><h2>US-Aktien-Bot {escape(VERSION)}</h2>'
                  f'<div class="subtitle" id="orb-stand" data-updated-ms="{int(now.timestamp()*1000)}" '
                  f'data-quiet="{"1" if quiet else "0"}" data-replay="0" data-lag-ms="0">Modus '
                  f'<strong>{escape(self.info["mode"])}</strong> · Stand {now.tz_convert(NY):%H:%M:%S} NY '
                  '(Verarbeitungszeit)</div><div class="tiles"></div></div>')
        return ('<!doctype html><html lang="de"><head><meta charset="utf-8"><style>' + style +
                '.dual-head small{font-size:13px;font-weight:400;color:#58677a}'
                '.orb-cockpit{margin-bottom:14px}</style></head><body>' + header +
                ''.join(sections) + self.comparison_section() + '</body></html>')

    def order_status(self):
        from html import escape
        gateway = self.gateway
        if gateway is None:
            text, tone = 'Alpaca-Paper-Orders: AUS (keine Freigabe) – nur IEX-Simulation', 'neutral'
        elif gateway.enabled:
            text = (f'Alpaca-Paper-Orders: FREIGEGEBEN · Konto {gateway.account_number} · '
                    f'Start-Equity {gateway.start_equity:,.2f} USD · gesendete Orders {gateway.orders_sent}')
            tone = 'bad'
        else:
            text, tone = f'Alpaca-Paper-Orders: AUS – {gateway.disabled_reason}', 'neutral'
        if gateway is not None and gateway.flat_confirmed is not None:
            text += ' · Glattstellung: ' + ('Position 0 bestätigt' if gateway.flat_confirmed
                                           else 'NICHT bestätigt – manuell prüfen')
        return f'<p><span class="{tone}">{escape(text)}</span></p>'

    def comparison_section(self):
        from html import escape
        rows, decisions = [], []
        for bot_id, source in ((BOT1_ID, 'IEX Echtzeit (Simulation, Reihe 2)'),
                               (BOT2_ID, 'SIP verzögert (SIMULIERT, Reihe 3)')):
            data = self.snapshots.get(bot_id)
            ledger_id = BOT1_ID + '_SIM' if bot_id == BOT1_ID else BOT2_ID
            account = accounts.read_json(self.root / ledger_id / 'account.json')
            total = (sum(d['pnl_usd'] for d in account['days'].values()) if account else 0.0)
            if data is None:
                rows.append((escape(source), '—', '—', '—', '—', '—', '—', f'{total:,.2f} USD'))
                continue
            start = data.get('start_equity_usd') or 0
            day_pnl = (data.get('realized_pnl_proxy_usd') or 0) + (data.get('open_pnl_proxy_usd') or 0)
            if data.get('checked_decision_utc'):
                decisions.append(pd.Timestamp(data['checked_decision_utc']))
            rows.append((escape(source), f'{data.get("equity_proxy_usd", 0):,.2f} USD',
                         str(data.get('signal_total', 0)), str(data.get('closed_count', 0)),
                         f'{data.get("wins", 0)} / {data.get("losses", 0)}',
                         f'{day_pnl:,.2f} USD', f'{100 * day_pnl / start:+.2f} %' if start else '—',
                         f'{total:,.2f} USD'))
        both = self.info['bot1_active'] and self.info['bot2_active']
        common = min(decisions) if both and len(decisions) == 2 else None
        if not both:
            note = 'Vergleich erst im DUAL_MODE; hier nur der aktive Bot.'
        elif common is None:
            note = 'Noch kein gemeinsam verarbeiteter Marktzeitpunkt.'
        else:
            gap = (max(decisions) - min(decisions)).total_seconds() / 60
            note = (f'Zuletzt gemeinsam vollständig verarbeiteter Marktzeitpunkt: '
                    f'{common.tz_convert(NY):%H:%M} NY. ' +
                    (f'VORLÄUFIG: SIP liegt {gap:.0f} Min. zurück – Tageszahlen sind noch nicht '
                     'vergleichbar; abschließender Vergleich nach vollständiger Verarbeitung beider Bots.'
                     if gap > 1 else 'Beide Bots auf gleichem Marktzeitpunkt.'))
        body = cockpit_ui.table(('Bot / Datenquelle', 'Equity (virtuell)', 'Erkannte Signale',
                                 'Trades', 'Gewinn / Verlust', 'Tages-P&L', 'Tages-P&L %',
                                 'Gesamt-P&L (gebuchte Tage)'), rows, 'Noch keine Daten.')
        return (f'<div class="orb-cockpit"><h2>IEX vs. SIP – Vergleich</h2>'
                f'<p class="muted">{escape(note)}</p>{body}<div class="note">Reihe 1 (tatsächliche '
                'Alpaca-Paper-Fills von BOT 1) steht in IEX_REALTIME_PAPER_daily_performance.csv. '
                'Reihen 2 und 3 nutzen dasselbe Ausführungs- und Kostenmodell; einige Testtage '
                'beweisen keine langfristige Profitabilität.</div></div>')


def base_style():
    """The cockpit CSS, taken from an empty standard view."""
    from types import SimpleNamespace
    empty = SimpleNamespace(positions={}, closed=[], events=[], last_marks={}, report=lambda: dict(
        ending_equity_proxy_usd=0.0, ending_cash_usd=0.0, realized_pnl_usd=0.0,
        realized_pnl_10bp_stress_usd=0.0, data_degraded=False, entry_data_blocked=False,
        unreliable_virtual_trades=0, starting_cash_usd=0.0))
    now = pd.Timestamp.now(tz='UTC')
    return cockpit_ui.page_parts(cockpit_ui.render_html(
        cockpit_ui.snapshot(empty, now, None, 'WARTE_AUF_SIGNALFENSTER', 'live', 0)))[0]


class BotView:
    """Cockpit-like object handed to one bot's run loop."""

    def __init__(self, parent, bot_id):
        self.parent, self.bot_id = parent, bot_id

    def update(self, folder, *args, **kwargs):
        data = self.parent.cockpits[self.bot_id].update(folder, *args, **kwargs)
        if data is not None:
            self.parent.snapshots[self.bot_id] = data
            other = [b for b in (BOT1_ID, BOT2_ID) if b != self.bot_id]
            self.parent.refresh(read_files=other)
        return data


def serve_dual(dual, port=8765):
    """Browser link for the dual cockpit (same read-only server as V1.4.3)."""
    return cockpit_ui.open_browser_view(dual, port)


# ---------------------------------------------------------------- comparison log
def opening_stats(folder, feed):
    path = Path(folder) / f'session_bars_{feed}.csv'
    manifest = accounts.read_json(Path(folder) / 'manifest.json', {})
    if not path.exists() or path.stat().st_size <= 1:
        return {}
    bars = pd.read_csv(path)
    bars['timestamp'] = pd.to_datetime(bars.timestamp, utc=True)
    opening = pd.Timestamp(manifest['session_open_utc'])
    first = bars[(bars.timestamp >= opening) & (bars.timestamp < opening + pd.Timedelta(minutes=15))]
    return {symbol: dict(or_high=g.high.max(), or_low=g.low.min(), or_volume=g.volume.sum(),
                         or_bars=len(g)) for symbol, g in first.groupby('symbol')}


def first_signals(folder):
    found = {}
    for path in sorted((Path(folder) / 'signals').glob('*.json')):
        item = accounts.read_json(path)
        if isinstance(item, dict) and 'symbol' in item and item['symbol'] not in found:
            found[item['symbol']] = item
    return found


def eligible(folder):
    path = Path(folder) / 'universe_filter_results.csv'
    if not path.exists():
        return set()
    frame = pd.read_csv(path)
    return set(frame.loc[frame.result == 'PASS', 'symbol'])


def comparison_rows(day, iex_folder, sip_folder):
    sides = {}
    for name, folder, feed in (('iex', iex_folder, 'iex'), ('sip', sip_folder, 'sip')):
        sides[name] = dict(folder=folder, complete=bool(folder) and accounts.run_is_complete(folder),
                           stats=opening_stats(folder, feed) if folder else {},
                           signals=first_signals(folder) if folder else {},
                           eligible=eligible(folder) if folder else set())
    symbols = sorted(sides['iex']['eligible'] | sides['sip']['eligible'])
    complete = sides['iex']['complete'] and sides['sip']['complete']
    rows = []
    for symbol in symbols:
        row = dict(date=day, symbol=symbol,
                   same_universe=symbol in sides['iex']['eligible'] and symbol in sides['sip']['eligible'],
                   comparison_complete=complete)
        for name, side in sides.items():
            stats, signal = side['stats'].get(symbol, {}), side['signals'].get(symbol)
            decision = pd.Timestamp(signal['decision_time_utc']) if signal else None
            paper = signal.get('paper_order') if signal else None
            row.update({f'{name}_or_high': stats.get('or_high'), f'{name}_or_low': stats.get('or_low'),
                        f'{name}_or_volume': stats.get('or_volume'), f'{name}_or_bars': stats.get('or_bars'),
                        f'{name}_signal': signal is not None,
                        f'{name}_signal_time_ny': decision.tz_convert(NY).strftime('%H:%M') if decision is not None else None,
                        f'{name}_rvol': signal.get('rvol') if signal else None,
                        f'{name}_status': signal.get('status') if signal else None,
                        f'{name}_virtual_entry': (signal or {}).get('shadow_status') == 'VIRTUAL_ENTRY',
                        f'{name}_reason': (signal or {}).get('shadow_status') or (signal or {}).get('status')})
            if name == 'iex':
                row['iex_paper_order_status'] = paper.get('status') if isinstance(paper, dict) else None
        rows.append(row)
    return rows


def build_comparison(root):
    """IEX_SIP_comparison.csv and performance_series.csv from all recorded days."""
    root = Path(root)
    sip = accounts.read_json(root / BOT2_ID / 'account.json', {'days': {}})
    iex = accounts.read_json(root / (BOT1_ID + '_SIM') / 'account.json', {'days': {}})
    paper = accounts.read_json(root / BOT1_ID / 'paper_days.json', {})
    rows, series = [], []
    for day in sorted(set(sip['days']) | set(iex['days'])):
        rows += comparison_rows(day, iex['days'].get(day, {}).get('run_folder'),
                                sip['days'].get(day, {}).get('run_folder'))
    for label, source in (('IEX_SIMULATION', iex['days']), ('SIP_SIMULATION', sip['days'])):
        for day, entry in sorted(source.items()):
            series.append(dict(date=day, series=label, pnl_usd=entry['pnl_usd'],
                               return_pct=100 * entry['pnl_usd'] / entry['start_equity_usd'],
                               trades=entry['trades'], simulated=True))
    for day, entry in sorted(paper.items()):
        if entry.get('orders_enabled'):
            series.append(dict(date=day, series='IEX_ALPACA_PAPER', pnl_usd=entry['pnl_usd'],
                               return_pct=100 * entry['pnl_usd'] / entry['start_equity_usd'],
                               trades=None, simulated=False))
    accounts.write_csv(root / 'IEX_SIP_comparison.csv', pd.DataFrame(rows))
    accounts.write_csv(root / 'performance_series.csv', pd.DataFrame(series))
    return root / 'IEX_SIP_comparison.csv'


# ---------------------------------------------------------------- orchestration
def run_session(mode, trading_client, data_client, data_keys, project_root, c, s,
                approval=None, display=True, serve=True, shared=None, clock=None,
                bot2_process_options=None):
    """Central start of V1.5 (one notebook cell). Returns a dict of run folders.

    `shared`, `clock` and `bot2_process_options` exist for tests only."""
    info = resolve_mode(mode)
    root = Path(project_root) / 'v1_5'
    clock = clock or sim.SYSTEM_CLOCK
    day = clock.now().tz_convert(NY).strftime('%Y-%m-%d')
    print(f'{VERSION} · Modus {info["mode"]} · '
          f'{BOT1_NAME}: {"aktiv" if info["bot1_active"] else "deaktiviert"} · '
          f'{BOT2_NAME}: {"aktiv (SIMULIERT)" if info["bot2_active"] else "deaktiviert"}')
    dual = DualCockpit(root, info, display=display, latest_path=Path(project_root) / 'cockpit_latest.html')
    if serve:
        try:
            serve_dual(dual)
        except Exception as exc:
            print('Browser-Link nicht verfügbar:', type(exc).__name__)
    dual.refresh()
    calendar, universe = shared or prepare_shared(trading_client, day)
    results = {}
    if info['mode'] == 'SIP_ONLY':
        results[BOT2_ID] = bot2.run_sip_shadow_day(data_client, project_root, c, s, calendar, universe,
                                                   cockpit=dual.view(BOT2_ID),
                                                   base_clock=None if clock is sim.SYSTEM_CLOCK else clock)
    else:
        import iex_paper_bot_v15 as bot1   # only BOT 1 code paths load the order gateway
        child = None
        if info['bot2_active']:
            child = start_bot2_process(data_keys, project_root, c, s, calendar, universe, root,
                                       **(bot2_process_options or {}))
        factory = capture_gateway(dual, bot1.paper.PaperOrderGateway)
        try:
            results[BOT1_ID], _ = bot1.run_iex_day(trading_client, data_client, project_root, c, s,
                                                   universe, approval=approval, clock=clock,
                                                   cockpit=dual.view(BOT1_ID), gateway_factory=factory)
        finally:
            if child is not None:
                wait_for_bot2(child, dual)
    build_comparison(root)
    dual.refresh(read_files=(BOT1_ID, BOT2_ID))
    return results


def capture_gateway(dual, gateway_class):
    def factory(*args, **kwargs):
        dual.gateway = gateway_class(*args, **kwargs)
        return dual.gateway
    return factory


def start_bot2_process(data_keys, project_root, c, s, calendar, universe, root,
                       data_factory='sip_shadow_bot_v15:alpaca_market_data',
                       factory_kwargs=None, clock_factory=None):
    """BOT 2 in its own (spawned) process: own memory, no trading client."""
    context = multiprocessing.get_context('spawn')
    kwargs = factory_kwargs if factory_kwargs is not None else dict(
        api_key=data_keys[0], secret_key=data_keys[1])
    process = context.Process(target=bot2.child_main, name='SIP_DELAYED_SHADOW', daemon=True, args=(
        data_factory, kwargs, str(project_root), c, s, calendar, universe.to_dict('records'),
        str(root / BOT2_ID)), kwargs=dict(clock_factory=clock_factory))
    process.start()
    return process


def wait_for_bot2(process, dual, poll_seconds=30):
    """After BOT 1 finished: keep the cockpit current until BOT 2 has closed its day."""
    while process.is_alive():
        dual.refresh(read_files=(BOT2_ID,))
        process.join(poll_seconds)
    dual.refresh(read_files=(BOT2_ID,))


## Zentraler Start

Diese Zelle startet die Bots gemäß `BOT_MODE`. Kalender und Aktienliste werden einmal gelesen und beiden Bots gleich übergeben. BOT 2 erhält nur einen Marktdaten-Client, keinen Order-Client. Die Zelle läuft bis nach Börsenschluss; BOT 2 endet 16 Minuten nach BOT 1.


In [ ]:
from pathlib import Path
from dataclasses import replace
from google.colab import drive, userdata
from alpaca.trading.client import TradingClient
from alpaca.data.historical import StockHistoricalDataClient
import hashlib, importlib, json, sys
import pandas as pd

drive.mount('/content/drive')
project = Path('/content/drive/MyDrive/US_Aktien_Bot')
source = project / 'v0_3_runs' / PREVIOUS_SCANNER_RUN
prior = json.loads((source / 'manifest.json').read_text())
comparison = {'base_source.py': '/content/us_orb_test_v02.py',
              'scanner_source.py': '/content/us_orb_scanner_v03.py'}
for filename, digest in prior['code_hashes'].items():
    if hashlib.sha256((source / filename).read_bytes()).hexdigest() != digest:
        raise ValueError('Scannerquellcode wurde verändert: ' + filename)
    if hashlib.sha256(Path(comparison[filename]).read_bytes()).hexdigest() != digest:
        raise ValueError('Notebook-Regeln stimmen nicht mit dem Testlauf überein: ' + filename)

paper_key = (userdata.get('ALPACA_PAPER_API_KEY') or '').strip()
paper_secret = (userdata.get('ALPACA_PAPER_SECRET_KEY') or '').strip()
if not paper_key or not paper_secret:
    raise RuntimeError('Bitte die beiden Paper-Secrets freigeben.')
# Ausschließlich der Paper-Endpunkt; BOT 2 erhält diesen Client nie.
broker = TradingClient(api_key=paper_key, secret_key=paper_secret,
                       paper=True, url_override='https://paper-api.alpaca.markets')
market_data = StockHistoricalDataClient(api_key=paper_key, secret_key=paper_secret)
data_keys = (paper_key, paper_secret)   # nur für den Marktdaten-Client von BOT 2 im DUAL_MODE
del paper_key, paper_secret
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
modules = {}
for name in ('us_orb_test_v02', 'us_orb_scanner_v03', 'orb_portfolio_v15', 'orb_sim_v15',
             'orb_cockpit_v15', 'orb_replay_v15', 'bot_accounts_v15', 'sip_shadow_bot_v15',
             'paper_orders_v15', 'iex_paper_bot_v15', 'dual_bot_v15'):
    modules[name] = importlib.reload(importlib.import_module(name))
base, scanner, dual = modules['us_orb_test_v02'], modules['us_orb_scanner_v03'], modules['dual_bot_v15']

ny_date = pd.Timestamp.now(tz='UTC').tz_convert('America/New_York').strftime('%Y-%m-%d')
config = replace(base.Config(**prior['base_config']), test_date=ny_date)
scan_config = scanner.ScannerConfig(**prior['scanner_config'])
if BOT_MODE == 'SIP_ONLY' and PAPER_ORDER_APPROVAL:
    print('Hinweis: Freigabe wird im Modus SIP_ONLY ignoriert – keine Orders möglich.')
result_folders = dual.run_session(
    BOT_MODE, broker, market_data, data_keys, project, config, scan_config,
    approval=PAPER_ORDER_APPROVAL if BOT_MODE in ('IEX_ONLY', 'DUAL_MODE') else None)
print('Ergebnisordner:', {bot: str(folder) for bot, folder in result_folders.items()})


## Nach dem Lauf

Je Bot bitte `status.json`, `coverage.json` und `shadow_state.json` aus dem jeweiligen Laufordner teilen, dazu aus `MyDrive/US_Aktien_Bot/v1_5/` die Dateien `*_daily_performance.csv`, `*_trades.csv`, `IEX_SIP_comparison.csv` und `performance_series.csv`.

Alle Ergebnisse von BOT 2 und der IEX-Vergleichssimulation sind **SIMULIERT** und keine Alpaca-Fills. Einige Testtage beweisen keine langfristige Profitabilität.


In [ ]:
from IPython.display import display, HTML
v15 = project / 'v1_5'
for bot, folder in result_folders.items():
    print(f'\n=== {bot} · {folder}')
    for name in ('status.json', 'coverage.json', 'shadow_state.json'):
        path = Path(folder) / name
        if path.exists():
            print(name, json.dumps(json.loads(path.read_text()), ensure_ascii=False, indent=2)[:1500])
for name in ('SIP_DELAYED_SHADOW/SIP_DELAYED_SHADOW_daily_performance.csv',
             'IEX_REALTIME_PAPER_SIM/IEX_REALTIME_PAPER_SIM_daily_performance.csv',
             'IEX_REALTIME_PAPER/IEX_REALTIME_PAPER_daily_performance.csv',
             'performance_series.csv', 'IEX_SIP_comparison.csv'):
    path = v15 / name
    if path.exists() and path.stat().st_size > 1:
        print('\n' + name)
        display(pd.read_csv(path).tail(20))
latest = project / 'cockpit_latest.html'
if latest.exists():
    print('Letzter Cockpit-Stand:')
    display(HTML(latest.read_text(encoding='utf-8')))
